# Subject: test_tkinter
Source: /home/devuser/edu_data/datasets/TrainingDB/EncryT/cpython/Lib/test/test_tkinter

### File: __main__.py

In [ ]:
from . import load_tests
import unittest

unittest.main()

### File: support.py

In [ ]:
import functools
import tkinter

class AbstractTkTest:

    @classmethod
    def setUpClass(cls):
        cls._old_support_default_root = tkinter._support_default_root
        destroy_default_root()
        tkinter.NoDefaultRoot()
        cls.root = tkinter.Tk()
        cls.wantobjects = cls.root.wantobjects()
        # De-maximize main window.
        # Some window managers can maximize new windows.
        cls.root.wm_state('normal')
        try:
            cls.root.wm_attributes(zoomed=False)
        except tkinter.TclError:
            pass

    @classmethod
    def tearDownClass(cls):
        cls.root.update_idletasks()
        cls.root.destroy()
        del cls.root
        tkinter._default_root = None
        tkinter._support_default_root = cls._old_support_default_root

    def setUp(self):
        self.root.deiconify()

    def tearDown(self):
        for w in self.root.winfo_children():
            w.destroy()
        self.root.withdraw()


class AbstractDefaultRootTest:

    def setUp(self):
        self._old_support_default_root = tkinter._support_default_root
        destroy_default_root()
        tkinter._support_default_root = True
        self.wantobjects = tkinter.wantobjects

    def tearDown(self):
        destroy_default_root()
        tkinter._default_root = None
        tkinter._support_default_root = self._old_support_default_root

    def _test_widget(self, constructor):
        # no master passing
        x = constructor()
        self.assertIsNotNone(tkinter._default_root)
        self.assertIs(x.master, tkinter._default_root)
        self.assertIs(x.tk, tkinter._default_root.tk)
        x.destroy()
        destroy_default_root()
        tkinter.NoDefaultRoot()
        self.assertRaises(RuntimeError, constructor)
        self.assertNotHasAttr(tkinter, '_default_root')


def destroy_default_root():
    if getattr(tkinter, '_default_root', None):
        tkinter._default_root.update_idletasks()
        tkinter._default_root.destroy()
        tkinter._default_root = None

def simulate_mouse_click(widget, x, y):
    """Generate proper events to click at the x, y position (tries to act
    like an X server)."""
    widget.event_generate('<Enter>', x=0, y=0)
    widget.event_generate('<Motion>', x=x, y=y)
    widget.event_generate('<ButtonPress-1>', x=x, y=y)
    widget.event_generate('<ButtonRelease-1>', x=x, y=y)


import _tkinter
tcl_version = tuple(map(int, _tkinter.TCL_VERSION.split('.')))
tk_version = tuple(map(int, _tkinter.TK_VERSION.split('.')))

def requires_tk(*version):
    if len(version) <= 2 and tk_version >= version:
        return lambda test: test

    def deco(test):
        @functools.wraps(test)
        def newtest(self):
            root = getattr(self, 'root', None)
            if get_tk_patchlevel(root) < version:
                self.skipTest('requires Tk version >= ' +
                                '.'.join(map(str, version)))
            test(self)
        return newtest
    return deco

_tk_patchlevel = None
def get_tk_patchlevel(root):
    global _tk_patchlevel
    if _tk_patchlevel is None:
        _tk_patchlevel = tkinter._parse_version(root.tk.globalgetvar('tk_patchLevel'))
    return _tk_patchlevel

units = {
    'c': 72 / 2.54,     # centimeters
    'i': 72,            # inches
    'm': 72 / 25.4,     # millimeters
    'p': 1,             # points
}

def pixels_conv(value):
    return float(value[:-1]) * units[value[-1:]]

def tcl_obj_eq(actual, expected):
    if actual == expected:
        return True
    if isinstance(actual, _tkinter.Tcl_Obj):
        if isinstance(expected, str):
            return str(actual) == expected
    if isinstance(actual, tuple):
        if isinstance(expected, tuple):
            return (len(actual) == len(expected) and
                    all(tcl_obj_eq(act, exp)
                        for act, exp in zip(actual, expected)))
    return False

def widget_eq(actual, expected):
    if actual == expected:
        return True
    if isinstance(actual, (str, tkinter.Widget)):
        if isinstance(expected, (str, tkinter.Widget)):
            return str(actual) == str(expected)
    return False

### File: test_colorchooser.py

In [ ]:
import unittest
import tkinter
from test.support import requires, swap_attr
from test.test_tkinter.support import AbstractDefaultRootTest, AbstractTkTest
from tkinter import colorchooser
from tkinter.colorchooser import askcolor
from tkinter.commondialog import Dialog

requires('gui')


class ChooserTest(AbstractTkTest, unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        AbstractTkTest.setUpClass.__func__(cls)
        cls.cc = colorchooser.Chooser(initialcolor='dark blue slate')

    def test_fixoptions(self):
        cc = self.cc
        cc._fixoptions()
        self.assertEqual(cc.options['initialcolor'], 'dark blue slate')

        cc.options['initialcolor'] = '#D2D269691E1E'
        cc._fixoptions()
        self.assertEqual(cc.options['initialcolor'], '#D2D269691E1E')

        cc.options['initialcolor'] = (210, 105, 30)
        cc._fixoptions()
        self.assertEqual(cc.options['initialcolor'], '#d2691e')

    def test_fixresult(self):
        cc = self.cc
        self.assertEqual(cc._fixresult(self.root, ()), (None, None))
        self.assertEqual(cc._fixresult(self.root, ''), (None, None))
        self.assertEqual(cc._fixresult(self.root, 'chocolate'),
                         ((210, 105, 30), 'chocolate'))
        self.assertEqual(cc._fixresult(self.root, '#4a3c8c'),
                         ((74, 60, 140), '#4a3c8c'))


class DefaultRootTest(AbstractDefaultRootTest, unittest.TestCase):

    def test_askcolor(self):
        def test_callback(dialog, master):
            nonlocal ismapped
            master.update()
            ismapped = master.winfo_ismapped()
            raise ZeroDivisionError

        with swap_attr(Dialog, '_test_callback', test_callback):
            ismapped = None
            self.assertRaises(ZeroDivisionError, askcolor)
            #askcolor()
            self.assertEqual(ismapped, False)

            root = tkinter.Tk()
            ismapped = None
            self.assertRaises(ZeroDivisionError, askcolor)
            self.assertEqual(ismapped, True)
            root.destroy()

            tkinter.NoDefaultRoot()
            self.assertRaises(RuntimeError, askcolor)


if __name__ == "__main__":
    unittest.main()

### File: test_font.py

In [ ]:
import unittest
import tkinter
from tkinter import font
from test.support import requires, gc_collect, ALWAYS_EQ
from test.test_tkinter.support import AbstractTkTest, AbstractDefaultRootTest

requires('gui')

fontname = "TkDefaultFont"

class FontTest(AbstractTkTest, unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        AbstractTkTest.setUpClass.__func__(cls)
        try:
            cls.font = font.Font(root=cls.root, name=fontname, exists=True)
        except tkinter.TclError:
            cls.font = font.Font(root=cls.root, name=fontname, exists=False)

    def test_configure(self):
        options = self.font.configure()
        self.assertGreaterEqual(set(options),
            {'family', 'size', 'weight', 'slant', 'underline', 'overstrike'})
        for key in options:
            self.assertEqual(self.font.cget(key), options[key])
            self.assertEqual(self.font[key], options[key])
        for key in 'family', 'weight', 'slant':
            self.assertIsInstance(options[key], str)
            self.assertIsInstance(self.font.cget(key), str)
            self.assertIsInstance(self.font[key], str)
        sizetype = int if self.wantobjects else str
        for key in 'size', 'underline', 'overstrike':
            self.assertIsInstance(options[key], sizetype)
            self.assertIsInstance(self.font.cget(key), sizetype)
            self.assertIsInstance(self.font[key], sizetype)

    def test_unicode_family(self):
        family = 'MS \u30b4\u30b7\u30c3\u30af'
        try:
            f = font.Font(root=self.root, family=family, exists=True)
        except tkinter.TclError:
            f = font.Font(root=self.root, family=family, exists=False)
        self.assertEqual(f.cget('family'), family)
        del f
        gc_collect()

    def test_actual(self):
        options = self.font.actual()
        self.assertGreaterEqual(set(options),
            {'family', 'size', 'weight', 'slant', 'underline', 'overstrike'})
        for key in options:
            self.assertEqual(self.font.actual(key), options[key])
        for key in 'family', 'weight', 'slant':
            self.assertIsInstance(options[key], str)
            self.assertIsInstance(self.font.actual(key), str)
        sizetype = int if self.wantobjects else str
        for key in 'size', 'underline', 'overstrike':
            self.assertIsInstance(options[key], sizetype)
            self.assertIsInstance(self.font.actual(key), sizetype)

    def test_name(self):
        self.assertEqual(self.font.name, fontname)
        self.assertEqual(str(self.font), fontname)

    def test_equality(self):
        font1 = font.Font(root=self.root, name=fontname, exists=True)
        font2 = font.Font(root=self.root, name=fontname, exists=True)
        self.assertIsNot(font1, font2)
        self.assertEqual(font1, font2)
        self.assertNotEqual(font1, font1.copy())

        self.assertNotEqual(font1, 0)
        self.assertEqual(font1, ALWAYS_EQ)

        root2 = tkinter.Tk()
        self.addCleanup(root2.destroy)
        font3 = font.Font(root=root2, name=fontname, exists=True)
        self.assertEqual(str(font1), str(font3))
        self.assertNotEqual(font1, font3)

    def test_measure(self):
        self.assertIsInstance(self.font.measure('abc'), int)

    def test_metrics(self):
        metrics = self.font.metrics()
        self.assertGreaterEqual(set(metrics),
            {'ascent', 'descent', 'linespace', 'fixed'})
        for key in metrics:
            self.assertEqual(self.font.metrics(key), metrics[key])
            self.assertIsInstance(metrics[key], int)
            self.assertIsInstance(self.font.metrics(key), int)

    def test_families(self):
        families = font.families(self.root)
        self.assertIsInstance(families, tuple)
        self.assertTrue(families)
        for family in families:
            self.assertIsInstance(family, str)
            self.assertTrue(family)

    def test_names(self):
        names = font.names(self.root)
        self.assertIsInstance(names, tuple)
        self.assertTrue(names)
        for name in names:
            self.assertIsInstance(name, str)
            self.assertTrue(name)
        self.assertIn(fontname, names)

    def test_nametofont(self):
        testfont = font.nametofont(fontname, root=self.root)
        self.assertIsInstance(testfont, font.Font)
        self.assertEqual(testfont.name, fontname)

    def test_repr(self):
        self.assertEqual(
            repr(self.font), f'<tkinter.font.Font object {fontname!r}>'
        )


class DefaultRootTest(AbstractDefaultRootTest, unittest.TestCase):

    def test_families(self):
        self.assertRaises(RuntimeError, font.families)
        root = tkinter.Tk()
        families = font.families()
        self.assertIsInstance(families, tuple)
        self.assertTrue(families)
        for family in families:
            self.assertIsInstance(family, str)
            self.assertTrue(family)
        root.destroy()
        tkinter.NoDefaultRoot()
        self.assertRaises(RuntimeError, font.families)

    def test_names(self):
        self.assertRaises(RuntimeError, font.names)
        root = tkinter.Tk()
        names = font.names()
        self.assertIsInstance(names, tuple)
        self.assertTrue(names)
        for name in names:
            self.assertIsInstance(name, str)
            self.assertTrue(name)
        self.assertIn(fontname, names)
        root.destroy()
        tkinter.NoDefaultRoot()
        self.assertRaises(RuntimeError, font.names)

    def test_nametofont(self):
        self.assertRaises(RuntimeError, font.nametofont, fontname)
        root = tkinter.Tk()
        testfont = font.nametofont(fontname)
        self.assertIsInstance(testfont, font.Font)
        self.assertEqual(testfont.name, fontname)
        root.destroy()
        tkinter.NoDefaultRoot()
        self.assertRaises(RuntimeError, font.nametofont, fontname)


class TestModule(unittest.TestCase):
    def test_deprecated__version__(self):
        with self.assertWarnsRegex(
            DeprecationWarning,
            "'__version__' is deprecated and slated for removal in Python 3.20",
        ) as cm:
            getattr(font, "__version__")
        self.assertEqual(cm.filename, __file__)


if __name__ == "__main__":
    unittest.main()

### File: test_geometry_managers.py

In [ ]:
import unittest
import re
import tkinter
from tkinter import TclError
from test.support import requires

from test.test_tkinter.support import pixels_conv
from test.test_tkinter.widget_tests import AbstractWidgetTest

requires('gui')


EXPECTED_FLOAT_ERRMSG = 'expected floating-point number but got "{}"'
EXPECTED_FLOAT_OR_EMPTY_ERRMSG = 'expected floating-point number (or "" )?but got "{}"'
EXPECTED_SCREEN_DISTANCE_ERRMSG = '(bad|expected) screen distance (but got )?"{}"'
EXPECTED_SCREEN_DISTANCE_OR_EMPTY_ERRMSG = '(bad|expected) screen distance (or "" but got )?"{}"'

class PackTest(AbstractWidgetTest, unittest.TestCase):

    test_keys = None

    def create2(self):
        pack = tkinter.Toplevel(self.root, name='pack')
        pack.wm_geometry('300x200+0+0')
        pack.wm_minsize(1, 1)
        a = tkinter.Frame(pack, name='a', width=20, height=40, bg='red')
        b = tkinter.Frame(pack, name='b', width=50, height=30, bg='blue')
        c = tkinter.Frame(pack, name='c', width=80, height=80, bg='green')
        d = tkinter.Frame(pack, name='d', width=40, height=30, bg='yellow')
        return pack, a, b, c, d

    def test_pack_configure_after(self):
        pack, a, b, c, d = self.create2()
        with self.assertRaisesRegex(TclError, 'window "%s" isn\'t packed' % b):
            a.pack_configure(after=b)
        with self.assertRaisesRegex(TclError, 'bad window path name ".foo"'):
            a.pack_configure(after='.foo')
        a.pack_configure(side='top')
        b.pack_configure(side='top')
        c.pack_configure(side='top')
        d.pack_configure(side='top')
        self.assertEqual(pack.pack_slaves(), [a, b, c, d])
        a.pack_configure(after=b)
        self.assertEqual(pack.pack_slaves(), [b, a, c, d])
        a.pack_configure(after=a)
        self.assertEqual(pack.pack_slaves(), [b, a, c, d])

    def test_pack_configure_anchor(self):
        pack, a, b, c, d = self.create2()
        def check(anchor, geom):
            a.pack_configure(side='top', ipadx=5, padx=10, ipady=15, pady=20,
                             expand=True, anchor=anchor)
            self.root.update()
            self.assertEqual(a.winfo_geometry(), geom)
        check('n', '30x70+135+20')
        check('ne', '30x70+260+20')
        check('e', '30x70+260+65')
        check('se', '30x70+260+110')
        check('s', '30x70+135+110')
        check('sw', '30x70+10+110')
        check('w', '30x70+10+65')
        check('nw', '30x70+10+20')
        check('center', '30x70+135+65')

    def test_pack_configure_before(self):
        pack, a, b, c, d = self.create2()
        with self.assertRaisesRegex(TclError, 'window "%s" isn\'t packed' % b):
            a.pack_configure(before=b)
        with self.assertRaisesRegex(TclError, 'bad window path name ".foo"'):
            a.pack_configure(before='.foo')
        a.pack_configure(side='top')
        b.pack_configure(side='top')
        c.pack_configure(side='top')
        d.pack_configure(side='top')
        self.assertEqual(pack.pack_slaves(), [a, b, c, d])
        a.pack_configure(before=d)
        self.assertEqual(pack.pack_slaves(), [b, c, a, d])
        a.pack_configure(before=a)
        self.assertEqual(pack.pack_slaves(), [b, c, a, d])

    def test_pack_configure_expand(self):
        pack, a, b, c, d = self.create2()
        def check(*geoms):
            self.root.update()
            self.assertEqual(a.winfo_geometry(), geoms[0])
            self.assertEqual(b.winfo_geometry(), geoms[1])
            self.assertEqual(c.winfo_geometry(), geoms[2])
            self.assertEqual(d.winfo_geometry(), geoms[3])
        a.pack_configure(side='left')
        b.pack_configure(side='top')
        c.pack_configure(side='right')
        d.pack_configure(side='bottom')
        check('20x40+0+80', '50x30+135+0', '80x80+220+75', '40x30+100+170')
        a.pack_configure(side='left', expand='yes')
        b.pack_configure(side='top', expand='on')
        c.pack_configure(side='right', expand=True)
        d.pack_configure(side='bottom', expand=1)
        check('20x40+40+80', '50x30+175+35', '80x80+180+110', '40x30+100+135')
        a.pack_configure(side='left', expand='yes', fill='both')
        b.pack_configure(side='top', expand='on', fill='both')
        c.pack_configure(side='right', expand=True, fill='both')
        d.pack_configure(side='bottom', expand=1, fill='both')
        check('100x200+0+0', '200x100+100+0', '160x100+140+100', '40x100+100+100')

    def test_pack_configure_in(self):
        pack, a, b, c, d = self.create2()
        a.pack_configure(side='top')
        b.pack_configure(side='top')
        c.pack_configure(side='top')
        d.pack_configure(side='top')
        a.pack_configure(in_=pack)
        self.assertEqual(pack.pack_slaves(), [b, c, d, a])
        a.pack_configure(in_=c)
        self.assertEqual(pack.pack_slaves(), [b, c, d])
        self.assertEqual(c.pack_slaves(), [a])
        with self.assertRaisesRegex(
                TclError, """can't pack "?%s"? inside itself""" % (a,)):
            a.pack_configure(in_=a)
        with self.assertRaisesRegex(TclError, 'bad window path name ".foo"'):
            a.pack_configure(in_='.foo')

    def test_pack_configure_padx_ipadx_fill(self):
        pack, a, b, c, d = self.create2()
        def check(geom1, geom2, **kwargs):
            a.pack_forget()
            b.pack_forget()
            a.pack_configure(**kwargs)
            b.pack_configure(expand=True, fill='both')
            self.root.update()
            self.assertEqual(a.winfo_geometry(), geom1)
            self.assertEqual(b.winfo_geometry(), geom2)
        check('20x40+260+80', '240x200+0+0', side='right', padx=20)
        check('20x40+250+80', '240x200+0+0', side='right', padx=(10, 30))
        check('60x40+240+80', '240x200+0+0', side='right', ipadx=20)
        check('30x40+260+80', '250x200+0+0', side='right', ipadx=5, padx=10)
        check('20x40+260+80', '240x200+0+0', side='right', padx=20, fill='x')
        check('20x40+249+80', '240x200+0+0',
              side='right', padx=(9, 31), fill='x')
        check('60x40+240+80', '240x200+0+0', side='right', ipadx=20, fill='x')
        check('30x40+260+80', '250x200+0+0',
              side='right', ipadx=5, padx=10, fill='x')
        check('30x40+255+80', '250x200+0+0',
              side='right', ipadx=5, padx=(5, 15), fill='x')
        check('20x40+140+0', '300x160+0+40', side='top', padx=20)
        check('20x40+120+0', '300x160+0+40', side='top', padx=(0, 40))
        check('60x40+120+0', '300x160+0+40', side='top', ipadx=20)
        check('30x40+135+0', '300x160+0+40', side='top', ipadx=5, padx=10)
        check('30x40+130+0', '300x160+0+40', side='top', ipadx=5, padx=(5, 15))
        check('260x40+20+0', '300x160+0+40', side='top', padx=20, fill='x')
        check('260x40+25+0', '300x160+0+40',
              side='top', padx=(25, 15), fill='x')
        check('300x40+0+0', '300x160+0+40', side='top', ipadx=20, fill='x')
        check('280x40+10+0', '300x160+0+40',
              side='top', ipadx=5, padx=10, fill='x')
        check('280x40+5+0', '300x160+0+40',
              side='top', ipadx=5, padx=(5, 15), fill='x')
        a.pack_configure(padx='1c')
        self.assertEqual(a.pack_info()['padx'],
                         self._str(pack.winfo_pixels('1c')))
        a.pack_configure(ipadx='1c')
        self.assertEqual(a.pack_info()['ipadx'],
                         self._str(pack.winfo_pixels('1c')))

    def test_pack_configure_pady_ipady_fill(self):
        pack, a, b, c, d = self.create2()
        def check(geom1, geom2, **kwargs):
            a.pack_forget()
            b.pack_forget()
            a.pack_configure(**kwargs)
            b.pack_configure(expand=True, fill='both')
            self.root.update()
            self.assertEqual(a.winfo_geometry(), geom1)
            self.assertEqual(b.winfo_geometry(), geom2)
        check('20x40+280+80', '280x200+0+0', side='right', pady=20)
        check('20x40+280+70', '280x200+0+0', side='right', pady=(10, 30))
        check('20x80+280+60', '280x200+0+0', side='right', ipady=20)
        check('20x50+280+75', '280x200+0+0', side='right', ipady=5, pady=10)
        check('20x40+280+80', '280x200+0+0', side='right', pady=20, fill='x')
        check('20x40+280+69', '280x200+0+0',
              side='right', pady=(9, 31), fill='x')
        check('20x80+280+60', '280x200+0+0', side='right', ipady=20, fill='x')
        check('20x50+280+75', '280x200+0+0',
              side='right', ipady=5, pady=10, fill='x')
        check('20x50+280+70', '280x200+0+0',
              side='right', ipady=5, pady=(5, 15), fill='x')
        check('20x40+140+20', '300x120+0+80', side='top', pady=20)
        check('20x40+140+0', '300x120+0+80', side='top', pady=(0, 40))
        check('20x80+140+0', '300x120+0+80', side='top', ipady=20)
        check('20x50+140+10', '300x130+0+70', side='top', ipady=5, pady=10)
        check('20x50+140+5', '300x130+0+70', side='top', ipady=5, pady=(5, 15))
        check('300x40+0+20', '300x120+0+80', side='top', pady=20, fill='x')
        check('300x40+0+25', '300x120+0+80',
              side='top', pady=(25, 15), fill='x')
        check('300x80+0+0', '300x120+0+80', side='top', ipady=20, fill='x')
        check('300x50+0+10', '300x130+0+70',
              side='top', ipady=5, pady=10, fill='x')
        check('300x50+0+5', '300x130+0+70',
              side='top', ipady=5, pady=(5, 15), fill='x')
        a.pack_configure(pady='1c')
        self.assertEqual(a.pack_info()['pady'],
                         self._str(pack.winfo_pixels('1c')))
        a.pack_configure(ipady='1c')
        self.assertEqual(a.pack_info()['ipady'],
                         self._str(pack.winfo_pixels('1c')))

    def test_pack_configure_side(self):
        pack, a, b, c, d = self.create2()
        def check(side, geom1, geom2):
            a.pack_configure(side=side)
            self.assertEqual(a.pack_info()['side'], side)
            b.pack_configure(expand=True, fill='both')
            self.root.update()
            self.assertEqual(a.winfo_geometry(), geom1)
            self.assertEqual(b.winfo_geometry(), geom2)
        check('top', '20x40+140+0', '300x160+0+40')
        check('bottom', '20x40+140+160', '300x160+0+0')
        check('left', '20x40+0+80', '280x200+20+0')
        check('right', '20x40+280+80', '280x200+0+0')

    def test_pack_forget(self):
        pack, a, b, c, d = self.create2()
        a.pack_configure()
        b.pack_configure()
        c.pack_configure()
        self.assertEqual(pack.pack_slaves(), [a, b, c])
        b.pack_forget()
        self.assertEqual(pack.pack_slaves(), [a, c])
        b.pack_forget()
        self.assertEqual(pack.pack_slaves(), [a, c])
        d.pack_forget()

    def test_pack_info(self):
        pack, a, b, c, d = self.create2()
        with self.assertRaisesRegex(TclError, 'window "%s" isn\'t packed' % a):
            a.pack_info()
        a.pack_configure()
        b.pack_configure(side='right', in_=a, anchor='s', expand=True, fill='x',
                         ipadx=5, padx=10, ipady=2, pady=(5, 15))
        info = a.pack_info()
        self.assertIsInstance(info, dict)
        self.assertEqual(info['anchor'], 'center')
        self.assertEqual(info['expand'], self._str(0))
        self.assertEqual(info['fill'], 'none')
        self.assertEqual(info['in'], pack)
        self.assertEqual(info['ipadx'], self._str(0))
        self.assertEqual(info['ipady'], self._str(0))
        self.assertEqual(info['padx'], self._str(0))
        self.assertEqual(info['pady'], self._str(0))
        self.assertEqual(info['side'], 'top')
        info = b.pack_info()
        self.assertIsInstance(info, dict)
        self.assertEqual(info['anchor'], 's')
        self.assertEqual(info['expand'], self._str(1))
        self.assertEqual(info['fill'], 'x')
        self.assertEqual(info['in'], a)
        self.assertEqual(info['ipadx'], self._str(5))
        self.assertEqual(info['ipady'], self._str(2))
        self.assertEqual(info['padx'], self._str(10))
        self.assertEqual(info['pady'], self._str((5, 15)))
        self.assertEqual(info['side'], 'right')

    def test_pack_propagate(self):
        pack, a, b, c, d = self.create2()
        pack.configure(width=300, height=200)
        a.pack_configure()
        pack.pack_propagate(False)
        self.root.update()
        self.assertEqual(pack.winfo_reqwidth(), 300)
        self.assertEqual(pack.winfo_reqheight(), 200)
        pack.pack_propagate(True)
        self.root.update()
        self.assertEqual(pack.winfo_reqwidth(), 20)
        self.assertEqual(pack.winfo_reqheight(), 40)

    def test_pack_slaves(self):
        pack, a, b, c, d = self.create2()
        self.assertEqual(pack.pack_slaves(), [])
        a.pack_configure()
        self.assertEqual(pack.pack_slaves(), [a])
        b.pack_configure()
        self.assertEqual(pack.pack_slaves(), [a, b])


class PlaceTest(AbstractWidgetTest, unittest.TestCase):

    test_keys = None

    def create2(self):
        t = tkinter.Toplevel(self.root, width=300, height=200, bd=0)
        t.wm_geometry('300x200+0+0')
        f = tkinter.Frame(t, width=154, height=84, bd=2, relief='raised')
        f.place_configure(x=48, y=38)
        f2 = tkinter.Frame(t, width=30, height=60, bd=2, relief='raised')
        self.root.update()
        return t, f, f2

    def test_place_configure_in(self):
        t, f, f2 = self.create2()
        self.assertEqual(f2.winfo_manager(), '')
        with self.assertRaisesRegex(
                TclError,
                """can't place "?%s"? relative to itself"""
                 % re.escape(str(f2))):
            f2.place_configure(in_=f2)
        self.assertEqual(f2.winfo_manager(), '')
        with self.assertRaisesRegex(TclError, 'bad window path name'):
            f2.place_configure(in_='spam')
        f2.place_configure(in_=f)
        self.assertEqual(f2.winfo_manager(), 'place')

    def test_place_configure_x(self):
        t, f, f2 = self.create2()
        f2.place_configure(in_=f)
        self.assertEqual(f2.place_info()['x'], '0')
        self.root.update()
        self.assertEqual(f2.winfo_x(), 50)
        f2.place_configure(x=100)
        self.assertEqual(f2.place_info()['x'], '100')
        self.root.update()
        self.assertEqual(f2.winfo_x(), 150)
        f2.place_configure(x=-10, relx=1)
        self.assertEqual(f2.place_info()['x'], '-10')
        self.root.update()
        self.assertEqual(f2.winfo_x(), 190)
        with self.assertRaisesRegex(TclError,
                EXPECTED_SCREEN_DISTANCE_ERRMSG.format('spam')):
            f2.place_configure(in_=f, x='spam')

    def test_place_configure_y(self):
        t, f, f2 = self.create2()
        f2.place_configure(in_=f)
        self.assertEqual(f2.place_info()['y'], '0')
        self.root.update()
        self.assertEqual(f2.winfo_y(), 40)
        f2.place_configure(y=50)
        self.assertEqual(f2.place_info()['y'], '50')
        self.root.update()
        self.assertEqual(f2.winfo_y(), 90)
        f2.place_configure(y=-10, rely=1)
        self.assertEqual(f2.place_info()['y'], '-10')
        self.root.update()
        self.assertEqual(f2.winfo_y(), 110)
        with self.assertRaisesRegex(TclError,
                EXPECTED_SCREEN_DISTANCE_ERRMSG.format('spam')):
            f2.place_configure(in_=f, y='spam')

    def test_place_configure_relx(self):
        t, f, f2 = self.create2()
        f2.place_configure(in_=f)
        self.assertEqual(f2.place_info()['relx'], '0')
        self.root.update()
        self.assertEqual(f2.winfo_x(), 50)
        f2.place_configure(relx=0.5)
        self.assertEqual(f2.place_info()['relx'], '0.5')
        self.root.update()
        self.assertEqual(f2.winfo_x(), 125)
        f2.place_configure(relx=1)
        self.assertEqual(f2.place_info()['relx'], '1')
        self.root.update()
        self.assertEqual(f2.winfo_x(), 200)
        with self.assertRaisesRegex(TclError, EXPECTED_FLOAT_ERRMSG.format('spam')):
            f2.place_configure(in_=f, relx='spam')

    def test_place_configure_rely(self):
        t, f, f2 = self.create2()
        f2.place_configure(in_=f)
        self.assertEqual(f2.place_info()['rely'], '0')
        self.root.update()
        self.assertEqual(f2.winfo_y(), 40)
        f2.place_configure(rely=0.5)
        self.assertEqual(f2.place_info()['rely'], '0.5')
        self.root.update()
        self.assertEqual(f2.winfo_y(), 80)
        f2.place_configure(rely=1)
        self.assertEqual(f2.place_info()['rely'], '1')
        self.root.update()
        self.assertEqual(f2.winfo_y(), 120)
        with self.assertRaisesRegex(TclError, EXPECTED_FLOAT_ERRMSG.format('spam')):
            f2.place_configure(in_=f, rely='spam')

    def test_place_configure_anchor(self):
        f = tkinter.Frame(self.root)
        with self.assertRaisesRegex(TclError, 'bad anchor "j"'):
            f.place_configure(anchor='j')
        with self.assertRaisesRegex(TclError, 'ambiguous anchor ""'):
            f.place_configure(anchor='')
        for value in 'n', 'ne', 'e', 'se', 's', 'sw', 'w', 'nw', 'center':
            f.place_configure(anchor=value)
            self.assertEqual(f.place_info()['anchor'], value)

    def test_place_configure_width(self):
        t, f, f2 = self.create2()
        f2.place_configure(in_=f, width=120)
        self.root.update()
        self.assertEqual(f2.winfo_width(), 120)
        f2.place_configure(width='')
        self.root.update()
        self.assertEqual(f2.winfo_width(), 30)
        with self.assertRaisesRegex(TclError,
                EXPECTED_SCREEN_DISTANCE_OR_EMPTY_ERRMSG.format('abcd')):
            f2.place_configure(width='abcd')

    def test_place_configure_height(self):
        t, f, f2 = self.create2()
        f2.place_configure(in_=f, height=120)
        self.root.update()
        self.assertEqual(f2.winfo_height(), 120)
        f2.place_configure(height='')
        self.root.update()
        self.assertEqual(f2.winfo_height(), 60)
        with self.assertRaisesRegex(TclError,
                EXPECTED_SCREEN_DISTANCE_OR_EMPTY_ERRMSG.format('abcd')):
            f2.place_configure(height='abcd')

    def test_place_configure_relwidth(self):
        t, f, f2 = self.create2()
        f2.place_configure(in_=f, relwidth=0.5)
        self.root.update()
        self.assertEqual(f2.winfo_width(), 75)
        f2.place_configure(relwidth='')
        self.root.update()
        self.assertEqual(f2.winfo_width(), 30)
        with self.assertRaisesRegex(TclError, EXPECTED_FLOAT_OR_EMPTY_ERRMSG.format('abcd')):
            f2.place_configure(relwidth='abcd')

    def test_place_configure_relheight(self):
        t, f, f2 = self.create2()
        f2.place_configure(in_=f, relheight=0.5)
        self.root.update()
        self.assertEqual(f2.winfo_height(), 40)
        f2.place_configure(relheight='')
        self.root.update()
        self.assertEqual(f2.winfo_height(), 60)
        with self.assertRaisesRegex(TclError, EXPECTED_FLOAT_OR_EMPTY_ERRMSG.format('abcd')):
            f2.place_configure(relheight='abcd')

    def test_place_configure_bordermode(self):
        f = tkinter.Frame(self.root)
        with self.assertRaisesRegex(TclError, 'bad bordermode "j"'):
            f.place_configure(bordermode='j')
        with self.assertRaisesRegex(TclError, 'ambiguous bordermode ""'):
            f.place_configure(bordermode='')
        for value in 'inside', 'outside', 'ignore':
            f.place_configure(bordermode=value)
            self.assertEqual(f.place_info()['bordermode'], value)

    def test_place_forget(self):
        foo = tkinter.Frame(self.root)
        foo.place_configure(width=50, height=50)
        self.root.update()
        foo.place_forget()
        self.root.update()
        self.assertFalse(foo.winfo_ismapped())
        with self.assertRaises(TypeError):
            foo.place_forget(0)

    def test_place_info(self):
        t, f, f2 = self.create2()
        f2.place_configure(in_=f, x=1, y=2, width=3, height=4,
                           relx=0.1, rely=0.2, relwidth=0.3, relheight=0.4,
                           anchor='se', bordermode='outside')
        info = f2.place_info()
        self.assertIsInstance(info, dict)
        self.assertEqual(info['x'], '1')
        self.assertEqual(info['y'], '2')
        self.assertEqual(info['width'], '3')
        self.assertEqual(info['height'], '4')
        self.assertEqual(info['relx'], '0.1')
        self.assertEqual(info['rely'], '0.2')
        self.assertEqual(info['relwidth'], '0.3')
        self.assertEqual(info['relheight'], '0.4')
        self.assertEqual(info['anchor'], 'se')
        self.assertEqual(info['bordermode'], 'outside')
        self.assertEqual(info['x'], '1')
        self.assertEqual(info['x'], '1')
        with self.assertRaises(TypeError):
            f2.place_info(0)

    def test_place_slaves(self):
        foo = tkinter.Frame(self.root)
        bar = tkinter.Frame(self.root)
        self.assertEqual(foo.place_slaves(), [])
        bar.place_configure(in_=foo)
        self.assertEqual(foo.place_slaves(), [bar])
        with self.assertRaises(TypeError):
            foo.place_slaves(0)


class GridTest(AbstractWidgetTest, unittest.TestCase):

    test_keys = None

    def tearDown(self):
        cols, rows = self.root.grid_size()
        for i in range(cols + 1):
            self.root.grid_columnconfigure(i, weight=0, minsize=0, pad=0, uniform='')
        for i in range(rows + 1):
            self.root.grid_rowconfigure(i, weight=0, minsize=0, pad=0, uniform='')
        self.root.grid_propagate(1)
        self.root.grid_anchor('nw')
        super().tearDown()

    def test_grid_configure(self):
        b = tkinter.Button(self.root)
        self.assertEqual(b.grid_info(), {})
        b.grid_configure()
        self.assertEqual(b.grid_info()['in'], self.root)
        self.assertEqual(b.grid_info()['column'], self._str(0))
        self.assertEqual(b.grid_info()['row'], self._str(0))
        b.grid_configure({'column': 1}, row=2)
        self.assertEqual(b.grid_info()['column'], self._str(1))
        self.assertEqual(b.grid_info()['row'], self._str(2))

    def test_grid_configure_column(self):
        b = tkinter.Button(self.root)
        with self.assertRaisesRegex(TclError, 'bad column value "-1": '
                                    'must be a non-negative integer'):
            b.grid_configure(column=-1)
        b.grid_configure(column=2)
        self.assertEqual(b.grid_info()['column'], self._str(2))

    def test_grid_configure_columnspan(self):
        b = tkinter.Button(self.root)
        with self.assertRaisesRegex(TclError, 'bad columnspan value "0": '
                                    'must be a positive integer'):
            b.grid_configure(columnspan=0)
        b.grid_configure(columnspan=2)
        self.assertEqual(b.grid_info()['columnspan'], self._str(2))

    def test_grid_configure_in(self):
        f = tkinter.Frame(self.root)
        b = tkinter.Button(self.root)
        self.assertEqual(b.grid_info(), {})
        b.grid_configure()
        self.assertEqual(b.grid_info()['in'], self.root)
        b.grid_configure(in_=f)
        self.assertEqual(b.grid_info()['in'], f)
        b.grid_configure({'in': self.root})
        self.assertEqual(b.grid_info()['in'], self.root)

    def test_grid_configure_ipadx(self):
        b = tkinter.Button(self.root)
        with self.assertRaisesRegex(TclError, 'bad ipadx value "-1": '
                                    'must be positive screen distance'):
            b.grid_configure(ipadx=-1)
        b.grid_configure(ipadx=1)
        self.assertEqual(b.grid_info()['ipadx'], self._str(1))
        b.grid_configure(ipadx='.5c')
        self.assertEqual(b.grid_info()['ipadx'],
                         self._str(round(pixels_conv('.5c') * self.scaling)))

    def test_grid_configure_ipady(self):
        b = tkinter.Button(self.root)
        with self.assertRaisesRegex(TclError, 'bad ipady value "-1": '
                                    'must be positive screen distance'):
            b.grid_configure(ipady=-1)
        b.grid_configure(ipady=1)
        self.assertEqual(b.grid_info()['ipady'], self._str(1))
        b.grid_configure(ipady='.5c')
        self.assertEqual(b.grid_info()['ipady'],
                         self._str(round(pixels_conv('.5c') * self.scaling)))

    def test_grid_configure_padx(self):
        b = tkinter.Button(self.root)
        with self.assertRaisesRegex(TclError, 'bad pad value "-1": '
                                    'must be positive screen distance'):
            b.grid_configure(padx=-1)
        b.grid_configure(padx=1)
        self.assertEqual(b.grid_info()['padx'], self._str(1))
        b.grid_configure(padx=(10, 5))
        self.assertEqual(b.grid_info()['padx'], self._str((10, 5)))
        b.grid_configure(padx='.5c')
        self.assertEqual(b.grid_info()['padx'],
                         self._str(round(pixels_conv('.5c') * self.scaling)))

    def test_grid_configure_pady(self):
        b = tkinter.Button(self.root)
        with self.assertRaisesRegex(TclError, 'bad pad value "-1": '
                                    'must be positive screen distance'):
            b.grid_configure(pady=-1)
        b.grid_configure(pady=1)
        self.assertEqual(b.grid_info()['pady'], self._str(1))
        b.grid_configure(pady=(10, 5))
        self.assertEqual(b.grid_info()['pady'], self._str((10, 5)))
        b.grid_configure(pady='.5c')
        self.assertEqual(b.grid_info()['pady'],
                         self._str(round(pixels_conv('.5c') * self.scaling)))

    def test_grid_configure_row(self):
        b = tkinter.Button(self.root)
        with self.assertRaisesRegex(TclError, 'bad (row|grid) value "-1": '
                                    'must be a non-negative integer'):
            b.grid_configure(row=-1)
        b.grid_configure(row=2)
        self.assertEqual(b.grid_info()['row'], self._str(2))

    def test_grid_configure_rownspan(self):
        b = tkinter.Button(self.root)
        with self.assertRaisesRegex(TclError, 'bad rowspan value "0": '
                                    'must be a positive integer'):
            b.grid_configure(rowspan=0)
        b.grid_configure(rowspan=2)
        self.assertEqual(b.grid_info()['rowspan'], self._str(2))

    def test_grid_configure_sticky(self):
        f = tkinter.Frame(self.root, bg='red')
        with self.assertRaisesRegex(TclError, 'bad stickyness value "glue"'):
            f.grid_configure(sticky='glue')
        f.grid_configure(sticky='ne')
        self.assertEqual(f.grid_info()['sticky'], 'ne')
        f.grid_configure(sticky='n,s,e,w')
        self.assertEqual(f.grid_info()['sticky'], 'nesw')

    def test_grid_columnconfigure(self):
        with self.assertRaises(TypeError):
            self.root.grid_columnconfigure()
        self.assertEqual(self.root.grid_columnconfigure(0),
                         {'minsize': 0, 'pad': 0, 'uniform': None, 'weight': 0})
        with self.assertRaisesRegex(TclError, 'bad option "-foo"'):
            self.root.grid_columnconfigure(0, 'foo')
        self.root.grid_columnconfigure((0, 3), weight=2)
        with self.assertRaisesRegex(TclError,
                                    'must specify a single element on retrieval'):
            self.root.grid_columnconfigure((0, 3))
        b = tkinter.Button(self.root)
        b.grid_configure(column=0, row=0)
        self.root.grid_columnconfigure('all', weight=3)
        with self.assertRaisesRegex(TclError, 'expected integer but got "all"'):
            self.root.grid_columnconfigure('all')
        self.assertEqual(self.root.grid_columnconfigure(0, 'weight'), 3)
        self.assertEqual(self.root.grid_columnconfigure(3, 'weight'), 2)
        self.assertEqual(self.root.grid_columnconfigure(265, 'weight'), 0)
        self.root.grid_columnconfigure(b, weight=4)
        self.assertEqual(self.root.grid_columnconfigure(0, 'weight'), 4)

    def test_grid_columnconfigure_minsize(self):
        with self.assertRaisesRegex(TclError,
                EXPECTED_SCREEN_DISTANCE_ERRMSG.format('foo')):
            self.root.grid_columnconfigure(0, minsize='foo')
        self.root.grid_columnconfigure(0, minsize=10)
        self.assertEqual(self.root.grid_columnconfigure(0, 'minsize'), 10)
        self.assertEqual(self.root.grid_columnconfigure(0)['minsize'], 10)

    def test_grid_columnconfigure_weight(self):
        with self.assertRaisesRegex(TclError, 'expected integer but got "bad"'):
            self.root.grid_columnconfigure(0, weight='bad')
        with self.assertRaisesRegex(TclError, 'invalid arg "-weight": '
                                    'should be non-negative'):
            self.root.grid_columnconfigure(0, weight=-3)
        self.root.grid_columnconfigure(0, weight=3)
        self.assertEqual(self.root.grid_columnconfigure(0, 'weight'), 3)
        self.assertEqual(self.root.grid_columnconfigure(0)['weight'], 3)

    def test_grid_columnconfigure_pad(self):
        with self.assertRaisesRegex(TclError,
                EXPECTED_SCREEN_DISTANCE_ERRMSG.format('foo')):
            self.root.grid_columnconfigure(0, pad='foo')
        with self.assertRaisesRegex(TclError, 'invalid arg "-pad": '
                                    'should be non-negative'):
            self.root.grid_columnconfigure(0, pad=-3)
        self.root.grid_columnconfigure(0, pad=3)
        self.assertEqual(self.root.grid_columnconfigure(0, 'pad'), 3)
        self.assertEqual(self.root.grid_columnconfigure(0)['pad'], 3)

    def test_grid_columnconfigure_uniform(self):
        self.root.grid_columnconfigure(0, uniform='foo')
        self.assertEqual(self.root.grid_columnconfigure(0, 'uniform'), 'foo')
        self.assertEqual(self.root.grid_columnconfigure(0)['uniform'], 'foo')

    def test_grid_rowconfigure(self):
        with self.assertRaises(TypeError):
            self.root.grid_rowconfigure()
        self.assertEqual(self.root.grid_rowconfigure(0),
                         {'minsize': 0, 'pad': 0, 'uniform': None, 'weight': 0})
        with self.assertRaisesRegex(TclError, 'bad option "-foo"'):
            self.root.grid_rowconfigure(0, 'foo')
        self.root.grid_rowconfigure((0, 3), weight=2)
        with self.assertRaisesRegex(TclError,
                                    'must specify a single element on retrieval'):
            self.root.grid_rowconfigure((0, 3))
        b = tkinter.Button(self.root)
        b.grid_configure(column=0, row=0)
        self.root.grid_rowconfigure('all', weight=3)
        with self.assertRaisesRegex(TclError, 'expected integer but got "all"'):
            self.root.grid_rowconfigure('all')
        self.assertEqual(self.root.grid_rowconfigure(0, 'weight'), 3)
        self.assertEqual(self.root.grid_rowconfigure(3, 'weight'), 2)
        self.assertEqual(self.root.grid_rowconfigure(265, 'weight'), 0)
        self.root.grid_rowconfigure(b, weight=4)
        self.assertEqual(self.root.grid_rowconfigure(0, 'weight'), 4)

    def test_grid_rowconfigure_minsize(self):
        with self.assertRaisesRegex(TclError,
                EXPECTED_SCREEN_DISTANCE_ERRMSG.format('foo')):
            self.root.grid_rowconfigure(0, minsize='foo')
        self.root.grid_rowconfigure(0, minsize=10)
        self.assertEqual(self.root.grid_rowconfigure(0, 'minsize'), 10)
        self.assertEqual(self.root.grid_rowconfigure(0)['minsize'], 10)

    def test_grid_rowconfigure_weight(self):
        with self.assertRaisesRegex(TclError, 'expected integer but got "bad"'):
            self.root.grid_rowconfigure(0, weight='bad')
        with self.assertRaisesRegex(TclError, 'invalid arg "-weight": '
                                    'should be non-negative'):
            self.root.grid_rowconfigure(0, weight=-3)
        self.root.grid_rowconfigure(0, weight=3)
        self.assertEqual(self.root.grid_rowconfigure(0, 'weight'), 3)
        self.assertEqual(self.root.grid_rowconfigure(0)['weight'], 3)

    def test_grid_rowconfigure_pad(self):
        with self.assertRaisesRegex(TclError,
                EXPECTED_SCREEN_DISTANCE_ERRMSG.format('foo')):
            self.root.grid_rowconfigure(0, pad='foo')
        with self.assertRaisesRegex(TclError, 'invalid arg "-pad": '
                                    'should be non-negative'):
            self.root.grid_rowconfigure(0, pad=-3)
        self.root.grid_rowconfigure(0, pad=3)
        self.assertEqual(self.root.grid_rowconfigure(0, 'pad'), 3)
        self.assertEqual(self.root.grid_rowconfigure(0)['pad'], 3)

    def test_grid_rowconfigure_uniform(self):
        self.root.grid_rowconfigure(0, uniform='foo')
        self.assertEqual(self.root.grid_rowconfigure(0, 'uniform'), 'foo')
        self.assertEqual(self.root.grid_rowconfigure(0)['uniform'], 'foo')

    def test_grid_forget(self):
        b = tkinter.Button(self.root)
        c = tkinter.Button(self.root)
        b.grid_configure(row=2, column=2, rowspan=2, columnspan=2,
                         padx=3, pady=4, sticky='ns')
        self.assertEqual(self.root.grid_slaves(), [b])
        b.grid_forget()
        c.grid_forget()
        self.assertEqual(self.root.grid_slaves(), [])
        self.assertEqual(b.grid_info(), {})
        b.grid_configure(row=0, column=0)
        info = b.grid_info()
        self.assertEqual(info['row'], self._str(0))
        self.assertEqual(info['column'], self._str(0))
        self.assertEqual(info['rowspan'], self._str(1))
        self.assertEqual(info['columnspan'], self._str(1))
        self.assertEqual(info['padx'], self._str(0))
        self.assertEqual(info['pady'], self._str(0))
        self.assertEqual(info['sticky'], '')

    def test_grid_remove(self):
        b = tkinter.Button(self.root)
        c = tkinter.Button(self.root)
        b.grid_configure(row=2, column=2, rowspan=2, columnspan=2,
                         padx=3, pady=4, sticky='ns')
        self.assertEqual(self.root.grid_slaves(), [b])
        b.grid_remove()
        c.grid_remove()
        self.assertEqual(self.root.grid_slaves(), [])
        self.assertEqual(b.grid_info(), {})
        b.grid_configure(row=0, column=0)
        info = b.grid_info()
        self.assertEqual(info['row'], self._str(0))
        self.assertEqual(info['column'], self._str(0))
        self.assertEqual(info['rowspan'], self._str(2))
        self.assertEqual(info['columnspan'], self._str(2))
        self.assertEqual(info['padx'], self._str(3))
        self.assertEqual(info['pady'], self._str(4))
        self.assertEqual(info['sticky'], 'ns')

    def test_grid_info(self):
        b = tkinter.Button(self.root)
        self.assertEqual(b.grid_info(), {})
        b.grid_configure(row=2, column=2, rowspan=2, columnspan=2,
                         padx=3, pady=4, sticky='ns')
        info = b.grid_info()
        self.assertIsInstance(info, dict)
        self.assertEqual(info['in'], self.root)
        self.assertEqual(info['row'], self._str(2))
        self.assertEqual(info['column'], self._str(2))
        self.assertEqual(info['rowspan'], self._str(2))
        self.assertEqual(info['columnspan'], self._str(2))
        self.assertEqual(info['padx'], self._str(3))
        self.assertEqual(info['pady'], self._str(4))
        self.assertEqual(info['sticky'], 'ns')

    def test_grid_anchor(self):
        with self.assertRaisesRegex(TclError, 'bad anchor "x"'):
            self.root.grid_anchor('x')
        with self.assertRaisesRegex(TclError, 'ambiguous anchor ""'):
            self.root.grid_anchor('')
        with self.assertRaises(TypeError):
            self.root.grid_anchor('se', 'nw')
        self.root.grid_anchor('se')
        self.assertEqual(self.root.tk.call('grid', 'anchor', self.root), 'se')

    def test_grid_bbox(self):
        self.assertEqual(self.root.grid_bbox(), (0, 0, 0, 0))
        self.assertEqual(self.root.grid_bbox(0, 0), (0, 0, 0, 0))
        self.assertEqual(self.root.grid_bbox(0, 0, 1, 1), (0, 0, 0, 0))
        with self.assertRaisesRegex(TclError, 'expected integer but got "x"'):
            self.root.grid_bbox('x', 0)
        with self.assertRaisesRegex(TclError, 'expected integer but got "x"'):
            self.root.grid_bbox(0, 'x')
        with self.assertRaisesRegex(TclError, 'expected integer but got "x"'):
            self.root.grid_bbox(0, 0, 'x', 0)
        with self.assertRaisesRegex(TclError, 'expected integer but got "x"'):
            self.root.grid_bbox(0, 0, 0, 'x')
        with self.assertRaises(TypeError):
            self.root.grid_bbox(0, 0, 0, 0, 0)
        t = self.root
        # de-maximize
        t.wm_geometry('1x1+0+0')
        t.wm_geometry('')
        f1 = tkinter.Frame(t, width=75, height=75, bg='red')
        f2 = tkinter.Frame(t, width=90, height=90, bg='blue')
        f1.grid_configure(row=0, column=0)
        f2.grid_configure(row=1, column=1)
        self.root.update()
        self.assertEqual(t.grid_bbox(), (0, 0, 165, 165))
        self.assertEqual(t.grid_bbox(0, 0), (0, 0, 75, 75))
        self.assertEqual(t.grid_bbox(0, 0, 1, 1), (0, 0, 165, 165))
        self.assertEqual(t.grid_bbox(1, 1), (75, 75, 90, 90))
        self.assertEqual(t.grid_bbox(10, 10, 0, 0), (0, 0, 165, 165))
        self.assertEqual(t.grid_bbox(-2, -2, -1, -1), (0, 0, 0, 0))
        self.assertEqual(t.grid_bbox(10, 10, 12, 12), (165, 165, 0, 0))

    def test_grid_location(self):
        with self.assertRaises(TypeError):
            self.root.grid_location()
        with self.assertRaises(TypeError):
            self.root.grid_location(0)
        with self.assertRaises(TypeError):
            self.root.grid_location(0, 0, 0)
        with self.assertRaisesRegex(TclError,
                EXPECTED_SCREEN_DISTANCE_ERRMSG.format('x')):
            self.root.grid_location('x', 'y')
        with self.assertRaisesRegex(TclError,
                EXPECTED_SCREEN_DISTANCE_ERRMSG.format('y')):
            self.root.grid_location('1c', 'y')
        t = self.root
        # de-maximize
        t.wm_geometry('1x1+0+0')
        t.wm_geometry('')
        f = tkinter.Frame(t, width=200, height=100,
                          highlightthickness=0, bg='red')
        self.assertEqual(f.grid_location(10, 10), (-1, -1))
        f.grid_configure()
        self.root.update()
        self.assertEqual(t.grid_location(-10, -10), (-1, -1))
        self.assertEqual(t.grid_location(-10, 0), (-1, 0))
        self.assertEqual(t.grid_location(-1, 0), (-1, 0))
        self.assertEqual(t.grid_location(0, -10), (0, -1))
        self.assertEqual(t.grid_location(0, -1), (0, -1))
        self.assertEqual(t.grid_location(0, 0), (0, 0))
        self.assertEqual(t.grid_location(200, 0), (0, 0))
        self.assertEqual(t.grid_location(201, 0), (1, 0))
        self.assertEqual(t.grid_location(0, 100), (0, 0))
        self.assertEqual(t.grid_location(0, 101), (0, 1))
        self.assertEqual(t.grid_location(201, 101), (1, 1))

    def test_grid_propagate(self):
        self.assertEqual(self.root.grid_propagate(), True)
        with self.assertRaises(TypeError):
            self.root.grid_propagate(False, False)
        self.root.grid_propagate(False)
        self.assertFalse(self.root.grid_propagate())
        f = tkinter.Frame(self.root, width=100, height=100, bg='red')
        f.grid_configure(row=0, column=0)
        self.root.update()
        self.assertEqual(f.winfo_width(), 100)
        self.assertEqual(f.winfo_height(), 100)
        f.grid_propagate(False)
        g = tkinter.Frame(self.root, width=75, height=85, bg='green')
        g.grid_configure(in_=f, row=0, column=0)
        self.root.update()
        self.assertEqual(f.winfo_width(), 100)
        self.assertEqual(f.winfo_height(), 100)
        f.grid_propagate(True)
        self.root.update()
        self.assertEqual(f.winfo_width(), 75)
        self.assertEqual(f.winfo_height(), 85)

    def test_grid_size(self):
        with self.assertRaises(TypeError):
            self.root.grid_size(0)
        self.assertEqual(self.root.grid_size(), (0, 0))
        f = tkinter.Scale(self.root)
        f.grid_configure(row=0, column=0)
        self.assertEqual(self.root.grid_size(), (1, 1))
        f.grid_configure(row=4, column=5)
        self.assertEqual(self.root.grid_size(), (6, 5))

    def test_grid_slaves(self):
        self.assertEqual(self.root.grid_slaves(), [])
        a = tkinter.Label(self.root)
        a.grid_configure(row=0, column=1)
        b = tkinter.Label(self.root)
        b.grid_configure(row=1, column=0)
        c = tkinter.Label(self.root)
        c.grid_configure(row=1, column=1)
        d = tkinter.Label(self.root)
        d.grid_configure(row=1, column=1)
        self.assertEqual(self.root.grid_slaves(), [d, c, b, a])
        self.assertEqual(self.root.grid_slaves(row=0), [a])
        self.assertEqual(self.root.grid_slaves(row=1), [d, c, b])
        self.assertEqual(self.root.grid_slaves(column=0), [b])
        self.assertEqual(self.root.grid_slaves(column=1), [d, c, a])
        self.assertEqual(self.root.grid_slaves(row=1, column=1), [d, c])


if __name__ == '__main__':
    unittest.main()

### File: test_images.py

In [ ]:
import unittest
import tkinter
from test import support
from test.support import os_helper
from test.test_tkinter.support import AbstractTkTest, AbstractDefaultRootTest, requires_tk

support.requires('gui')


class MiscTest(AbstractTkTest, unittest.TestCase):

    def test_image_types(self):
        image_types = self.root.image_types()
        self.assertIsInstance(image_types, tuple)
        self.assertIn('photo', image_types)
        self.assertIn('bitmap', image_types)

    def test_image_names(self):
        image_names = self.root.image_names()
        self.assertIsInstance(image_names, tuple)


class DefaultRootTest(AbstractDefaultRootTest, unittest.TestCase):

    def test_image_types(self):
        self.assertRaises(RuntimeError, tkinter.image_types)
        root = tkinter.Tk()
        image_types = tkinter.image_types()
        self.assertIsInstance(image_types, tuple)
        self.assertIn('photo', image_types)
        self.assertIn('bitmap', image_types)
        root.destroy()
        tkinter.NoDefaultRoot()
        self.assertRaises(RuntimeError, tkinter.image_types)

    def test_image_names(self):
        self.assertRaises(RuntimeError, tkinter.image_names)
        root = tkinter.Tk()
        image_names = tkinter.image_names()
        self.assertIsInstance(image_names, tuple)
        root.destroy()
        tkinter.NoDefaultRoot()
        self.assertRaises(RuntimeError, tkinter.image_names)

    def test_image_create_bitmap(self):
        self.assertRaises(RuntimeError, tkinter.BitmapImage)
        root = tkinter.Tk()
        image = tkinter.BitmapImage()
        self.assertIn(image.name, tkinter.image_names())
        root.destroy()
        tkinter.NoDefaultRoot()
        self.assertRaises(RuntimeError, tkinter.BitmapImage)

    def test_image_create_photo(self):
        self.assertRaises(RuntimeError, tkinter.PhotoImage)
        root = tkinter.Tk()
        image = tkinter.PhotoImage()
        self.assertIn(image.name, tkinter.image_names())
        root.destroy()
        tkinter.NoDefaultRoot()
        self.assertRaises(RuntimeError, tkinter.PhotoImage)


class BitmapImageTest(AbstractTkTest, unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        AbstractTkTest.setUpClass.__func__(cls)
        cls.testfile = support.findfile('python.xbm', subdir='tkinterdata')

    def test_create_from_file(self):
        image = tkinter.BitmapImage('::img::test', master=self.root,
                                    foreground='yellow', background='blue',
                                    file=self.testfile)
        self.assertEqual(str(image), '::img::test')
        self.assertEqual(image.type(), 'bitmap')
        self.assertEqual(image.width(), 16)
        self.assertEqual(image.height(), 16)
        self.assertIn('::img::test', self.root.image_names())
        del image
        support.gc_collect()  # For PyPy or other GCs.
        self.assertNotIn('::img::test', self.root.image_names())

    def test_create_from_data(self):
        with open(self.testfile, 'rb') as f:
            data = f.read()
        image = tkinter.BitmapImage('::img::test', master=self.root,
                                    foreground='yellow', background='blue',
                                    data=data)
        self.assertEqual(str(image), '::img::test')
        self.assertEqual(image.type(), 'bitmap')
        self.assertEqual(image.width(), 16)
        self.assertEqual(image.height(), 16)
        self.assertIn('::img::test', self.root.image_names())
        del image
        support.gc_collect()  # For PyPy or other GCs.
        self.assertNotIn('::img::test', self.root.image_names())

    def assertEqualStrList(self, actual, expected):
        self.assertIsInstance(actual, str)
        self.assertEqual(self.root.splitlist(actual), expected)

    def test_configure_data(self):
        image = tkinter.BitmapImage('::img::test', master=self.root)
        self.assertEqual(image['data'], '-data {} {} {} {}')
        with open(self.testfile, 'rb') as f:
            data = f.read()
        image.configure(data=data)
        self.assertEqualStrList(image['data'],
                                ('-data', '', '', '', data.decode('ascii')))
        self.assertEqual(image.width(), 16)
        self.assertEqual(image.height(), 16)

        self.assertEqual(image['maskdata'], '-maskdata {} {} {} {}')
        image.configure(maskdata=data)
        self.assertEqualStrList(image['maskdata'],
                                ('-maskdata', '', '', '', data.decode('ascii')))

    def test_configure_file(self):
        image = tkinter.BitmapImage('::img::test', master=self.root)
        self.assertEqual(image['file'], '-file {} {} {} {}')
        image.configure(file=self.testfile)
        self.assertEqualStrList(image['file'],
                                ('-file', '', '', '',self.testfile))
        self.assertEqual(image.width(), 16)
        self.assertEqual(image.height(), 16)

        self.assertEqual(image['maskfile'], '-maskfile {} {} {} {}')
        image.configure(maskfile=self.testfile)
        self.assertEqualStrList(image['maskfile'],
                                ('-maskfile', '', '', '', self.testfile))

    def test_configure_background(self):
        image = tkinter.BitmapImage('::img::test', master=self.root)
        self.assertEqual(image['background'], '-background {} {} {} {}')
        image.configure(background='blue')
        self.assertEqual(image['background'], '-background {} {} {} blue')

    def test_configure_foreground(self):
        image = tkinter.BitmapImage('::img::test', master=self.root)
        self.assertEqual(image['foreground'],
                         '-foreground {} {} #000000 #000000')
        image.configure(foreground='yellow')
        self.assertEqual(image['foreground'],
                         '-foreground {} {} #000000 yellow')

    def test_bug_100814(self):
        # gh-100814: Passing a callable option value causes AttributeError.
        with self.assertRaises(tkinter.TclError):
            tkinter.BitmapImage('::img::test', master=self.root, spam=print)
        image = tkinter.BitmapImage('::img::test', master=self.root)
        with self.assertRaises(tkinter.TclError):
            image.configure(spam=print)


class PhotoImageTest(AbstractTkTest, unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        AbstractTkTest.setUpClass.__func__(cls)
        cls.testfile = support.findfile('python.gif', subdir='tkinterdata')

    def create(self):
        return tkinter.PhotoImage('::img::test', master=self.root,
                                  file=self.testfile)

    def colorlist(self, *args):
        if tkinter.TkVersion >= 8.6 and self.wantobjects:
            return args
        else:
            return tkinter._join(args)

    def check_create_from_file(self, ext):
        testfile = support.findfile('python.' + ext, subdir='tkinterdata')
        image = tkinter.PhotoImage('::img::test', master=self.root,
                                   file=testfile)
        self.assertEqual(str(image), '::img::test')
        self.assertEqual(image.type(), 'photo')
        self.assertEqual(image.width(), 16)
        self.assertEqual(image.height(), 16)
        self.assertEqual(image['data'], '')
        self.assertEqual(image['file'], testfile)
        self.assertIn('::img::test', self.root.image_names())
        del image
        support.gc_collect()  # For PyPy or other GCs.
        self.assertNotIn('::img::test', self.root.image_names())

    def check_create_from_data(self, ext):
        testfile = support.findfile('python.' + ext, subdir='tkinterdata')
        with open(testfile, 'rb') as f:
            data = f.read()
        image = tkinter.PhotoImage('::img::test', master=self.root,
                                   data=data)
        self.assertEqual(str(image), '::img::test')
        self.assertEqual(image.type(), 'photo')
        self.assertEqual(image.width(), 16)
        self.assertEqual(image.height(), 16)
        self.assertEqual(image['data'], data if self.wantobjects
                                        else data.decode('latin1'))
        self.assertEqual(image['file'], '')
        self.assertIn('::img::test', self.root.image_names())
        del image
        support.gc_collect()  # For PyPy or other GCs.
        self.assertNotIn('::img::test', self.root.image_names())

    def test_create_from_ppm_file(self):
        self.check_create_from_file('ppm')

    def test_create_from_ppm_data(self):
        self.check_create_from_data('ppm')

    def test_create_from_pgm_file(self):
        self.check_create_from_file('pgm')

    def test_create_from_pgm_data(self):
        self.check_create_from_data('pgm')

    def test_create_from_gif_file(self):
        self.check_create_from_file('gif')

    def test_create_from_gif_data(self):
        self.check_create_from_data('gif')

    @requires_tk(8, 6)
    def test_create_from_png_file(self):
        self.check_create_from_file('png')

    @requires_tk(8, 6)
    def test_create_from_png_data(self):
        self.check_create_from_data('png')

    def test_configure_data(self):
        image = tkinter.PhotoImage('::img::test', master=self.root)
        self.assertEqual(image['data'], '')
        with open(self.testfile, 'rb') as f:
            data = f.read()
        image.configure(data=data)
        self.assertEqual(image['data'], data if self.wantobjects
                                        else data.decode('latin1'))
        self.assertEqual(image.width(), 16)
        self.assertEqual(image.height(), 16)

    def test_configure_format(self):
        image = tkinter.PhotoImage('::img::test', master=self.root)
        self.assertEqual(image['format'], '')
        image.configure(file=self.testfile, format='gif')
        self.assertEqual(image['format'], ('gif',) if self.wantobjects
                                          else 'gif')
        self.assertEqual(image.width(), 16)
        self.assertEqual(image.height(), 16)

    def test_configure_file(self):
        image = tkinter.PhotoImage('::img::test', master=self.root)
        self.assertEqual(image['file'], '')
        image.configure(file=self.testfile)
        self.assertEqual(image['file'], self.testfile)
        self.assertEqual(image.width(), 16)
        self.assertEqual(image.height(), 16)

    def test_configure_gamma(self):
        image = tkinter.PhotoImage('::img::test', master=self.root)
        self.assertEqual(image['gamma'], '1.0')
        image.configure(gamma=2.0)
        self.assertEqual(image['gamma'], '2.0')

    def test_configure_width_height(self):
        image = tkinter.PhotoImage('::img::test', master=self.root)
        self.assertEqual(image['width'], '0')
        self.assertEqual(image['height'], '0')
        image.configure(width=20)
        image.configure(height=10)
        self.assertEqual(image['width'], '20')
        self.assertEqual(image['height'], '10')
        self.assertEqual(image.width(), 20)
        self.assertEqual(image.height(), 10)

    def test_configure_palette(self):
        image = tkinter.PhotoImage('::img::test', master=self.root)
        self.assertEqual(image['palette'], '')
        image.configure(palette=256)
        self.assertEqual(image['palette'], '256')
        image.configure(palette='3/4/2')
        self.assertEqual(image['palette'], '3/4/2')

    def test_bug_100814(self):
        # gh-100814: Passing a callable option value causes AttributeError.
        with self.assertRaises(tkinter.TclError):
            tkinter.PhotoImage('::img::test', master=self.root, spam=print)
        image = tkinter.PhotoImage('::img::test', master=self.root)
        with self.assertRaises(tkinter.TclError):
            image.configure(spam=print)

    def test_blank(self):
        image = self.create()
        image.blank()
        self.assertEqual(image.width(), 16)
        self.assertEqual(image.height(), 16)
        self.assertEqual(image.get(4, 6), self.colorlist(0, 0, 0))

    def test_copy(self):
        image = self.create()
        image2 = image.copy()
        self.assertEqual(image2.width(), 16)
        self.assertEqual(image2.height(), 16)
        self.assertEqual(image2.get(4, 6), image.get(4, 6))

        image2 = image.copy(from_coords=(2, 3, 14, 11))
        self.assertEqual(image2.width(), 12)
        self.assertEqual(image2.height(), 8)
        self.assertEqual(image2.get(0, 0), image.get(2, 3))
        self.assertEqual(image2.get(11, 7), image.get(13, 10))
        self.assertEqual(image2.get(2, 4), image.get(2+2, 4+3))

        image2 = image.copy(from_coords=(2, 3, 14, 11), zoom=2)
        self.assertEqual(image2.width(), 24)
        self.assertEqual(image2.height(), 16)
        self.assertEqual(image2.get(0, 0), image.get(2, 3))
        self.assertEqual(image2.get(23, 15), image.get(13, 10))
        self.assertEqual(image2.get(2*2, 4*2), image.get(2+2, 4+3))
        self.assertEqual(image2.get(2*2+1, 4*2+1), image.get(6+2, 2+3))

        image2 = image.copy(from_coords=(2, 3, 14, 11), subsample=2)
        self.assertEqual(image2.width(), 6)
        self.assertEqual(image2.height(), 4)
        self.assertEqual(image2.get(0, 0), image.get(2, 3))
        self.assertEqual(image2.get(5, 3), image.get(12, 9))
        self.assertEqual(image2.get(3, 2), image.get(3*2+2, 2*2+3))

        image2 = image.copy(from_coords=(2, 3, 14, 11), subsample=2, zoom=3)
        self.assertEqual(image2.width(), 18)
        self.assertEqual(image2.height(), 12)
        self.assertEqual(image2.get(0, 0), image.get(2, 3))
        self.assertEqual(image2.get(17, 11), image.get(12, 9))
        self.assertEqual(image2.get(1*3, 2*3), image.get(1*2+2, 2*2+3))
        self.assertEqual(image2.get(1*3+2, 2*3+2), image.get(1*2+2, 2*2+3))

    def test_subsample(self):
        image = self.create()
        image2 = image.subsample(2, 3)
        self.assertEqual(image2.width(), 8)
        self.assertEqual(image2.height(), 6)
        self.assertEqual(image2.get(2, 2), image.get(4, 6))

        image2 = image.subsample(2)
        self.assertEqual(image2.width(), 8)
        self.assertEqual(image2.height(), 8)
        self.assertEqual(image2.get(2, 3), image.get(4, 6))

        image2 = image.subsample(2, from_coords=(2, 3, 14, 11))
        self.assertEqual(image2.width(), 6)
        self.assertEqual(image2.height(), 4)
        self.assertEqual(image2.get(0, 0), image.get(2, 3))
        self.assertEqual(image2.get(5, 3), image.get(12, 9))
        self.assertEqual(image2.get(1, 2), image.get(1*2+2, 2*2+3))

    def test_zoom(self):
        image = self.create()
        image2 = image.zoom(2, 3)
        self.assertEqual(image2.width(), 32)
        self.assertEqual(image2.height(), 48)
        self.assertEqual(image2.get(8, 18), image.get(4, 6))
        self.assertEqual(image2.get(9, 20), image.get(4, 6))

        image2 = image.zoom(2)
        self.assertEqual(image2.width(), 32)
        self.assertEqual(image2.height(), 32)
        self.assertEqual(image2.get(8, 12), image.get(4, 6))
        self.assertEqual(image2.get(9, 13), image.get(4, 6))

        image2 = image.zoom(2, from_coords=(2, 3, 14, 11))
        self.assertEqual(image2.width(), 24)
        self.assertEqual(image2.height(), 16)
        self.assertEqual(image2.get(0, 0), image.get(2, 3))
        self.assertEqual(image2.get(23, 15), image.get(13, 10))
        self.assertEqual(image2.get(2*2, 4*2), image.get(2+2, 4+3))
        self.assertEqual(image2.get(2*2+1, 4*2+1), image.get(6+2, 2+3))

    def test_copy_replace(self):
        image = self.create()
        image2 = tkinter.PhotoImage(master=self.root)
        image2.copy_replace(image)
        self.assertEqual(image2.width(), 16)
        self.assertEqual(image2.height(), 16)
        self.assertEqual(image2.get(4, 6), image.get(4, 6))

        image2 = tkinter.PhotoImage(master=self.root)
        image2.copy_replace(image, from_coords=(2, 3, 14, 11))
        self.assertEqual(image2.width(), 12)
        self.assertEqual(image2.height(), 8)
        self.assertEqual(image2.get(0, 0), image.get(2, 3))
        self.assertEqual(image2.get(11, 7), image.get(13, 10))
        self.assertEqual(image2.get(2, 4), image.get(2+2, 4+3))

        image2 = tkinter.PhotoImage(master=self.root)
        image2.copy_replace(image)
        image2.copy_replace(image, from_coords=(2, 3, 14, 11), shrink=True)
        self.assertEqual(image2.width(), 12)
        self.assertEqual(image2.height(), 8)
        self.assertEqual(image2.get(0, 0), image.get(2, 3))
        self.assertEqual(image2.get(11, 7), image.get(13, 10))
        self.assertEqual(image2.get(2, 4), image.get(2+2, 4+3))

        image2 = tkinter.PhotoImage(master=self.root)
        image2.copy_replace(image, from_coords=(2, 3, 14, 11), to=(3, 6))
        self.assertEqual(image2.width(), 15)
        self.assertEqual(image2.height(), 14)
        self.assertEqual(image2.get(0+3, 0+6), image.get(2, 3))
        self.assertEqual(image2.get(11+3, 7+6), image.get(13, 10))
        self.assertEqual(image2.get(2+3, 4+6), image.get(2+2, 4+3))

        image2 = tkinter.PhotoImage(master=self.root)
        image2.copy_replace(image, from_coords=(2, 3, 14, 11), to=(0, 0, 100, 50))
        self.assertEqual(image2.width(), 100)
        self.assertEqual(image2.height(), 50)
        self.assertEqual(image2.get(0, 0), image.get(2, 3))
        self.assertEqual(image2.get(11, 7), image.get(13, 10))
        self.assertEqual(image2.get(2, 4), image.get(2+2, 4+3))
        self.assertEqual(image2.get(2+12, 4+8), image.get(2+2, 4+3))
        self.assertEqual(image2.get(2+12*2, 4), image.get(2+2, 4+3))
        self.assertEqual(image2.get(2, 4+8*3), image.get(2+2, 4+3))

        image2 = tkinter.PhotoImage(master=self.root)
        image2.copy_replace(image, from_coords=(2, 3, 14, 11), zoom=2)
        self.assertEqual(image2.width(), 24)
        self.assertEqual(image2.height(), 16)
        self.assertEqual(image2.get(0, 0), image.get(2, 3))
        self.assertEqual(image2.get(23, 15), image.get(13, 10))
        self.assertEqual(image2.get(2*2, 4*2), image.get(2+2, 4+3))
        self.assertEqual(image2.get(2*2+1, 4*2+1), image.get(6+2, 2+3))

        image2 = tkinter.PhotoImage(master=self.root)
        image2.copy_replace(image, from_coords=(2, 3, 14, 11), subsample=2)
        self.assertEqual(image2.width(), 6)
        self.assertEqual(image2.height(), 4)
        self.assertEqual(image2.get(0, 0), image.get(2, 3))
        self.assertEqual(image2.get(5, 3), image.get(12, 9))
        self.assertEqual(image2.get(1, 2), image.get(1*2+2, 2*2+3))

        image2 = tkinter.PhotoImage(master=self.root)
        image2.copy_replace(image, from_coords=(2, 3, 14, 11), subsample=2, zoom=3)
        self.assertEqual(image2.width(), 18)
        self.assertEqual(image2.height(), 12)
        self.assertEqual(image2.get(0, 0), image.get(2, 3))
        self.assertEqual(image2.get(17, 11), image.get(12, 9))
        self.assertEqual(image2.get(3*3, 2*3), image.get(3*2+2, 2*2+3))
        self.assertEqual(image2.get(3*3+2, 2*3+2), image.get(3*2+2, 2*2+3))
        self.assertEqual(image2.get(1*3, 2*3), image.get(1*2+2, 2*2+3))
        self.assertEqual(image2.get(1*3+2, 2*3+2), image.get(1*2+2, 2*2+3))

    def checkImgTrans(self, image, expected):
        actual = {(x, y)
                  for x in range(image.width())
                  for y in range(image.height())
                  if image.transparency_get(x, y)}
        self.assertEqual(actual, expected)

    def test_copy_replace_compositingrule(self):
        image1 = tkinter.PhotoImage(master=self.root, width=2, height=2)
        image1.blank()
        image1.put('black', to=(0, 0, 2, 2))
        image1.transparency_set(0, 0, True)

        # default compositingrule
        image2 = tkinter.PhotoImage(master=self.root, width=3, height=3)
        image2.blank()
        image2.put('white', to=(0, 0, 2, 2))
        image2.copy_replace(image1, to=(1, 1))
        self.checkImgTrans(image2, {(0, 2), (2, 0)})

        image3 = tkinter.PhotoImage(master=self.root, width=3, height=3)
        image3.blank()
        image3.put('white', to=(0, 0, 2, 2))
        image3.copy_replace(image1, to=(1, 1), compositingrule='overlay')
        self.checkImgTrans(image3, {(0, 2), (2, 0)})

        image4 = tkinter.PhotoImage(master=self.root, width=3, height=3)
        image4.blank()
        image4.put('white', to=(0, 0, 2, 2))
        image4.copy_replace(image1, to=(1, 1), compositingrule='set')
        self.checkImgTrans(image4, {(0, 2), (1, 1), (2, 0)})

    def test_put(self):
        image = self.create()
        image.put('{red green} {blue yellow}', to=(4, 6))
        self.assertEqual(image.get(4, 6), self.colorlist(255, 0, 0))
        self.assertEqual(image.get(5, 6),
                         self.colorlist(0, 128 if tkinter.TkVersion >= 8.6
                                           else 255, 0))
        self.assertEqual(image.get(4, 7), self.colorlist(0, 0, 255))
        self.assertEqual(image.get(5, 7), self.colorlist(255, 255, 0))

        image.put((('#f00', '#00ff00'), ('#000000fff', '#ffffffff0000')))
        self.assertEqual(image.get(0, 0), self.colorlist(255, 0, 0))
        self.assertEqual(image.get(1, 0), self.colorlist(0, 255, 0))
        self.assertEqual(image.get(0, 1), self.colorlist(0, 0, 255))
        self.assertEqual(image.get(1, 1), self.colorlist(255, 255, 0))

    def test_get(self):
        image = self.create()
        self.assertEqual(image.get(4, 6), self.colorlist(62, 116, 162))
        self.assertEqual(image.get(0, 0), self.colorlist(0, 0, 0))
        self.assertEqual(image.get(15, 15), self.colorlist(0, 0, 0))
        self.assertRaises(tkinter.TclError, image.get, -1, 0)
        self.assertRaises(tkinter.TclError, image.get, 0, -1)
        self.assertRaises(tkinter.TclError, image.get, 16, 15)
        self.assertRaises(tkinter.TclError, image.get, 15, 16)

    def test_read(self):
        # Due to the Tk bug https://core.tcl-lang.org/tk/tktview/1576528
        # the -from option does not work correctly for GIF and PNG files.
        # Use the PPM file for this test.
        testfile = support.findfile('python.ppm', subdir='tkinterdata')
        image = tkinter.PhotoImage(master=self.root, file=testfile)

        image2 = tkinter.PhotoImage(master=self.root)
        image2.read(testfile)
        self.assertEqual(image2.type(), 'photo')
        self.assertEqual(image2.width(), 16)
        self.assertEqual(image2.height(), 16)
        self.assertEqual(image2.get(0, 0), image.get(0, 0))
        self.assertEqual(image2.get(4, 6), image.get(4, 6))

        self.assertRaises(tkinter.TclError, image2.read, self.testfile, 'ppm')

        image2 = tkinter.PhotoImage(master=self.root)
        image2.read(testfile, from_coords=(2, 3, 14, 11))
        self.assertEqual(image2.width(), 12)
        self.assertEqual(image2.height(), 8)
        self.assertEqual(image2.get(0, 0), image.get(2, 3))
        self.assertEqual(image2.get(11, 7), image.get(13, 10))
        self.assertEqual(image2.get(2, 4), image.get(2+2, 4+3))

        image2 = tkinter.PhotoImage(master=self.root, file=testfile)
        self.assertEqual(image2.width(), 16)
        self.assertEqual(image2.height(), 16)
        image2.read(testfile, from_coords=(2, 3, 14, 11), shrink=True)
        self.assertEqual(image2.width(), 12)
        self.assertEqual(image2.height(), 8)
        self.assertEqual(image2.get(0, 0), image.get(2, 3))
        self.assertEqual(image2.get(11, 7), image.get(13, 10))
        self.assertEqual(image2.get(2, 4), image.get(2+2, 4+3))

        image2 = tkinter.PhotoImage(master=self.root)
        image2.read(testfile, from_coords=(2, 3, 14, 11), to=(3, 6))
        self.assertEqual(image2.type(), 'photo')
        self.assertEqual(image2.width(), 15)
        self.assertEqual(image2.height(), 14)
        self.assertEqual(image2.get(0+3, 0+6), image.get(2, 3))
        self.assertEqual(image2.get(11+3, 7+6), image.get(13, 10))
        self.assertEqual(image2.get(2+3, 4+6), image.get(2+2, 4+3))

    def test_write(self):
        filename = os_helper.TESTFN
        import locale
        if locale.getlocale()[0] is None:
            # Tcl uses Latin1 in the C locale
            filename = os_helper.TESTFN_ASCII
        image = self.create()
        self.addCleanup(os_helper.unlink, filename)

        image.write(filename)
        image2 = tkinter.PhotoImage('::img::test2', master=self.root,
                                    format='ppm', file=filename)
        self.assertEqual(str(image2), '::img::test2')
        self.assertEqual(image2.type(), 'photo')
        self.assertEqual(image2.width(), 16)
        self.assertEqual(image2.height(), 16)
        self.assertEqual(image2.get(0, 0), image.get(0, 0))
        self.assertEqual(image2.get(4, 6), image.get(4, 6))

        image.write(filename, format='gif', from_coords=(4, 6, 6, 9))
        image3 = tkinter.PhotoImage('::img::test3', master=self.root,
                                    format='gif', file=filename)
        self.assertEqual(str(image3), '::img::test3')
        self.assertEqual(image3.type(), 'photo')
        self.assertEqual(image3.width(), 2)
        self.assertEqual(image3.height(), 3)
        self.assertEqual(image3.get(0, 0), image.get(4, 6))
        self.assertEqual(image3.get(1, 2), image.get(5, 8))

        image.write(filename, background='#ff0000')
        image4 = tkinter.PhotoImage('::img::test4', master=self.root,
                                    format='ppm', file=filename)
        self.assertEqual(image4.get(0, 0), (255, 0, 0) if self.wantobjects else '255 0 0')
        self.assertEqual(image4.get(4, 6), image.get(4, 6))

        image.write(filename, grayscale=True)
        image5 = tkinter.PhotoImage('::img::test5', master=self.root,
                                    format='ppm', file=filename)
        c = image5.get(4, 6)
        if not self.wantobjects:
            c = c.split()
        self.assertTrue(c[0] == c[1] == c[2], c)

    def test_data(self):
        image = self.create()

        data = image.data()
        self.assertIsInstance(data, tuple)
        for row in data:
            self.assertIsInstance(row, str)
        c = image.get(4, 6)
        if not self.wantobjects:
            c = tuple(map(int, c.split()))
        self.assertEqual(data[6].split()[4], '#%02x%02x%02x' % c)

        data = image.data('ppm')
        image2 = tkinter.PhotoImage('::img::test2', master=self.root,
                                    format='ppm', data=data)
        self.assertEqual(str(image2), '::img::test2')
        self.assertEqual(image2.type(), 'photo')
        self.assertEqual(image2.width(), 16)
        self.assertEqual(image2.height(), 16)
        self.assertEqual(image2.get(0, 0), image.get(0, 0))
        self.assertEqual(image2.get(4, 6), image.get(4, 6))

        data = image.data(format='gif', from_coords=(4, 6, 6, 9))
        image3 = tkinter.PhotoImage('::img::test3', master=self.root,
                                    format='gif', data=data)
        self.assertEqual(str(image3), '::img::test3')
        self.assertEqual(image3.type(), 'photo')
        self.assertEqual(image3.width(), 2)
        self.assertEqual(image3.height(), 3)
        self.assertEqual(image3.get(0, 0), image.get(4, 6))
        self.assertEqual(image3.get(1, 2), image.get(5, 8))

        data = image.data('ppm', background='#ff0000')
        image4 = tkinter.PhotoImage('::img::test4', master=self.root,
                                    format='ppm', data=data)
        self.assertEqual(image4.get(0, 0), (255, 0, 0) if self.wantobjects else '255 0 0')
        self.assertEqual(image4.get(4, 6), image.get(4, 6))

        data = image.data('ppm', grayscale=True)
        image5 = tkinter.PhotoImage('::img::test5', master=self.root,
                                    format='ppm', data=data)
        c = image5.get(4, 6)
        if not self.wantobjects:
            c = c.split()
        self.assertTrue(c[0] == c[1] == c[2], c)


    def test_transparency(self):
        image = self.create()
        self.assertEqual(image.transparency_get(0, 0), True)
        self.assertEqual(image.transparency_get(4, 6), False)
        image.transparency_set(4, 6, True)
        self.assertEqual(image.transparency_get(4, 6), True)
        image.transparency_set(4, 6, False)
        self.assertEqual(image.transparency_get(4, 6), False)


if __name__ == "__main__":
    unittest.main()

### File: test_loadtk.py

In [ ]:
import os
import sys
import unittest
import test.support as test_support
from test.support import os_helper
from tkinter import Tcl, TclError

test_support.requires('gui')

class TkLoadTest(unittest.TestCase):

    @unittest.skipIf('DISPLAY' not in os.environ, 'No $DISPLAY set.')
    def testLoadTk(self):
        tcl = Tcl()
        self.assertRaises(TclError,tcl.winfo_geometry)
        tcl.loadtk()
        self.assertEqual('1x1+0+0', tcl.winfo_geometry())
        tcl.destroy()

    def testLoadTkFailure(self):
        old_display = None
        if sys.platform.startswith(('win', 'darwin', 'cygwin')):
            # no failure possible on windows?

            # XXX Maybe on tk older than 8.4.13 it would be possible,
            # see tkinter.h.
            return
        with os_helper.EnvironmentVarGuard() as env:
            if 'DISPLAY' in os.environ:
                del env['DISPLAY']
                # on some platforms, deleting environment variables
                # doesn't actually carry through to the process level
                # because they don't support unsetenv
                # If that's the case, abort.
                with os.popen('echo $DISPLAY') as pipe:
                    display = pipe.read().strip()
                if display:
                    return

            tcl = Tcl()
            self.assertRaises(TclError, tcl.winfo_geometry)
            self.assertRaises(TclError, tcl.loadtk)


if __name__ == "__main__":
    unittest.main()

### File: test_messagebox.py

In [ ]:
import unittest
import tkinter
from test.support import requires, swap_attr
from test.test_tkinter.support import AbstractDefaultRootTest
from tkinter.commondialog import Dialog
from tkinter.messagebox import showinfo

requires('gui')


class DefaultRootTest(AbstractDefaultRootTest, unittest.TestCase):

    def test_showinfo(self):
        def test_callback(dialog, master):
            nonlocal ismapped
            master.update()
            ismapped = master.winfo_ismapped()
            raise ZeroDivisionError

        with swap_attr(Dialog, '_test_callback', test_callback):
            ismapped = None
            self.assertRaises(ZeroDivisionError, showinfo, "Spam", "Egg Information")
            self.assertEqual(ismapped, False)

            root = tkinter.Tk()
            ismapped = None
            self.assertRaises(ZeroDivisionError, showinfo, "Spam", "Egg Information")
            self.assertEqual(ismapped, True)
            root.destroy()

            tkinter.NoDefaultRoot()
            self.assertRaises(RuntimeError, showinfo, "Spam", "Egg Information")


if __name__ == "__main__":
    unittest.main()

### File: test_misc.py

In [ ]:
import functools
import unittest
import tkinter
from tkinter import TclError
import enum
from test import support
from test.test_tkinter.support import (AbstractTkTest, AbstractDefaultRootTest,
                                       requires_tk, get_tk_patchlevel)

support.requires('gui')

class MiscTest(AbstractTkTest, unittest.TestCase):

    def test_all(self):
        self.assertIn("Widget", tkinter.__all__)
        # Check that variables from tkinter.constants are also in tkinter.__all__
        self.assertIn("CASCADE", tkinter.__all__)
        self.assertIsNotNone(tkinter.CASCADE)
        # Check that sys, re, and constants are not in tkinter.__all__
        self.assertNotIn("re", tkinter.__all__)
        self.assertNotIn("sys", tkinter.__all__)
        self.assertNotIn("constants", tkinter.__all__)
        # Check that an underscored functions is not in tkinter.__all__
        self.assertNotIn("_tkerror", tkinter.__all__)
        # Check that wantobjects is not in tkinter.__all__
        self.assertNotIn("wantobjects", tkinter.__all__)

    def test_repr(self):
        t = tkinter.Toplevel(self.root, name='top')
        f = tkinter.Frame(t, name='child')
        self.assertEqual(repr(f), '<tkinter.Frame object .top.child>')

    def test_generated_names(self):
        class Button2(tkinter.Button):
            pass

        t = tkinter.Toplevel(self.root)
        f = tkinter.Frame(t)
        f2 = tkinter.Frame(t)
        self.assertNotEqual(str(f), str(f2))
        b = tkinter.Button(f2)
        b2 = Button2(f2)
        for name in str(b).split('.') + str(b2).split('.'):
            self.assertFalse(name.isidentifier(), msg=repr(name))
        b3 = tkinter.Button(f2)
        b4 = Button2(f2)
        self.assertEqual(len({str(b), str(b2), str(b3), str(b4)}), 4)

    @requires_tk(8, 6, 6)
    def test_tk_busy(self):
        root = self.root
        f = tkinter.Frame(root, name='myframe')
        f2 = tkinter.Frame(root)
        f.pack()
        f2.pack()
        b = tkinter.Button(f)
        b.pack()
        f.tk_busy_hold()
        with self.assertRaisesRegex(TclError, 'unknown option "-spam"'):
            f.tk_busy_configure(spam='eggs')
        with self.assertRaisesRegex(TclError, 'unknown option "-spam"'):
            f.tk_busy_cget('spam')
        with self.assertRaisesRegex(TclError, 'unknown option "-spam"'):
            f.tk_busy_configure('spam')
        self.assertIsInstance(f.tk_busy_configure(), dict)

        self.assertTrue(f.tk_busy_status())
        self.assertFalse(root.tk_busy_status())
        self.assertFalse(f2.tk_busy_status())
        self.assertFalse(b.tk_busy_status())
        self.assertIn(f, f.tk_busy_current())
        self.assertIn(f, f.tk_busy_current('*.m?f*me'))
        self.assertNotIn(f, f.tk_busy_current('*spam'))

        f.tk_busy_forget()
        self.assertFalse(f.tk_busy_status())
        self.assertFalse(f.tk_busy_current())
        errmsg = r"can(no|')t find busy window.*"
        with self.assertRaisesRegex(TclError, errmsg):
            f.tk_busy_configure()
        with self.assertRaisesRegex(TclError, errmsg):
            f.tk_busy_forget()

    @requires_tk(8, 6, 6)
    def test_tk_busy_with_cursor(self):
        root = self.root
        if root._windowingsystem == 'aqua':
            self.skipTest('the cursor option is not supported on OSX/Aqua')
        f = tkinter.Frame(root, name='myframe')
        f.pack()
        f.tk_busy_hold(cursor='gumby')

        self.assertEqual(f.tk_busy_cget('cursor'), 'gumby')
        f.tk_busy_configure(cursor='heart')
        self.assertEqual(f.tk_busy_cget('cursor'), 'heart')
        self.assertEqual(f.tk_busy_configure()['cursor'][4], 'heart')
        self.assertEqual(f.tk_busy_configure('cursor')[4], 'heart')

        f.tk_busy_forget()
        errmsg = r"can(no|')t find busy window.*"
        with self.assertRaisesRegex(TclError, errmsg):
            f.tk_busy_cget('cursor')

    def test_tk_setPalette(self):
        root = self.root
        root.tk_setPalette('black')
        self.assertEqual(root['background'], 'black')
        root.tk_setPalette('white')
        self.assertEqual(root['background'], 'white')
        self.assertRaisesRegex(tkinter.TclError,
                '^unknown color name "spam"$',
                root.tk_setPalette, 'spam')

        root.tk_setPalette(background='black')
        self.assertEqual(root['background'], 'black')
        root.tk_setPalette(background='blue', highlightColor='yellow')
        self.assertEqual(root['background'], 'blue')
        self.assertEqual(root['highlightcolor'], 'yellow')
        root.tk_setPalette(background='yellow', highlightColor='blue')
        self.assertEqual(root['background'], 'yellow')
        self.assertEqual(root['highlightcolor'], 'blue')
        self.assertRaisesRegex(tkinter.TclError,
                '^unknown color name "spam"$',
                root.tk_setPalette, background='spam')
        self.assertRaisesRegex(tkinter.TclError,
                '^must specify a background color$',
                root.tk_setPalette, spam='white')
        self.assertRaisesRegex(tkinter.TclError,
                '^must specify a background color$',
                root.tk_setPalette, highlightColor='blue')

    def test_after(self):
        root = self.root

        def callback(start=0, step=1, *, end=0):
            nonlocal count
            count = start + step + end

        # Without function, sleeps for ms.
        self.assertIsNone(root.after(1))

        # Set up with callback with no args.
        count = 0
        timer1 = root.after(0, callback)
        self.assertIn(timer1, root.tk.call('after', 'info'))
        (script, _) = root.tk.splitlist(root.tk.call('after', 'info', timer1))
        root.update()  # Process all pending events.
        self.assertEqual(count, 1)
        with self.assertRaises(tkinter.TclError):
            root.tk.call(script)

        # Set up with callback with args.
        count = 0
        timer1 = root.after(0, callback, 42, 11)
        root.update()  # Process all pending events.
        self.assertEqual(count, 53)

        # Cancel before called.
        timer1 = root.after(1000, callback)
        self.assertIn(timer1, root.tk.call('after', 'info'))
        (script, _) = root.tk.splitlist(root.tk.call('after', 'info', timer1))
        root.after_cancel(timer1)  # Cancel this event.
        self.assertEqual(count, 53)
        with self.assertRaises(tkinter.TclError):
            root.tk.call(script)

        # Call with a callable class
        count = 0
        timer1 = root.after(0, functools.partial(callback, 42, 11))
        root.update()  # Process all pending events.
        self.assertEqual(count, 53)

        # Set up with callback with keyword args.
        count = 0
        timer1 = root.after(0, callback, 42, step=11, end=1)
        root.update()  # Process all pending events.
        self.assertEqual(count, 54)

    def test_after_idle(self):
        root = self.root

        def callback(start=0, step=1, *, end=0):
            nonlocal count
            count = start + step + end

        # Set up with callback with no args.
        count = 0
        idle1 = root.after_idle(callback)
        self.assertIn(idle1, root.tk.call('after', 'info'))
        (script, _) = root.tk.splitlist(root.tk.call('after', 'info', idle1))
        root.update_idletasks()  # Process all pending events.
        self.assertEqual(count, 1)
        with self.assertRaises(tkinter.TclError):
            root.tk.call(script)

        # Set up with callback with args.
        count = 0
        idle1 = root.after_idle(callback, 42, 11)
        root.update_idletasks()  # Process all pending events.
        self.assertEqual(count, 53)

        # Cancel before called.
        idle1 = root.after_idle(callback)
        self.assertIn(idle1, root.tk.call('after', 'info'))
        (script, _) = root.tk.splitlist(root.tk.call('after', 'info', idle1))
        root.after_cancel(idle1)  # Cancel this event.
        self.assertEqual(count, 53)
        with self.assertRaises(tkinter.TclError):
            root.tk.call(script)

        # Set up with callback with keyword args.
        count = 0
        idle1 = root.after_idle(callback, 42, step=11, end=1)
        root.update()  # Process all pending events.
        self.assertEqual(count, 54)

    def test_after_cancel(self):
        root = self.root

        def callback():
            nonlocal count
            count += 1

        timer1 = root.after(5000, callback)
        idle1 = root.after_idle(callback)

        # No value for id raises a ValueError.
        with self.assertRaises(ValueError):
            root.after_cancel(None)

        # Cancel timer event.
        count = 0
        (script, _) = root.tk.splitlist(root.tk.call('after', 'info', timer1))
        root.tk.call(script)
        self.assertEqual(count, 1)
        root.after_cancel(timer1)
        with self.assertRaises(tkinter.TclError):
            root.tk.call(script)
        self.assertEqual(count, 1)
        with self.assertRaises(tkinter.TclError):
            root.tk.call('after', 'info', timer1)

        # Cancel same event - nothing happens.
        root.after_cancel(timer1)

        # Cancel idle event.
        count = 0
        (script, _) = root.tk.splitlist(root.tk.call('after', 'info', idle1))
        root.tk.call(script)
        self.assertEqual(count, 1)
        root.after_cancel(idle1)
        with self.assertRaises(tkinter.TclError):
            root.tk.call(script)
        self.assertEqual(count, 1)
        with self.assertRaises(tkinter.TclError):
            root.tk.call('after', 'info', idle1)

    def test_after_info(self):
        root = self.root

        # No events.
        self.assertEqual(root.after_info(), ())

        # Add timer.
        timer = root.after(1, lambda: 'break')

        # With no parameter, it returns a tuple of the event handler ids.
        self.assertEqual(root.after_info(), (timer, ))
        root.after_cancel(timer)

        timer1 = root.after(5000, lambda: 'break')
        timer2 = root.after(5000, lambda: 'break')
        idle1 = root.after_idle(lambda: 'break')
        # Only contains new events and not 'timer'.
        self.assertEqual(root.after_info(), (idle1, timer2, timer1))

        # With a parameter returns a tuple of (script, type).
        timer1_info = root.after_info(timer1)
        self.assertEqual(len(timer1_info), 2)
        self.assertEqual(timer1_info[1], 'timer')
        idle1_info = root.after_info(idle1)
        self.assertEqual(len(idle1_info), 2)
        self.assertEqual(idle1_info[1], 'idle')

        root.after_cancel(timer1)
        with self.assertRaises(tkinter.TclError):
            root.after_info(timer1)
        root.after_cancel(timer2)
        with self.assertRaises(tkinter.TclError):
            root.after_info(timer2)
        root.after_cancel(idle1)
        with self.assertRaises(tkinter.TclError):
            root.after_info(idle1)

        # No events.
        self.assertEqual(root.after_info(), ())

    def test_clipboard(self):
        root = self.root
        root.clipboard_clear()
        root.clipboard_append('Ùñî')
        self.assertEqual(root.clipboard_get(), 'Ùñî')
        root.clipboard_append('çōđě')
        self.assertEqual(root.clipboard_get(), 'Ùñîçōđě')
        root.clipboard_clear()
        with self.assertRaises(tkinter.TclError):
            root.clipboard_get()

    def test_clipboard_astral(self):
        root = self.root
        root.clipboard_clear()
        root.clipboard_append('𝔘𝔫𝔦')
        self.assertEqual(root.clipboard_get(), '𝔘𝔫𝔦')
        root.clipboard_append('𝔠𝔬𝔡𝔢')
        self.assertEqual(root.clipboard_get(), '𝔘𝔫𝔦𝔠𝔬𝔡𝔢')
        root.clipboard_clear()
        with self.assertRaises(tkinter.TclError):
            root.clipboard_get()

    def test_winfo_rgb(self):

        def assertApprox(col1, col2):
            # A small amount of flexibility is required (bpo-45496)
            # 33 is ~0.05% of 65535, which is a reasonable margin
            for col1_channel, col2_channel in zip(col1, col2):
                self.assertAlmostEqual(col1_channel, col2_channel, delta=33)

        root = self.root
        rgb = root.winfo_rgb

        # Color name.
        self.assertEqual(rgb('red'), (65535, 0, 0))
        self.assertEqual(rgb('dark slate blue'), (18504, 15677, 35723))
        # #RGB - extends each 4-bit hex value to be 16-bit.
        self.assertEqual(rgb('#F0F'), (0xFFFF, 0x0000, 0xFFFF))
        # #RRGGBB - extends each 8-bit hex value to be 16-bit.
        assertApprox(rgb('#4a3c8c'), (0x4a4a, 0x3c3c, 0x8c8c))
        # #RRRRGGGGBBBB
        assertApprox(rgb('#dede14143939'), (0xdede, 0x1414, 0x3939))
        # Invalid string.
        with self.assertRaises(tkinter.TclError):
            rgb('#123456789a')
        # RGB triplet is invalid input.
        with self.assertRaises(tkinter.TclError):
            rgb((111, 78, 55))

    def test_winfo_pathname(self):
        t = tkinter.Toplevel(self.root)
        w = tkinter.Button(t)
        wid = w.winfo_id()
        self.assertIsInstance(wid, int)
        self.assertEqual(self.root.winfo_pathname(hex(wid)), str(w))
        self.assertEqual(self.root.winfo_pathname(hex(wid), displayof=None), str(w))
        self.assertEqual(self.root.winfo_pathname(hex(wid), displayof=t), str(w))
        self.assertEqual(self.root.winfo_pathname(wid), str(w))
        self.assertEqual(self.root.winfo_pathname(wid, displayof=None), str(w))
        self.assertEqual(self.root.winfo_pathname(wid, displayof=t), str(w))

    def test_event_repr_defaults(self):
        e = tkinter.Event()
        e.serial = 12345
        e.num = '??'
        e.height = '??'
        e.keycode = '??'
        e.state = 0
        e.time = 123456789
        e.width = '??'
        e.x = '??'
        e.y = '??'
        e.char = ''
        e.keysym = '??'
        e.keysym_num = '??'
        e.type = '100'
        e.widget = '??'
        e.x_root = '??'
        e.y_root = '??'
        e.delta = 0
        self.assertEqual(repr(e), '<100 event>')

    def test_event_repr(self):
        e = tkinter.Event()
        e.serial = 12345
        e.num = 3
        e.focus = True
        e.height = 200
        e.keycode = 65
        e.state = 0x30405
        e.time = 123456789
        e.width = 300
        e.x = 10
        e.y = 20
        e.char = 'A'
        e.send_event = True
        e.keysym = 'Key-A'
        e.keysym_num = ord('A')
        e.type = tkinter.EventType.Configure
        e.widget = '.text'
        e.x_root = 1010
        e.y_root = 1020
        e.delta = -1
        self.assertEqual(repr(e),
                         "<Configure event send_event=True"
                         " state=Shift|Control|Button3|0x30000"
                         " keysym=Key-A keycode=65 char='A'"
                         " num=3 delta=-1 focus=True"
                         " x=10 y=20 width=300 height=200>")

    def test_eventtype_enum(self):
        class CheckedEventType(enum.StrEnum):
            KeyPress = '2'
            Key = KeyPress
            KeyRelease = '3'
            ButtonPress = '4'
            Button = ButtonPress
            ButtonRelease = '5'
            Motion = '6'
            Enter = '7'
            Leave = '8'
            FocusIn = '9'
            FocusOut = '10'
            Keymap = '11'           # undocumented
            Expose = '12'
            GraphicsExpose = '13'   # undocumented
            NoExpose = '14'         # undocumented
            Visibility = '15'
            Create = '16'
            Destroy = '17'
            Unmap = '18'
            Map = '19'
            MapRequest = '20'
            Reparent = '21'
            Configure = '22'
            ConfigureRequest = '23'
            Gravity = '24'
            ResizeRequest = '25'
            Circulate = '26'
            CirculateRequest = '27'
            Property = '28'
            SelectionClear = '29'   # undocumented
            SelectionRequest = '30' # undocumented
            Selection = '31'        # undocumented
            Colormap = '32'
            ClientMessage = '33'    # undocumented
            Mapping = '34'          # undocumented
            VirtualEvent = '35'     # undocumented
            Activate = '36'
            Deactivate = '37'
            MouseWheel = '38'
        enum._test_simple_enum(CheckedEventType, tkinter.EventType)

    def test_getboolean(self):
        for v in 'true', 'yes', 'on', '1', 't', 'y', 1, True:
            self.assertIs(self.root.getboolean(v), True)
        for v in 'false', 'no', 'off', '0', 'f', 'n', 0, False:
            self.assertIs(self.root.getboolean(v), False)
        self.assertRaises(ValueError, self.root.getboolean, 'yea')
        self.assertRaises(ValueError, self.root.getboolean, '')
        self.assertRaises(TypeError, self.root.getboolean, None)
        self.assertRaises(TypeError, self.root.getboolean, ())

    def test_mainloop(self):
        log = []
        def callback():
            log.append(1)
            self.root.after(100, self.root.quit)
        self.root.after(100, callback)
        self.root.mainloop(1)
        self.assertEqual(log, [])
        self.root.mainloop(0)
        self.assertEqual(log, [1])
        self.assertTrue(self.root.winfo_exists())

    def test_info_patchlevel(self):
        vi = self.root.info_patchlevel()
        f = tkinter.Frame(self.root)
        self.assertEqual(f.info_patchlevel(), vi)
        # The following is almost a copy of tests for sys.version_info.
        self.assertIsInstance(vi[:], tuple)
        self.assertEqual(len(vi), 5)
        self.assertIsInstance(vi[0], int)
        self.assertIsInstance(vi[1], int)
        self.assertIsInstance(vi[2], int)
        self.assertIn(vi[3], ("alpha", "beta", "candidate", "final"))
        self.assertIsInstance(vi[4], int)
        self.assertIsInstance(vi.major, int)
        self.assertIsInstance(vi.minor, int)
        self.assertIsInstance(vi.micro, int)
        self.assertIn(vi.releaselevel, ("alpha", "beta", "final"))
        self.assertIsInstance(vi.serial, int)
        self.assertEqual(vi[0], vi.major)
        self.assertEqual(vi[1], vi.minor)
        self.assertEqual(vi[2], vi.micro)
        self.assertEqual(vi[3], vi.releaselevel)
        self.assertEqual(vi[4], vi.serial)
        self.assertTrue(vi > (1,0,0))
        if vi.releaselevel == 'final':
            self.assertEqual(vi.serial, 0)
        else:
            self.assertEqual(vi.micro, 0)
        self.assertStartsWith(str(vi), f'{vi.major}.{vi.minor}')

    def test_embedded_null(self):
        widget = tkinter.Entry(self.root)
        widget.insert(0, 'abc\0def')  # ASCII-only
        widget.selection_range(0, 'end')
        self.assertEqual(widget.selection_get(), 'abc\x00def')
        widget.insert(0, '\u20ac\0')  # non-ASCII
        widget.selection_range(0, 'end')
        self.assertEqual(widget.selection_get(), '\u20ac\0abc\x00def')


class WmTest(AbstractTkTest, unittest.TestCase):

    def test_wm_attribute(self):
        w = self.root
        attributes = w.wm_attributes(return_python_dict=True)
        self.assertIsInstance(attributes, dict)
        attributes2 = w.wm_attributes()
        self.assertIsInstance(attributes2, tuple)
        self.assertEqual(attributes2[::2],
                         tuple('-' + k for k in attributes))
        self.assertEqual(attributes2[1::2], tuple(attributes.values()))
        # silently deprecated
        attributes3 = w.wm_attributes(None)
        if self.wantobjects:
            self.assertEqual(attributes3, attributes2)
        else:
            self.assertIsInstance(attributes3, str)

        for name in attributes:
            self.assertEqual(w.wm_attributes(name), attributes[name])
        # silently deprecated
        for name in attributes:
            self.assertEqual(w.wm_attributes('-' + name), attributes[name])

        self.assertIn('alpha', attributes)
        self.assertIn('fullscreen', attributes)
        self.assertIn('topmost', attributes)
        if w._windowingsystem == "win32":
            self.assertIn('disabled', attributes)
            self.assertIn('toolwindow', attributes)
            self.assertIn('transparentcolor', attributes)
        if w._windowingsystem == "aqua":
            self.assertIn('modified', attributes)
            self.assertIn('notify', attributes)
            self.assertIn('titlepath', attributes)
            self.assertIn('transparent', attributes)
        if w._windowingsystem == "x11":
            self.assertIn('type', attributes)
            self.assertIn('zoomed', attributes)

        w.wm_attributes(alpha=0.5)
        self.assertEqual(w.wm_attributes('alpha'),
                         0.5 if self.wantobjects else '0.5')
        w.wm_attributes(alpha=1.0)
        self.assertEqual(w.wm_attributes('alpha'),
                         1.0 if self.wantobjects else '1.0')
        # silently deprecated
        w.wm_attributes('-alpha', 0.5)
        self.assertEqual(w.wm_attributes('alpha'),
                         0.5 if self.wantobjects else '0.5')
        w.wm_attributes(alpha=1.0)
        self.assertEqual(w.wm_attributes('alpha'),
                         1.0 if self.wantobjects else '1.0')

    def test_wm_iconbitmap(self):
        t = tkinter.Toplevel(self.root)
        self.assertEqual(t.wm_iconbitmap(), '')
        t.wm_iconbitmap('hourglass')
        bug = False
        if t._windowingsystem == 'aqua':
            # Tk bug 13ac26b35dc55f7c37f70b39d59d7ef3e63017c8.
            patchlevel = get_tk_patchlevel(t)
            if patchlevel < (8, 6, 17) or (9, 0) <= patchlevel < (9, 0, 2):
                bug = True
        if not bug:
            self.assertEqual(t.wm_iconbitmap(), 'hourglass')
        self.assertEqual(self.root.wm_iconbitmap(), '')
        t.wm_iconbitmap('')
        self.assertEqual(t.wm_iconbitmap(), '')

        if t._windowingsystem == 'win32':
            t.wm_iconbitmap(default='hourglass')
            self.assertEqual(t.wm_iconbitmap(), 'hourglass')
            self.assertEqual(self.root.wm_iconbitmap(), '')
            t.wm_iconbitmap(default='')
            self.assertEqual(t.wm_iconbitmap(), '')

        t.destroy()


class EventTest(AbstractTkTest, unittest.TestCase):

    def test_focus(self):
        f = tkinter.Frame(self.root, width=150, height=100)
        f.pack()
        self.root.wait_visibility()  # needed on Windows
        self.root.update_idletasks()

        events = []
        f.bind('<FocusIn>', events.append)

        f.focus_force()
        self.root.update()
        self.assertEqual(len(events), 1, events)
        e = events[0]
        self.assertIs(e.type, tkinter.EventType.FocusIn)
        self.assertIs(e.widget, f)
        self.assertIsInstance(e.serial, int)
        self.assertEqual(e.time, '??')
        self.assertIs(e.send_event, False)
        self.assertNotHasAttr(e, 'focus')
        self.assertEqual(e.num, '??')
        self.assertEqual(e.state, '??')
        self.assertEqual(e.char, '??')
        self.assertEqual(e.keycode, '??')
        self.assertEqual(e.keysym, '??')
        self.assertEqual(e.keysym_num, '??')
        self.assertEqual(e.width, '??')
        self.assertEqual(e.height, '??')
        self.assertEqual(e.x, '??')
        self.assertEqual(e.y, '??')
        self.assertEqual(e.x_root, '??')
        self.assertEqual(e.y_root, '??')
        self.assertEqual(e.delta, 0)
        self.assertEqual(repr(e), '<FocusIn event>')

    def test_configure(self):
        f = tkinter.Frame(self.root, width=150, height=100)
        f.pack()
        self.root.wait_visibility()  # needed on Windows
        self.root.update_idletasks()

        events = []
        f.bind('<Configure>', events.append)

        f.configure(height=120, borderwidth=10)
        self.assertEqual(len(events), 1, events)
        e = events[0]
        self.assertIs(e.type, tkinter.EventType.Configure)
        self.assertIs(e.widget, f)
        self.assertIsInstance(e.serial, int)
        self.assertEqual(e.time, '??')
        self.assertIs(e.send_event, False)
        self.assertNotHasAttr(e, 'focus')
        self.assertEqual(e.num, '??')
        self.assertEqual(e.state, '??')
        self.assertEqual(e.char, '??')
        self.assertEqual(e.keycode, '??')
        self.assertEqual(e.keysym, '??')
        self.assertEqual(e.keysym_num, '??')
        self.assertEqual(e.width, 150)
        self.assertEqual(e.height, 100)
        self.assertEqual(e.x, 0)
        self.assertEqual(e.y, 0)
        self.assertEqual(e.x_root, '??')
        self.assertEqual(e.y_root, '??')
        self.assertEqual(e.delta, 0)
        self.assertEqual(repr(e), '<Configure event x=0 y=0 width=150 height=100>')

    def test_event_generate_key_press(self):
        f = tkinter.Frame(self.root, width=150, height=100)
        f.pack()
        self.root.wait_visibility()  # needed on Windows
        self.root.update_idletasks()

        events = []
        f.bind('<KeyPress>', events.append)
        f.focus_force()

        f.event_generate('<Alt-z>')
        self.assertEqual(len(events), 1, events)
        e = events[0]
        self.assertIs(e.type, tkinter.EventType.KeyPress)
        self.assertIs(e.widget, f)
        self.assertIsInstance(e.serial, int)
        self.assertEqual(e.time, 0)
        self.assertIs(e.send_event, False)
        self.assertNotHasAttr(e, 'focus')
        self.assertEqual(e.num, '??')
        self.assertIsInstance(e.state, int)
        self.assertNotEqual(e.state, 0)
        self.assertEqual(e.char, 'z')
        self.assertIsInstance(e.keycode, int)
        self.assertNotEqual(e.keycode, 0)
        self.assertEqual(e.keysym, 'z')
        self.assertEqual(e.keysym_num, ord('z'))
        self.assertEqual(e.width, '??')
        self.assertEqual(e.height, '??')
        self.assertEqual(e.x, -1 - f.winfo_rootx())
        self.assertEqual(e.y, -1 - f.winfo_rooty())
        self.assertEqual(e.x_root, -1)
        self.assertEqual(e.y_root, -1)
        self.assertEqual(e.delta, 0)
        self.assertEqual(repr(e),
            f"<KeyPress event state={e.state:#x} "
            f"keysym=z keycode={e.keycode} char='z' x={e.x} y={e.y}>")

    def test_event_generate_enter(self):
        f = tkinter.Frame(self.root, width=150, height=100)
        f.pack()
        self.root.wait_visibility()  # needed on Windows
        self.root.update_idletasks()

        events = []
        f.bind('<Enter>', events.append)

        f.event_generate('<Enter>', x=100, y=50)
        self.assertEqual(len(events), 1, events)
        e = events[0]
        self.assertIs(e.type, tkinter.EventType.Enter)
        self.assertIs(e.widget, f)
        self.assertIsInstance(e.serial, int)
        self.assertEqual(e.time, 0)
        self.assertIs(e.send_event, False)
        self.assertIs(e.focus, False)
        self.assertEqual(e.num, '??')
        self.assertEqual(e.state, 0)
        self.assertEqual(e.char, '??')
        self.assertEqual(e.keycode, '??')
        self.assertEqual(e.keysym, '??')
        self.assertEqual(e.keysym_num, '??')
        self.assertEqual(e.width, '??')
        self.assertEqual(e.height, '??')
        self.assertEqual(e.x, 100)
        self.assertEqual(e.y, 50)
        self.assertEqual(e.x_root, 100 + f.winfo_rootx())
        self.assertEqual(e.y_root, 50 + f.winfo_rooty())
        self.assertEqual(e.delta, 0)
        self.assertEqual(repr(e), '<Enter event focus=False x=100 y=50>')

    def test_event_generate_button_press(self):
        f = tkinter.Frame(self.root, width=150, height=100)
        f.pack()
        self.root.wait_visibility()  # needed on Windows
        self.root.update_idletasks()

        events = []
        f.bind('<ButtonPress>', events.append)
        f.focus_force()

        f.event_generate('<Button-1>', x=100, y=50)
        self.assertEqual(len(events), 1, events)
        e = events[0]
        self.assertIs(e.type, tkinter.EventType.ButtonPress)
        self.assertIs(e.widget, f)
        self.assertIsInstance(e.serial, int)
        self.assertEqual(e.time, 0)
        self.assertIs(e.send_event, False)
        self.assertNotHasAttr(e, 'focus')
        self.assertEqual(e.num, 1)
        self.assertEqual(e.state, 0)
        self.assertEqual(e.char, '??')
        self.assertEqual(e.keycode, '??')
        self.assertEqual(e.keysym, '??')
        self.assertEqual(e.keysym_num, '??')
        self.assertEqual(e.width, '??')
        self.assertEqual(e.height, '??')
        self.assertEqual(e.x, 100)
        self.assertEqual(e.y, 50)
        self.assertEqual(e.x_root, f.winfo_rootx() + 100)
        self.assertEqual(e.y_root, f.winfo_rooty() + 50)
        self.assertEqual(e.delta, 0)
        self.assertEqual(repr(e), '<ButtonPress event num=1 x=100 y=50>')

    def test_event_generate_motion(self):
        f = tkinter.Frame(self.root, width=150, height=100)
        f.pack()
        self.root.wait_visibility()  # needed on Windows
        self.root.update_idletasks()

        events = []
        f.bind('<Motion>', events.append)
        f.focus_force()

        f.event_generate('<B1-Motion>', x=100, y=50)
        self.assertEqual(len(events), 1, events)
        e = events[0]
        self.assertIs(e.type, tkinter.EventType.Motion)
        self.assertIs(e.widget, f)
        self.assertIsInstance(e.serial, int)
        self.assertEqual(e.time, 0)
        self.assertIs(e.send_event, False)
        self.assertNotHasAttr(e, 'focus')
        self.assertEqual(e.num, '??')
        self.assertEqual(e.state, 0x100)
        self.assertEqual(e.char, '??')
        self.assertEqual(e.keycode, '??')
        self.assertEqual(e.keysym, '??')
        self.assertEqual(e.keysym_num, '??')
        self.assertEqual(e.width, '??')
        self.assertEqual(e.height, '??')
        self.assertEqual(e.x, 100)
        self.assertEqual(e.y, 50)
        self.assertEqual(e.x_root, f.winfo_rootx() + 100)
        self.assertEqual(e.y_root, f.winfo_rooty() + 50)
        self.assertEqual(e.delta, 0)
        self.assertEqual(repr(e), '<Motion event state=Button1 x=100 y=50>')

    def test_event_generate_mouse_wheel(self):
        f = tkinter.Frame(self.root, width=150, height=100)
        f.pack()
        self.root.wait_visibility()  # needed on Windows
        self.root.update_idletasks()

        events = []
        f.bind('<MouseWheel>', events.append)
        f.focus_force()

        f.event_generate('<MouseWheel>', x=100, y=50, delta=-5)
        self.assertEqual(len(events), 1, events)
        e = events[0]
        self.assertIs(e.type, tkinter.EventType.MouseWheel)
        self.assertIs(e.widget, f)
        self.assertIsInstance(e.serial, int)
        self.assertIs(e.send_event, False)
        self.assertNotHasAttr(e, 'focus')
        self.assertEqual(e.time, 0)
        self.assertEqual(e.num, '??')
        self.assertEqual(e.state, 0)
        self.assertEqual(e.char, '??')
        self.assertEqual(e.keycode, '??')
        self.assertEqual(e.keysym, '??')
        self.assertEqual(e.keysym_num, '??')
        self.assertEqual(e.width, '??')
        self.assertEqual(e.height, '??')
        self.assertEqual(e.x, 100)
        self.assertEqual(e.y, 50)
        self.assertEqual(e.x_root, f.winfo_rootx() + 100)
        self.assertEqual(e.y_root, f.winfo_rooty() + 50)
        self.assertEqual(e.delta, -5)
        self.assertEqual(repr(e), '<MouseWheel event delta=-5 x=100 y=50>')

    def test_generate_event_virtual_event(self):
        f = tkinter.Frame(self.root, width=150, height=100)
        f.pack()
        self.root.wait_visibility()  # needed on Windows
        self.root.update_idletasks()

        events = []
        f.bind('<<Spam>>', events.append)
        f.focus_force()

        f.event_generate('<<Spam>>', x=50)
        self.assertEqual(len(events), 1, events)
        e = events[0]
        self.assertIs(e.type, tkinter.EventType.VirtualEvent)
        self.assertIs(e.widget, f)
        self.assertIsInstance(e.serial, int)
        self.assertEqual(e.time, 0)
        self.assertIs(e.send_event, False)
        self.assertNotHasAttr(e, 'focus')
        self.assertEqual(e.num, '??')
        self.assertEqual(e.state, 0)
        self.assertEqual(e.char, '??')
        self.assertEqual(e.keycode, '??')
        self.assertEqual(e.keysym, '??')
        self.assertEqual(e.keysym_num, '??')
        self.assertEqual(e.width, '??')
        self.assertEqual(e.height, '??')
        self.assertEqual(e.x, 50)
        self.assertEqual(e.y, 0)
        self.assertEqual(e.x_root, f.winfo_rootx() + 50)
        self.assertEqual(e.y_root, -1)
        self.assertEqual(e.delta, 0)
        self.assertEqual(repr(e),
            f"<VirtualEvent event x=50 y=0>")


class BindTest(AbstractTkTest, unittest.TestCase):

    def setUp(self):
        super().setUp()
        root = self.root
        self.frame = tkinter.Frame(self.root, class_='Test',
                                   width=150, height=100)
        self.frame.pack()

    def assertCommandExist(self, funcid):
        self.assertEqual(_info_commands(self.root, funcid), (funcid,))

    def assertCommandNotExist(self, funcid):
        self.assertEqual(_info_commands(self.root, funcid), ())

    def test_bind(self):
        event = '<Control-Alt-Key-a>'
        f = self.frame
        self.assertEqual(f.bind(), ())
        self.assertEqual(f.bind(event), '')
        def test1(e): pass
        def test2(e): pass

        funcid = f.bind(event, test1)
        self.assertEqual(f.bind(), (event,))
        script = f.bind(event)
        self.assertIn(funcid, script)
        self.assertCommandExist(funcid)

        funcid2 = f.bind(event, test2, add=True)
        script = f.bind(event)
        self.assertIn(funcid, script)
        self.assertIn(funcid2, script)
        self.assertCommandExist(funcid)
        self.assertCommandExist(funcid2)

    def test_unbind(self):
        event = '<Control-Alt-Key-b>'
        f = self.frame
        self.assertEqual(f.bind(), ())
        self.assertEqual(f.bind(event), '')
        def test1(e): pass
        def test2(e): pass

        funcid = f.bind(event, test1)
        funcid2 = f.bind(event, test2, add=True)

        self.assertRaises(TypeError, f.unbind)
        f.unbind(event)
        self.assertEqual(f.bind(event), '')
        self.assertEqual(f.bind(), ())

    def test_unbind2(self):
        f = self.frame
        f.wait_visibility()
        f.focus_force()
        f.update_idletasks()
        event = '<Control-Alt-Key-c>'
        self.assertEqual(f.bind(), ())
        self.assertEqual(f.bind(event), '')
        def test1(e): events.append('a')
        def test2(e): events.append('b')
        def test3(e): events.append('c')

        funcid = f.bind(event, test1)
        funcid2 = f.bind(event, test2, add=True)
        funcid3 = f.bind(event, test3, add=True)
        events = []
        f.event_generate(event)
        self.assertEqual(events, ['a', 'b', 'c'])

        f.unbind(event, funcid2)
        script = f.bind(event)
        self.assertNotIn(funcid2, script)
        self.assertIn(funcid, script)
        self.assertIn(funcid3, script)
        self.assertEqual(f.bind(), (event,))
        self.assertCommandNotExist(funcid2)
        self.assertCommandExist(funcid)
        self.assertCommandExist(funcid3)
        events = []
        f.event_generate(event)
        self.assertEqual(events, ['a', 'c'])

        f.unbind(event, funcid)
        f.unbind(event, funcid3)
        self.assertEqual(f.bind(event), '')
        self.assertEqual(f.bind(), ())
        self.assertCommandNotExist(funcid)
        self.assertCommandNotExist(funcid2)
        self.assertCommandNotExist(funcid3)
        events = []
        f.event_generate(event)
        self.assertEqual(events, [])

        # non-idempotent
        self.assertRaises(tkinter.TclError, f.unbind, event, funcid2)

    def test_bind_rebind(self):
        event = '<Control-Alt-Key-d>'
        f = self.frame
        self.assertEqual(f.bind(), ())
        self.assertEqual(f.bind(event), '')
        def test1(e): pass
        def test2(e): pass
        def test3(e): pass

        funcid = f.bind(event, test1)
        funcid2 = f.bind(event, test2, add=True)
        script = f.bind(event)
        self.assertIn(funcid2, script)
        self.assertIn(funcid, script)
        self.assertCommandExist(funcid)
        self.assertCommandExist(funcid2)

        funcid3 = f.bind(event, test3)
        script = f.bind(event)
        self.assertNotIn(funcid, script)
        self.assertNotIn(funcid2, script)
        self.assertIn(funcid3, script)
        self.assertCommandExist(funcid3)

    def test_bind_class(self):
        event = '<Control-Alt-Key-e>'
        bind_class = self.root.bind_class
        unbind_class = self.root.unbind_class
        self.assertRaises(TypeError, bind_class)
        self.assertEqual(bind_class('Test'), ())
        self.assertEqual(bind_class('Test', event), '')
        self.addCleanup(unbind_class, 'Test', event)
        def test1(e): pass
        def test2(e): pass

        funcid = bind_class('Test', event, test1)
        self.assertEqual(bind_class('Test'), (event,))
        script = bind_class('Test', event)
        self.assertIn(funcid, script)
        self.assertCommandExist(funcid)

        funcid2 = bind_class('Test', event, test2, add=True)
        script = bind_class('Test', event)
        self.assertIn(funcid, script)
        self.assertIn(funcid2, script)
        self.assertCommandExist(funcid)
        self.assertCommandExist(funcid2)

    def test_unbind_class(self):
        event = '<Control-Alt-Key-f>'
        bind_class = self.root.bind_class
        unbind_class = self.root.unbind_class
        self.assertEqual(bind_class('Test'), ())
        self.assertEqual(bind_class('Test', event), '')
        self.addCleanup(unbind_class, 'Test', event)
        def test1(e): pass
        def test2(e): pass

        funcid = bind_class('Test', event, test1)
        funcid2 = bind_class('Test', event, test2, add=True)

        self.assertRaises(TypeError, unbind_class)
        self.assertRaises(TypeError, unbind_class, 'Test')
        unbind_class('Test', event)
        self.assertEqual(bind_class('Test', event), '')
        self.assertEqual(bind_class('Test'), ())
        self.assertCommandExist(funcid)
        self.assertCommandExist(funcid2)

        unbind_class('Test', event)  # idempotent

    def test_bind_class_rebind(self):
        event = '<Control-Alt-Key-g>'
        bind_class = self.root.bind_class
        unbind_class = self.root.unbind_class
        self.assertEqual(bind_class('Test'), ())
        self.assertEqual(bind_class('Test', event), '')
        self.addCleanup(unbind_class, 'Test', event)
        def test1(e): pass
        def test2(e): pass
        def test3(e): pass

        funcid = bind_class('Test', event, test1)
        funcid2 = bind_class('Test', event, test2, add=True)
        script = bind_class('Test', event)
        self.assertIn(funcid2, script)
        self.assertIn(funcid, script)
        self.assertCommandExist(funcid)
        self.assertCommandExist(funcid2)

        funcid3 = bind_class('Test', event, test3)
        script = bind_class('Test', event)
        self.assertNotIn(funcid, script)
        self.assertNotIn(funcid2, script)
        self.assertIn(funcid3, script)
        self.assertCommandExist(funcid)
        self.assertCommandExist(funcid2)
        self.assertCommandExist(funcid3)

    def test_bind_all(self):
        event = '<Control-Alt-Key-h>'
        bind_all = self.root.bind_all
        unbind_all = self.root.unbind_all
        self.assertNotIn(event, bind_all())
        self.assertEqual(bind_all(event), '')
        self.addCleanup(unbind_all, event)
        def test1(e): pass
        def test2(e): pass

        funcid = bind_all(event, test1)
        self.assertIn(event, bind_all())
        script = bind_all(event)
        self.assertIn(funcid, script)
        self.assertCommandExist(funcid)

        funcid2 = bind_all(event, test2, add=True)
        script = bind_all(event)
        self.assertIn(funcid, script)
        self.assertIn(funcid2, script)
        self.assertCommandExist(funcid)
        self.assertCommandExist(funcid2)

    def test_unbind_all(self):
        event = '<Control-Alt-Key-i>'
        bind_all = self.root.bind_all
        unbind_all = self.root.unbind_all
        self.assertNotIn(event, bind_all())
        self.assertEqual(bind_all(event), '')
        self.addCleanup(unbind_all, event)
        def test1(e): pass
        def test2(e): pass

        funcid = bind_all(event, test1)
        funcid2 = bind_all(event, test2, add=True)

        unbind_all(event)
        self.assertEqual(bind_all(event), '')
        self.assertNotIn(event, bind_all())
        self.assertCommandExist(funcid)
        self.assertCommandExist(funcid2)

        unbind_all(event)  # idempotent

    def test_bind_all_rebind(self):
        event = '<Control-Alt-Key-j>'
        bind_all = self.root.bind_all
        unbind_all = self.root.unbind_all
        self.assertNotIn(event, bind_all())
        self.assertEqual(bind_all(event), '')
        self.addCleanup(unbind_all, event)
        def test1(e): pass
        def test2(e): pass
        def test3(e): pass

        funcid = bind_all(event, test1)
        funcid2 = bind_all(event, test2, add=True)
        script = bind_all(event)
        self.assertIn(funcid2, script)
        self.assertIn(funcid, script)
        self.assertCommandExist(funcid)
        self.assertCommandExist(funcid2)

        funcid3 = bind_all(event, test3)
        script = bind_all(event)
        self.assertNotIn(funcid, script)
        self.assertNotIn(funcid2, script)
        self.assertIn(funcid3, script)
        self.assertCommandExist(funcid)
        self.assertCommandExist(funcid2)
        self.assertCommandExist(funcid3)

    def _test_tag_bind(self, w):
        tag = 'sel'
        event = '<Control-Alt-Key-a>'
        w.pack()
        self.assertRaises(TypeError, w.tag_bind)
        tag_bind = w._tag_bind if isinstance(w, tkinter.Text) else w.tag_bind
        if isinstance(w, tkinter.Text):
            self.assertRaises(TypeError, w.tag_bind, tag)
            self.assertRaises(TypeError, w.tag_bind, tag, event)
        self.assertEqual(tag_bind(tag), ())
        self.assertEqual(tag_bind(tag, event), '')
        def test1(e): pass
        def test2(e): pass

        funcid = w.tag_bind(tag, event, test1)
        self.assertEqual(tag_bind(tag), (event,))
        script = tag_bind(tag, event)
        self.assertIn(funcid, script)
        self.assertCommandExist(funcid)

        funcid2 = w.tag_bind(tag, event, test2, add=True)
        script = tag_bind(tag, event)
        self.assertIn(funcid, script)
        self.assertIn(funcid2, script)
        self.assertCommandExist(funcid)
        self.assertCommandExist(funcid2)

    def _test_tag_unbind(self, w):
        tag = 'sel'
        event = '<Control-Alt-Key-b>'
        w.pack()
        tag_bind = w._tag_bind if isinstance(w, tkinter.Text) else w.tag_bind
        self.assertEqual(tag_bind(tag), ())
        self.assertEqual(tag_bind(tag, event), '')
        def test1(e): pass
        def test2(e): pass

        funcid = w.tag_bind(tag, event, test1)
        funcid2 = w.tag_bind(tag, event, test2, add=True)

        self.assertRaises(TypeError, w.tag_unbind, tag)
        w.tag_unbind(tag, event)
        self.assertEqual(tag_bind(tag, event), '')
        self.assertEqual(tag_bind(tag), ())

    def _test_tag_bind_rebind(self, w):
        tag = 'sel'
        event = '<Control-Alt-Key-d>'
        w.pack()
        tag_bind = w._tag_bind if isinstance(w, tkinter.Text) else w.tag_bind
        self.assertEqual(tag_bind(tag), ())
        self.assertEqual(tag_bind(tag, event), '')
        def test1(e): pass
        def test2(e): pass
        def test3(e): pass

        funcid = w.tag_bind(tag, event, test1)
        funcid2 = w.tag_bind(tag, event, test2, add=True)
        script = tag_bind(tag, event)
        self.assertIn(funcid2, script)
        self.assertIn(funcid, script)
        self.assertCommandExist(funcid)
        self.assertCommandExist(funcid2)

        funcid3 = w.tag_bind(tag, event, test3)
        script = tag_bind(tag, event)
        self.assertNotIn(funcid, script)
        self.assertNotIn(funcid2, script)
        self.assertIn(funcid3, script)
        self.assertCommandExist(funcid3)

    def test_canvas_tag_bind(self):
        c = tkinter.Canvas(self.frame)
        self._test_tag_bind(c)

    def test_canvas_tag_unbind(self):
        c = tkinter.Canvas(self.frame)
        self._test_tag_unbind(c)

    def test_canvas_tag_bind_rebind(self):
        c = tkinter.Canvas(self.frame)
        self._test_tag_bind_rebind(c)

    def test_text_tag_bind(self):
        t = tkinter.Text(self.frame)
        self._test_tag_bind(t)

    def test_text_tag_unbind(self):
        t = tkinter.Text(self.frame)
        self._test_tag_unbind(t)

    def test_text_tag_bind_rebind(self):
        t = tkinter.Text(self.frame)
        self._test_tag_bind_rebind(t)

    def test_bindtags(self):
        f = self.frame
        self.assertEqual(self.root.bindtags(), ('.', 'Tk', 'all'))
        self.assertEqual(f.bindtags(), (str(f), 'Test', '.', 'all'))
        f.bindtags(('a', 'b c'))
        self.assertEqual(f.bindtags(), ('a', 'b c'))

    def test_bind_events(self):
        event = '<Enter>'
        root = self.root
        t = tkinter.Toplevel(root)
        f = tkinter.Frame(t, class_='Test', width=150, height=100)
        f.pack()
        root.wait_visibility()  # needed on Windows
        root.update_idletasks()
        self.addCleanup(root.unbind_class, 'Test', event)
        self.addCleanup(root.unbind_class, 'Toplevel', event)
        self.addCleanup(root.unbind_class, 'tag', event)
        self.addCleanup(root.unbind_class, 'tag2', event)
        self.addCleanup(root.unbind_all, event)
        def test(what):
            return lambda e: events.append((what, e.widget))

        root.bind_all(event, test('all'))
        root.bind_class('Test', event, test('frame class'))
        root.bind_class('Toplevel', event, test('toplevel class'))
        root.bind_class('tag', event, test('tag'))
        root.bind_class('tag2', event, test('tag2'))
        f.bind(event, test('frame'))
        t.bind(event, test('toplevel'))

        events = []
        f.event_generate(event)
        self.assertEqual(events, [
            ('frame', f),
            ('frame class', f),
            ('toplevel', f),
            ('all', f),
        ])

        events = []
        t.event_generate(event)
        self.assertEqual(events, [
            ('toplevel', t),
            ('toplevel class', t),
            ('all', t),
        ])

        f.bindtags(('tag', 'tag3'))
        events = []
        f.event_generate(event)
        self.assertEqual(events, [('tag', f)])


class DefaultRootTest(AbstractDefaultRootTest, unittest.TestCase):

    def test_default_root(self):
        self.assertIs(tkinter._support_default_root, True)
        self.assertIsNone(tkinter._default_root)
        root = tkinter.Tk()
        root2 = tkinter.Tk()
        root3 = tkinter.Tk()
        self.assertIs(tkinter._default_root, root)
        root2.destroy()
        self.assertIs(tkinter._default_root, root)
        root.destroy()
        self.assertIsNone(tkinter._default_root)
        root3.destroy()
        self.assertIsNone(tkinter._default_root)

    def test_no_default_root(self):
        self.assertIs(tkinter._support_default_root, True)
        self.assertIsNone(tkinter._default_root)
        root = tkinter.Tk()
        self.assertIs(tkinter._default_root, root)
        tkinter.NoDefaultRoot()
        self.assertIs(tkinter._support_default_root, False)
        self.assertNotHasAttr(tkinter, '_default_root')
        # repeated call is no-op
        tkinter.NoDefaultRoot()
        self.assertIs(tkinter._support_default_root, False)
        self.assertNotHasAttr(tkinter, '_default_root')
        root.destroy()
        self.assertIs(tkinter._support_default_root, False)
        self.assertNotHasAttr(tkinter, '_default_root')
        root = tkinter.Tk()
        self.assertIs(tkinter._support_default_root, False)
        self.assertNotHasAttr(tkinter, '_default_root')
        root.destroy()

    def test_getboolean(self):
        self.assertRaises(RuntimeError, tkinter.getboolean, '1')
        root = tkinter.Tk()
        self.assertIs(tkinter.getboolean('1'), True)
        self.assertRaises(ValueError, tkinter.getboolean, 'yea')
        root.destroy()
        tkinter.NoDefaultRoot()
        self.assertRaises(RuntimeError, tkinter.getboolean, '1')

    def test_mainloop(self):
        self.assertRaises(RuntimeError, tkinter.mainloop)
        root = tkinter.Tk()
        root.after_idle(root.quit)
        tkinter.mainloop()
        root.destroy()
        tkinter.NoDefaultRoot()
        self.assertRaises(RuntimeError, tkinter.mainloop)


def _info_commands(widget, pattern=None):
    return widget.tk.splitlist(widget.tk.call('info', 'commands', pattern))


if __name__ == "__main__":
    unittest.main()

### File: test_simpledialog.py

In [ ]:
import unittest
import tkinter
from test.support import requires, swap_attr
from test.test_tkinter.support import AbstractDefaultRootTest
from tkinter.simpledialog import Dialog, askinteger

requires('gui')


class DefaultRootTest(AbstractDefaultRootTest, unittest.TestCase):

    def test_askinteger(self):
        @staticmethod
        def mock_wait_window(w):
            nonlocal ismapped
            ismapped = w.master.winfo_ismapped()
            w.destroy()

        with swap_attr(Dialog, 'wait_window', mock_wait_window):
            ismapped = None
            askinteger("Go To Line", "Line number")
            self.assertEqual(ismapped, False)

            root = tkinter.Tk()
            ismapped = None
            askinteger("Go To Line", "Line number")
            self.assertEqual(ismapped, True)
            root.destroy()

            tkinter.NoDefaultRoot()
            self.assertRaises(RuntimeError, askinteger, "Go To Line", "Line number")


if __name__ == "__main__":
    unittest.main()

### File: test_text.py

In [ ]:
import unittest
import tkinter
from test.support import requires
from test.test_tkinter.support import AbstractTkTest

requires('gui')

class TextTest(AbstractTkTest, unittest.TestCase):

    def setUp(self):
        super().setUp()
        self.text = tkinter.Text(self.root)
        self.text.pack()

    def test_debug(self):
        text = self.text
        olddebug = text.debug()
        try:
            text.debug(0)
            self.assertEqual(text.debug(), 0)
            text.debug(1)
            self.assertEqual(text.debug(), 1)
        finally:
            text.debug(olddebug)
            self.assertEqual(text.debug(), olddebug)

    def test_search(self):
        text = self.text

        # pattern and index are obligatory arguments.
        self.assertRaises(tkinter.TclError, text.search, None, '1.0')
        self.assertRaises(tkinter.TclError, text.search, 'a', None)
        self.assertRaises(tkinter.TclError, text.search, None, None)

        # Invalid text index.
        self.assertRaises(tkinter.TclError, text.search, '', 0)

        # Check if we are getting the indices as strings -- you are likely
        # to get Tcl_Obj under Tk 8.5 if Tkinter doesn't convert it.
        text.insert('1.0', 'hi-test')
        self.assertEqual(text.search('-test', '1.0', 'end'), '1.2')
        self.assertEqual(text.search('test', '1.0', 'end'), '1.3')

    def test_count(self):
        text = self.text
        text.insert('1.0',
            'Lorem ipsum dolor sit amet,\n'
            'consectetur adipiscing elit,\n'
            'sed do eiusmod tempor incididunt\n'
            'ut labore et dolore magna aliqua.')

        options = ('chars', 'indices', 'lines',
                   'displaychars', 'displayindices', 'displaylines',
                   'xpixels', 'ypixels')
        self.assertEqual(len(text.count('1.0', 'end', *options, return_ints=True)), 8)
        self.assertEqual(len(text.count('1.0', 'end', *options)), 8)
        self.assertEqual(text.count('1.0', 'end', 'chars', 'lines', return_ints=True),
                         (124, 4))
        self.assertEqual(text.count('1.3', '4.5', 'chars', 'lines'), (92, 3))
        self.assertEqual(text.count('4.5', '1.3', 'chars', 'lines', return_ints=True),
                         (-92, -3))
        self.assertEqual(text.count('4.5', '1.3', 'chars', 'lines'), (-92, -3))
        self.assertEqual(text.count('1.3', '1.3', 'chars', 'lines', return_ints=True),
                         (0, 0))
        self.assertEqual(text.count('1.3', '1.3', 'chars', 'lines'), (0, 0))
        self.assertEqual(text.count('1.0', 'end', 'lines', return_ints=True), 4)
        self.assertEqual(text.count('1.0', 'end', 'lines'), (4,))
        self.assertEqual(text.count('end', '1.0', 'lines', return_ints=True), -4)
        self.assertEqual(text.count('end', '1.0', 'lines'), (-4,))
        self.assertEqual(text.count('1.3', '1.5', 'lines', return_ints=True), 0)
        self.assertEqual(text.count('1.3', '1.5', 'lines'), None)
        self.assertEqual(text.count('1.3', '1.3', 'lines', return_ints=True), 0)
        self.assertEqual(text.count('1.3', '1.3', 'lines'), None)
        # Count 'indices' by default.
        self.assertEqual(text.count('1.0', 'end', return_ints=True), 124)
        self.assertEqual(text.count('1.0', 'end'), (124,))
        self.assertEqual(text.count('1.0', 'end', 'indices', return_ints=True), 124)
        self.assertEqual(text.count('1.0', 'end', 'indices'), (124,))
        self.assertRaises(tkinter.TclError, text.count, '1.0', 'end', 'spam')
        self.assertRaises(tkinter.TclError, text.count, '1.0', 'end', '-lines')

        self.assertIsInstance(text.count('1.3', '1.5', 'ypixels', return_ints=True), int)
        self.assertIsInstance(text.count('1.3', '1.5', 'ypixels'), tuple)
        self.assertIsInstance(text.count('1.3', '1.5', 'update', 'ypixels', return_ints=True), int)
        self.assertIsInstance(text.count('1.3', '1.5', 'update', 'ypixels'), int)
        self.assertEqual(text.count('1.3', '1.3', 'update', 'ypixels', return_ints=True), 0)
        self.assertEqual(text.count('1.3', '1.3', 'update', 'ypixels'), None)
        self.assertEqual(text.count('1.3', '1.5', 'update', 'indices', return_ints=True), 2)
        self.assertEqual(text.count('1.3', '1.5', 'update', 'indices'), 2)
        self.assertEqual(text.count('1.3', '1.3', 'update', 'indices', return_ints=True), 0)
        self.assertEqual(text.count('1.3', '1.3', 'update', 'indices'), None)
        self.assertEqual(text.count('1.3', '1.5', 'update', return_ints=True), 2)
        self.assertEqual(text.count('1.3', '1.5', 'update'), (2,))
        self.assertEqual(text.count('1.3', '1.3', 'update', return_ints=True), 0)
        self.assertEqual(text.count('1.3', '1.3', 'update'), None)


if __name__ == "__main__":
    unittest.main()

### File: test_variables.py

In [ ]:
import unittest
from test import support

import gc
import tkinter
from tkinter import (Variable, StringVar, IntVar, DoubleVar, BooleanVar, Tcl,
                     TclError)
from test.support import ALWAYS_EQ
from test.test_tkinter.support import AbstractDefaultRootTest, tcl_version


class Var(Variable):

    _default = "default"
    side_effect = False

    def set(self, value):
        self.side_effect = True
        super().set(value)


class TestBase(unittest.TestCase):

    def setUp(self):
        self.root = Tcl()

    def tearDown(self):
        del self.root


class TestVariable(TestBase):

    def info_exists(self, *args):
        return self.root.getboolean(self.root.call("info", "exists", *args))

    def test_default(self):
        v = Variable(self.root)
        self.assertEqual("", v.get())
        self.assertRegex(str(v), r"^PY_VAR(\d+)$")

    def test_name_and_value(self):
        v = Variable(self.root, "sample string", "varname")
        self.assertEqual("sample string", v.get())
        self.assertEqual("varname", str(v))

    def test___del__(self):
        self.assertFalse(self.info_exists("varname"))
        v = Variable(self.root, "sample string", "varname")
        self.assertTrue(self.info_exists("varname"))
        del v
        support.gc_collect()  # For PyPy or other GCs.
        self.assertFalse(self.info_exists("varname"))

    def test_dont_unset_not_existing(self):
        self.assertFalse(self.info_exists("varname"))
        v1 = Variable(self.root, name="name")
        v2 = Variable(self.root, name="name")
        del v1
        support.gc_collect()  # For PyPy or other GCs.
        self.assertFalse(self.info_exists("name"))
        # shouldn't raise exception
        del v2
        support.gc_collect()  # For PyPy or other GCs.
        self.assertFalse(self.info_exists("name"))

    def test_equality(self):
        # values doesn't matter, only class and name are checked
        v1 = Variable(self.root, name="abc")
        v2 = Variable(self.root, name="abc")
        self.assertIsNot(v1, v2)
        self.assertEqual(v1, v2)

        v3 = Variable(self.root, name="cba")
        self.assertNotEqual(v1, v3)

        v4 = StringVar(self.root, name="abc")
        self.assertEqual(str(v1), str(v4))
        self.assertNotEqual(v1, v4)

        V = type('Variable', (), {})
        self.assertNotEqual(v1, V())

        self.assertNotEqual(v1, object())
        self.assertEqual(v1, ALWAYS_EQ)

        root2 = tkinter.Tk()
        self.addCleanup(root2.destroy)
        v5 = Variable(root2, name="abc")
        self.assertEqual(str(v1), str(v5))
        self.assertNotEqual(v1, v5)

    def test_invalid_name(self):
        with self.assertRaises(TypeError):
            Variable(self.root, name=123)

    def test_null_in_name(self):
        with self.assertRaises(ValueError):
            Variable(self.root, name='var\x00name')
        with self.assertRaises(ValueError):
            self.root.globalsetvar('var\x00name', "value")
        with self.assertRaises(ValueError):
            self.root.globalsetvar(b'var\x00name', "value")
        with self.assertRaises(ValueError):
            self.root.setvar('var\x00name', "value")
        with self.assertRaises(ValueError):
            self.root.setvar(b'var\x00name', "value")

    def test_initialize(self):
        v = Var(self.root)
        self.assertFalse(v.side_effect)
        v.set("value")
        self.assertTrue(v.side_effect)

    def test_trace_old(self):
        if tcl_version >= (9, 0):
            self.skipTest('requires Tcl version < 9.0')
        # Old interface
        v = Variable(self.root)
        vname = str(v)
        trace = []
        def read_tracer(*args):
            trace.append(('read',) + args)
        def write_tracer(*args):
            trace.append(('write',) + args)
        with self.assertWarns(DeprecationWarning) as cm:
            cb1 = v.trace_variable('r', read_tracer)
        self.assertEqual(cm.filename, __file__)
        with self.assertWarns(DeprecationWarning):
            cb2 = v.trace_variable('wu', write_tracer)
        with self.assertWarns(DeprecationWarning) as cm:
            self.assertEqual(sorted(v.trace_vinfo()), [('r', cb1), ('wu', cb2)])
        self.assertEqual(cm.filename, __file__)
        self.assertEqual(trace, [])

        v.set('spam')
        self.assertEqual(trace, [('write', vname, '', 'w')])

        trace = []
        v.get()
        self.assertEqual(trace, [('read', vname, '', 'r')])

        trace = []
        with self.assertWarns(DeprecationWarning):
            info = sorted(v.trace_vinfo())
        with self.assertWarns(DeprecationWarning):
            v.trace_vdelete('w', cb1)  # Wrong mode
        with self.assertWarns(DeprecationWarning):
            self.assertEqual(sorted(v.trace_vinfo()), info)
        with self.assertRaises(TclError):
            with self.assertWarns(DeprecationWarning):
                v.trace_vdelete('r', 'spam')  # Wrong command name
        with self.assertWarns(DeprecationWarning):
            self.assertEqual(sorted(v.trace_vinfo()), info)
        with self.assertWarns(DeprecationWarning):
            v.trace_vdelete('r', (cb1, 43)) # Wrong arguments
        with self.assertWarns(DeprecationWarning):
            self.assertEqual(sorted(v.trace_vinfo()), info)
        v.get()
        self.assertEqual(trace, [('read', vname, '', 'r')])

        trace = []
        with self.assertWarns(DeprecationWarning) as cm:
            v.trace_vdelete('r', cb1)
        self.assertEqual(cm.filename, __file__)
        with self.assertWarns(DeprecationWarning):
            self.assertEqual(v.trace_vinfo(), [('wu', cb2)])
        v.get()
        self.assertEqual(trace, [])

        trace = []
        del write_tracer
        gc.collect()
        v.set('eggs')
        self.assertEqual(trace, [('write', vname, '', 'w')])

        trace = []
        del v
        gc.collect()
        self.assertEqual(trace, [('write', vname, '', 'u')])

    def test_trace(self):
        v = Variable(self.root)
        vname = str(v)
        trace = []
        def read_tracer(*args):
            trace.append(('read',) + args)
        def write_tracer(*args):
            trace.append(('write',) + args)
        tr1 = v.trace_add('read', read_tracer)
        tr2 = v.trace_add(['write', 'unset'], write_tracer)
        self.assertEqual(sorted(v.trace_info()), [
                         (('read',), tr1),
                         (('write', 'unset'), tr2)])
        self.assertEqual(trace, [])

        v.set('spam')
        self.assertEqual(trace, [('write', vname, '', 'write')])

        trace = []
        v.get()
        self.assertEqual(trace, [('read', vname, '', 'read')])

        trace = []
        info = sorted(v.trace_info())
        v.trace_remove('write', tr1)  # Wrong mode
        self.assertEqual(sorted(v.trace_info()), info)
        with self.assertRaises(TclError):
            v.trace_remove('read', 'spam')  # Wrong command name
        self.assertEqual(sorted(v.trace_info()), info)
        v.get()
        self.assertEqual(trace, [('read', vname, '', 'read')])

        trace = []
        v.trace_remove('read', tr1)
        self.assertEqual(v.trace_info(), [(('write', 'unset'), tr2)])
        v.get()
        self.assertEqual(trace, [])

        trace = []
        del write_tracer
        gc.collect()
        v.set('eggs')
        self.assertEqual(trace, [('write', vname, '', 'write')])

        trace = []
        del v
        gc.collect()
        self.assertEqual(trace, [('write', vname, '', 'unset')])


class TestStringVar(TestBase):

    def test_default(self):
        v = StringVar(self.root)
        self.assertEqual("", v.get())

    def test_get(self):
        v = StringVar(self.root, "abc", "name")
        self.assertEqual("abc", v.get())
        self.root.globalsetvar("name", "value")
        self.assertEqual("value", v.get())

    def test_get_null(self):
        v = StringVar(self.root, "abc\x00def", "name")
        self.assertEqual("abc\x00def", v.get())
        self.root.globalsetvar("name", "val\x00ue")
        self.assertEqual("val\x00ue", v.get())


class TestIntVar(TestBase):

    def test_default(self):
        v = IntVar(self.root)
        self.assertEqual(0, v.get())

    def test_get(self):
        v = IntVar(self.root, 123, "name")
        self.assertEqual(123, v.get())
        self.root.globalsetvar("name", "345")
        self.assertEqual(345, v.get())
        self.root.globalsetvar("name", "876.5")
        self.assertEqual(876, v.get())

    def test_invalid_value(self):
        v = IntVar(self.root, name="name")
        self.root.globalsetvar("name", "value")
        with self.assertRaises((ValueError, TclError)):
            v.get()


class TestDoubleVar(TestBase):

    def test_default(self):
        v = DoubleVar(self.root)
        self.assertEqual(0.0, v.get())

    def test_get(self):
        v = DoubleVar(self.root, 1.23, "name")
        self.assertAlmostEqual(1.23, v.get())
        self.root.globalsetvar("name", "3.45")
        self.assertAlmostEqual(3.45, v.get())

    def test_get_from_int(self):
        v = DoubleVar(self.root, 1.23, "name")
        self.assertAlmostEqual(1.23, v.get())
        self.root.globalsetvar("name", "3.45")
        self.assertAlmostEqual(3.45, v.get())
        self.root.globalsetvar("name", "456")
        self.assertAlmostEqual(456, v.get())

    def test_invalid_value(self):
        v = DoubleVar(self.root, name="name")
        self.root.globalsetvar("name", "value")
        with self.assertRaises((ValueError, TclError)):
            v.get()


class TestBooleanVar(TestBase):

    def test_default(self):
        v = BooleanVar(self.root)
        self.assertIs(v.get(), False)

    def test_get(self):
        v = BooleanVar(self.root, True, "name")
        self.assertIs(v.get(), True)
        self.root.globalsetvar("name", "0")
        self.assertIs(v.get(), False)
        self.root.globalsetvar("name", 42 if self.root.wantobjects() else 1)
        self.assertIs(v.get(), True)
        self.root.globalsetvar("name", 0)
        self.assertIs(v.get(), False)
        self.root.globalsetvar("name", "on")
        self.assertIs(v.get(), True)

    def test_set(self):
        true = 1 if self.root.wantobjects() else "1"
        false = 0 if self.root.wantobjects() else "0"
        v = BooleanVar(self.root, name="name")
        v.set(True)
        self.assertEqual(self.root.globalgetvar("name"), true)
        v.set("0")
        self.assertEqual(self.root.globalgetvar("name"), false)
        v.set(42)
        self.assertEqual(self.root.globalgetvar("name"), true)
        v.set(0)
        self.assertEqual(self.root.globalgetvar("name"), false)
        v.set("on")
        self.assertEqual(self.root.globalgetvar("name"), true)

    def test_invalid_value_domain(self):
        false = 0 if self.root.wantobjects() else "0"
        v = BooleanVar(self.root, name="name")
        with self.assertRaises(TclError):
            v.set("value")
        self.assertEqual(self.root.globalgetvar("name"), false)
        self.root.globalsetvar("name", "value")
        with self.assertRaises(ValueError):
            v.get()
        self.root.globalsetvar("name", "1.0")
        with self.assertRaises(ValueError):
            v.get()


class DefaultRootTest(AbstractDefaultRootTest, unittest.TestCase):

    def test_variable(self):
        self.assertRaises(RuntimeError, Variable)
        root = tkinter.Tk()
        v = Variable()
        v.set("value")
        self.assertEqual(v.get(), "value")
        root.destroy()
        tkinter.NoDefaultRoot()
        self.assertRaises(RuntimeError, Variable)


if __name__ == "__main__":
    unittest.main()

### File: test_widgets.py

In [ ]:
import unittest
import tkinter
from tkinter import TclError
import os
from test.support import requires

from test.test_tkinter.support import (requires_tk, tk_version,
                                  get_tk_patchlevel, widget_eq,
                                  AbstractDefaultRootTest)

from test.test_tkinter.widget_tests import (
    add_configure_tests,
    AbstractWidgetTest,
    StandardOptionsTests,
    IntegerSizeTests,
    PixelSizeTests)

requires('gui')


EXPECTED_SCREEN_DISTANCE_ERRMSG = '(bad|expected) screen distance (but got )?"{}"'
EXPECTED_SCREEN_DISTANCE_OR_EMPTY_ERRMSG = '(bad|expected) screen distance (or "" but got )?"{}"'

def float_round(x):
    return float(round(x))

class AbstractToplevelTest(AbstractWidgetTest, PixelSizeTests):
    if tk_version < (9, 0):
        _no_round = {'padx', 'pady'}
    else:
        _no_round = {'borderwidth', 'height', 'highlightthickness', 'padx',
                     'pady', 'width'}
    if tk_version < (9, 0):
        _clipped = {'highlightthickness'}
    else:
        _clipped = {'borderwidth', 'height', 'highlightthickness', 'padx',
                    'pady', 'width'}

    def test_configure_class(self):
        widget = self.create()
        self.assertEqual(widget['class'],
                         widget.__class__.__name__.title())
        self.checkInvalidParam(widget, 'class', 'Foo',
                errmsg="can't modify -class option after widget is created")
        widget2 = self.create(class_='Foo')
        self.assertEqual(widget2['class'], 'Foo')

    def test_configure_colormap(self):
        widget = self.create()
        self.assertEqual(widget['colormap'], '')
        self.checkInvalidParam(widget, 'colormap', 'new',
                errmsg="can't modify -colormap option after widget is created")
        widget2 = self.create(colormap='new')
        self.assertEqual(widget2['colormap'], 'new')

    def test_configure_container(self):
        widget = self.create()
        self.assertEqual(widget['container'], 0 if self.wantobjects else '0')
        self.checkInvalidParam(widget, 'container', 1,
                errmsg="can't modify -container option after widget is created")
        widget2 = self.create(container=True)
        self.assertEqual(widget2['container'], 1 if self.wantobjects else '1')

    def test_configure_visual(self):
        widget = self.create()
        self.assertEqual(widget['visual'], '')
        self.checkInvalidParam(widget, 'visual', 'default',
                errmsg="can't modify -visual option after widget is created")
        widget2 = self.create(visual='default')
        self.assertEqual(widget2['visual'], 'default')


@add_configure_tests(StandardOptionsTests)
class ToplevelTest(AbstractToplevelTest, unittest.TestCase):
    OPTIONS = (
        'background', 'backgroundimage', 'borderwidth',
        'class', 'colormap', 'container', 'cursor', 'height',
        'highlightbackground', 'highlightcolor', 'highlightthickness',
        'menu', 'padx', 'pady', 'relief', 'screen',
        'takefocus', 'tile', 'use', 'visual', 'width',
    )

    def create(self, **kwargs):
        return tkinter.Toplevel(self.root, **kwargs)

    def test_configure_menu(self):
        widget = self.create()
        menu = tkinter.Menu(self.root)
        self.checkParam(widget, 'menu', menu, eq=widget_eq)
        self.checkParam(widget, 'menu', '')

    def test_configure_screen(self):
        widget = self.create()
        if widget._windowingsystem != 'x11':
            self.skipTest('Not using Tk for X11')
        self.assertEqual(widget['screen'], '')
        try:
            display = os.environ['DISPLAY']
        except KeyError:
            self.skipTest('No $DISPLAY set.')
        self.checkInvalidParam(widget, 'screen', display,
                errmsg="can't modify -screen option after widget is created")
        widget2 = self.create(screen=display)
        self.assertEqual(widget2['screen'], display)

    def test_configure_use(self):
        widget = self.create()
        self.assertEqual(widget['use'], '')
        parent = self.create(container=True)
        wid = hex(parent.winfo_id())
        with self.subTest(wid=wid):
            widget2 = self.create(use=wid)
            self.assertEqual(widget2['use'], wid)


@add_configure_tests(StandardOptionsTests)
class FrameTest(AbstractToplevelTest, unittest.TestCase):
    OPTIONS = (
        'background', 'backgroundimage', 'borderwidth',
        'class', 'colormap', 'container', 'cursor', 'height',
        'highlightbackground', 'highlightcolor', 'highlightthickness',
        'padx', 'pady', 'relief', 'takefocus', 'tile', 'visual', 'width',
    )
    if tk_version < (9, 0):
        _no_round = {'padx', 'pady'}
    else:
        _no_round = {'borderwidth', 'height', 'highlightthickness', 'padx',
                     'pady', 'width'}

    def create(self, **kwargs):
        return tkinter.Frame(self.root, **kwargs)


@add_configure_tests(StandardOptionsTests)
class LabelFrameTest(AbstractToplevelTest, unittest.TestCase):
    OPTIONS = (
        'background', 'borderwidth',
        'class', 'colormap', 'container', 'cursor',
        'font', 'foreground', 'height',
        'highlightbackground', 'highlightcolor', 'highlightthickness',
        'labelanchor', 'labelwidget', 'padx', 'pady', 'relief',
        'takefocus', 'text', 'visual', 'width',
    )
    if tk_version < (9, 0):
        _no_round = {'padx', 'pady'}
    else:
        _no_round = {'borderwidth', 'height', 'highlightthickness', 'padx',
                     'pady', 'width'}

    def create(self, **kwargs):
        return tkinter.LabelFrame(self.root, **kwargs)

    def test_configure_labelanchor(self):
        widget = self.create()
        self.checkEnumParam(widget, 'labelanchor',
                            'e', 'en', 'es', 'n', 'ne', 'nw',
                            's', 'se', 'sw', 'w', 'wn', 'ws')
        self.checkInvalidParam(widget, 'labelanchor', 'center')

    def test_configure_labelwidget(self):
        widget = self.create()
        label = tkinter.Label(self.root, text='Mupp', name='foo')
        self.checkParam(widget, 'labelwidget', label, expected='.foo')
        label.destroy()

# Label, Button, Checkbutton, Radiobutton, MenuButton
class AbstractLabelTest(AbstractWidgetTest, IntegerSizeTests):
    _rounds_pixels = False
    if tk_version < (9, 0):
        _clipped = {}
    else:
        _clipped = {'borderwidth', 'insertborderwidth', 'highlightthickness',
                    'padx', 'pady'}

@add_configure_tests(StandardOptionsTests)
class LabelTest(AbstractLabelTest, unittest.TestCase):
    OPTIONS = (
        'activebackground', 'activeforeground', 'anchor',
        'background', 'bitmap', 'borderwidth', 'compound', 'cursor',
        'disabledforeground', 'font', 'foreground', 'height',
        'highlightbackground', 'highlightcolor', 'highlightthickness',
        'image', 'justify', 'padx', 'pady', 'relief', 'state',
        'takefocus', 'text', 'textvariable',
        'underline', 'width', 'wraplength',
    )

    def create(self, **kwargs):
        return tkinter.Label(self.root, **kwargs)


@add_configure_tests(StandardOptionsTests)
class ButtonTest(AbstractLabelTest, unittest.TestCase):
    OPTIONS = (
        'activebackground', 'activeforeground', 'anchor',
        'background', 'bitmap', 'borderwidth',
        'command', 'compound', 'cursor', 'default',
        'disabledforeground', 'font', 'foreground', 'height',
        'highlightbackground', 'highlightcolor', 'highlightthickness',
        'image', 'justify', 'overrelief', 'padx', 'pady', 'relief',
        'repeatdelay', 'repeatinterval',
        'state', 'takefocus', 'text', 'textvariable',
        'underline', 'width', 'wraplength')

    def create(self, **kwargs):
        return tkinter.Button(self.root, **kwargs)

    def test_configure_default(self):
        widget = self.create()
        self.checkEnumParam(widget, 'default', 'active', 'disabled', 'normal')


@add_configure_tests(StandardOptionsTests)
class CheckbuttonTest(AbstractLabelTest, unittest.TestCase):
    OPTIONS = (
        'activebackground', 'activeforeground', 'anchor',
        'background', 'bitmap', 'borderwidth',
        'command', 'compound', 'cursor',
        'disabledforeground', 'font', 'foreground', 'height',
        'highlightbackground', 'highlightcolor', 'highlightthickness',
        'image', 'indicatoron', 'justify',
        'offrelief', 'offvalue', 'onvalue', 'overrelief',
        'padx', 'pady', 'relief', 'selectcolor', 'selectimage', 'state',
        'takefocus', 'text', 'textvariable',
        'tristateimage', 'tristatevalue',
        'underline', 'variable', 'width', 'wraplength',
    )

    def create(self, **kwargs):
        return tkinter.Checkbutton(self.root, **kwargs)


    def test_configure_offvalue(self):
        widget = self.create()
        self.checkParams(widget, 'offvalue', 1, 2.3, '', 'any string')

    def test_configure_onvalue(self):
        widget = self.create()
        self.checkParams(widget, 'onvalue', 1, 2.3, '', 'any string')

    def test_unique_variables(self):
        frames = []
        buttons = []
        for i in range(2):
            f = tkinter.Frame(self.root)
            f.pack()
            frames.append(f)
            for j in 'AB':
                b = tkinter.Checkbutton(f, text=j)
                b.pack()
                buttons.append(b)
        variables = [str(b['variable']) for b in buttons]
        self.assertEqual(len(set(variables)), 4, variables)

    def test_same_name(self):
        f1 = tkinter.Frame(self.root)
        f2 = tkinter.Frame(self.root)
        b1 = tkinter.Checkbutton(f1, name='test', text='Test1')
        b2 = tkinter.Checkbutton(f2, name='test', text='Test2')

        v = tkinter.IntVar(self.root, name='test')
        b1.select()
        self.assertEqual(v.get(), 1)
        b2.deselect()
        self.assertEqual(v.get(), 0)

@add_configure_tests(StandardOptionsTests)
class RadiobuttonTest(AbstractLabelTest, unittest.TestCase):
    OPTIONS = (
        'activebackground', 'activeforeground', 'anchor',
        'background', 'bitmap', 'borderwidth',
        'command', 'compound', 'cursor',
        'disabledforeground', 'font', 'foreground', 'height',
        'highlightbackground', 'highlightcolor', 'highlightthickness',
        'image', 'indicatoron', 'justify', 'offrelief', 'overrelief',
        'padx', 'pady', 'relief', 'selectcolor', 'selectimage', 'state',
        'takefocus', 'text', 'textvariable',
        'tristateimage', 'tristatevalue',
        'underline', 'value', 'variable', 'width', 'wraplength',
    )

    def create(self, **kwargs):
        return tkinter.Radiobutton(self.root, **kwargs)

    def test_configure_value(self):
        widget = self.create()
        self.checkParams(widget, 'value', 1, 2.3, '', 'any string')


@add_configure_tests(StandardOptionsTests)
class MenubuttonTest(AbstractLabelTest, unittest.TestCase):
    OPTIONS = (
        'activebackground', 'activeforeground', 'anchor',
        'background', 'bitmap', 'borderwidth',
        'compound', 'cursor', 'direction',
        'disabledforeground', 'font', 'foreground', 'height',
        'highlightbackground', 'highlightcolor', 'highlightthickness',
        'image', 'indicatoron', 'justify', 'menu',
        'padx', 'pady', 'relief', 'state',
        'takefocus', 'text', 'textvariable',
        'underline', 'width', 'wraplength',
    )
    _rounds_pixels = (tk_version < (9, 0))
    if tk_version < (9, 0):
        _clipped = {'highlightthickness', 'padx', 'pady'}
    else:
        _clipped ={ 'insertborderwidth', 'highlightthickness', 'padx', 'pady'}

    def create(self, **kwargs):
        return tkinter.Menubutton(self.root, **kwargs)

    def test_configure_direction(self):
        widget = self.create()
        self.checkEnumParam(widget, 'direction',
                'above', 'below', 'flush', 'left', 'right')

    def test_configure_height(self):
        widget = self.create()
        if tk_version < (9, 0):
            self.checkIntegerParam(widget, 'height', 100, -100, 0, conv=str)
        else:
            self.checkIntegerParam(widget, 'height', 0, -100, 0)

    def test_configure_image(self):
        widget = self.create()
        image = tkinter.PhotoImage(master=self.root, name='image1')
        self.checkParam(widget, 'image', image, conv=str)
        if tk_version < (9, 0):
            errmsg = 'image "spam" doesn\'t exist'
        else:
            errmsg = 'image "spam" does not exist'
        with self.assertRaises(tkinter.TclError) as cm:
            widget['image'] = 'spam'
        if errmsg is not None:
            self.assertEqual(str(cm.exception), errmsg)
        with self.assertRaises(tkinter.TclError) as cm:
            widget.configure({'image': 'spam'})
        if errmsg is not None:
            self.assertEqual(str(cm.exception), errmsg)

    def test_configure_menu(self):
        widget = self.create()
        menu = tkinter.Menu(widget, name='menu')
        self.checkParam(widget, 'menu', menu, eq=widget_eq)
        menu.destroy()

    def test_configure_width(self):
        widget = self.create()
        if tk_version < (9, 0):
            self.checkIntegerParam(widget, 'width', 402, -402, 0, conv=str)
        else:
            self.checkIntegerParam(widget, 'width', 402, 0, 0)


class OptionMenuTest(MenubuttonTest, unittest.TestCase):

    def create(self, default='b', values=('a', 'b', 'c'), **kwargs):
        return tkinter.OptionMenu(self.root, None, default, *values, **kwargs)

    def test_bad_kwarg(self):
        with self.assertRaisesRegex(TclError, r"^unknown option -image$"):
            tkinter.OptionMenu(self.root, None, 'b', image='')

    def test_specify_name(self):
        widget = tkinter.OptionMenu(self.root, None, ':)', name="option_menu")
        self.assertEqual(str(widget), ".option_menu")
        self.assertIs(self.root.children["option_menu"], widget)

@add_configure_tests(IntegerSizeTests, StandardOptionsTests)
class EntryTest(AbstractWidgetTest, unittest.TestCase):
    _rounds_pixels = (tk_version < (9, 0))
    if tk_version < (9, 0):
        _clipped = {'highlightthickness'}
    else:
        _clipped = {'highlightthickness', 'borderwidth', 'insertborderwidth',
                    'selectborderwidth'}

    OPTIONS = (
        'background', 'borderwidth', 'cursor',
        'disabledbackground', 'disabledforeground',
        'exportselection', 'font', 'foreground',
        'highlightbackground', 'highlightcolor', 'highlightthickness',
        'insertbackground', 'insertborderwidth',
        'insertofftime', 'insertontime', 'insertwidth',
        'invalidcommand', 'justify', 'placeholder', 'placeholderforeground',
        'readonlybackground', 'relief',
        'selectbackground', 'selectborderwidth', 'selectforeground',
        'show', 'state', 'takefocus', 'textvariable',
        'validate', 'validatecommand', 'width', 'xscrollcommand',
    )

    def create(self, **kwargs):
        return tkinter.Entry(self.root, **kwargs)

    def test_configure_disabledbackground(self):
        widget = self.create()
        self.checkColorParam(widget, 'disabledbackground')

    def test_configure_insertborderwidth(self):
        widget = self.create(insertwidth=100)
        if tk_version < (9, 0):
            self.checkPixelsParam(widget, 'insertborderwidth',
                                  0, 1.3, 2.6, 6, '10p')
        else:
            self.checkPixelsParam(widget, 'insertborderwidth',
                                  0, 1.3, 3, 6, '10p')
        self.checkParam(widget, 'insertborderwidth', -2)
        # insertborderwidth is bounded above by a half of insertwidth.
        expected =  100 // 2 if tk_version < (9, 0) else 60
        self.checkParam(widget, 'insertborderwidth', 60, expected=expected)

    def test_configure_insertwidth(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'insertwidth', 1.3, 3.6, '10p')
        if tk_version < (9, 0):
            self.checkParam(widget, 'insertwidth', 0.1, expected=2)
            self.checkParam(widget, 'insertwidth', -2, expected=2)
            self.checkParam(widget, 'insertwidth', 0.9, expected=1)
        else:
            self.checkParam(widget, 'insertwidth', 0.1)
            self.checkParam(widget, 'insertwidth', -2, expected=0)
            self.checkParam(widget, 'insertwidth', 0.9)

    def test_configure_invalidcommand(self):
        widget = self.create()
        self.checkCommandParam(widget, 'invalidcommand')
        self.checkCommandParam(widget, 'invcmd')

    def test_configure_readonlybackground(self):
        widget = self.create()
        self.checkColorParam(widget, 'readonlybackground')

    def test_configure_show(self):
        widget = self.create()
        self.checkParam(widget, 'show', '*')
        self.checkParam(widget, 'show', '')
        self.checkParam(widget, 'show', ' ')

    def test_configure_state(self):
        widget = self.create()
        self.checkEnumParam(widget, 'state',
                            'disabled', 'normal', 'readonly')

    def test_configure_validate(self):
        widget = self.create()
        self.checkEnumParam(widget, 'validate',
                'all', 'key', 'focus', 'focusin', 'focusout', 'none')

    def test_configure_validatecommand(self):
        widget = self.create()
        self.checkCommandParam(widget, 'validatecommand')
        self.checkCommandParam(widget, 'vcmd')

    def test_selection_methods(self):
        widget = self.create()
        widget.insert(0, '12345')
        self.assertFalse(widget.selection_present())
        widget.selection_range(0, 'end')
        self.assertEqual(widget.selection_get(), '12345')
        self.assertTrue(widget.selection_present())
        widget.selection_from(1)
        widget.selection_to(2)
        self.assertEqual(widget.selection_get(), '2')
        widget.selection_range(3, 4)
        self.assertEqual(widget.selection_get(), '4')
        widget.selection_clear()
        self.assertFalse(widget.selection_present())
        widget.selection_range(0, 'end')
        widget.selection_adjust(4)
        self.assertEqual(widget.selection_get(), '1234')
        widget.selection_adjust(1)
        self.assertEqual(widget.selection_get(), '234')
        widget.selection_adjust(5)
        self.assertEqual(widget.selection_get(), '2345')
        widget.selection_adjust(0)
        self.assertEqual(widget.selection_get(), '12345')
        widget.selection_adjust(0)


@add_configure_tests(StandardOptionsTests)
class SpinboxTest(EntryTest, unittest.TestCase):
    OPTIONS = (
        'activebackground', 'background', 'borderwidth',
        'buttonbackground', 'buttoncursor', 'buttondownrelief', 'buttonuprelief',
        'command', 'cursor', 'disabledbackground', 'disabledforeground',
        'exportselection', 'font', 'foreground', 'format', 'from',
        'highlightbackground', 'highlightcolor', 'highlightthickness',
        'increment',
        'insertbackground', 'insertborderwidth',
        'insertofftime', 'insertontime', 'insertwidth',
        'invalidcommand', 'justify', 'placeholder', 'placeholderforeground',
        'relief', 'readonlybackground', 'repeatdelay', 'repeatinterval',
        'selectbackground', 'selectborderwidth', 'selectforeground',
        'state', 'takefocus', 'textvariable', 'to',
        'validate', 'validatecommand', 'values',
        'width', 'wrap', 'xscrollcommand',
    )

    def create(self, **kwargs):
        return tkinter.Spinbox(self.root, **kwargs)

    test_configure_show = None

    def test_configure_buttonbackground(self):
        widget = self.create()
        self.checkColorParam(widget, 'buttonbackground')

    def test_configure_buttoncursor(self):
        widget = self.create()
        self.checkCursorParam(widget, 'buttoncursor')

    def test_configure_buttondownrelief(self):
        widget = self.create()
        self.checkReliefParam(widget, 'buttondownrelief')

    def test_configure_buttonuprelief(self):
        widget = self.create()
        self.checkReliefParam(widget, 'buttonuprelief')

    def test_configure_format(self):
        widget = self.create()
        self.checkParam(widget, 'format', '%2f')
        self.checkParam(widget, 'format', '%2.2f')
        self.checkParam(widget, 'format', '%.2f')
        self.checkParam(widget, 'format', '%2.f')
        self.checkInvalidParam(widget, 'format', '%2e-1f')
        self.checkInvalidParam(widget, 'format', '2.2')
        self.checkInvalidParam(widget, 'format', '%2.-2f')
        self.checkParam(widget, 'format', '%-2.02f')
        self.checkParam(widget, 'format', '% 2.02f')
        self.checkParam(widget, 'format', '% -2.200f')
        self.checkParam(widget, 'format', '%09.200f')
        self.checkInvalidParam(widget, 'format', '%d')

    def test_configure_from(self):
        widget = self.create()
        self.checkParam(widget, 'to', 100.0)
        self.checkFloatParam(widget, 'from', -10, 10.2, 11.7)
        if tk_version >= (8, 7):
            self.checkFloatParam(widget, 'from', 200, expected=100)
        else:
            self.checkInvalidParam(
                    widget, 'from', 200,
                    errmsg='-to value must be greater than -from value')

    def test_configure_increment(self):
        widget = self.create()
        self.checkFloatParam(widget, 'increment', -1, 1, 10.2, 12.8, 0)

    def test_configure_to(self):
        widget = self.create()
        self.checkParam(widget, 'from', -100.0)
        self.checkFloatParam(widget, 'to', -10, 10.2, 11.7)
        if tk_version >= (8, 7):
            self.checkFloatParam(widget, 'to', -200, expected=-100)
        else:
            self.checkInvalidParam(
                    widget, 'to', -200,
                    errmsg='-to value must be greater than -from value')

    def test_configure_values(self):
        # XXX
        widget = self.create()
        self.assertEqual(widget['values'], '')
        if tk_version < (9, 0):
            expected = 'mon tue wed thur'
        else:
            expected = ('mon', 'tue', 'wed', 'thur')
        self.checkParam(widget, 'values', 'mon tue wed thur',
                        expected=expected)
        self.checkParam(widget, 'values', ('mon', 'tue', 'wed', 'thur'),
                        expected=expected)

        if tk_version < (9, 0):
            expected = '42 3.14 {} {any string}'
        else:
            expected = (42, 3.14, '', 'any string')
        self.checkParam(widget, 'values', (42, 3.14, '', 'any string'),
                        expected=expected)

        self.checkParam(widget, 'values', '')

    def test_configure_wrap(self):
        widget = self.create()
        self.checkBooleanParam(widget, 'wrap')

    def test_bbox(self):
        widget = self.create()
        self.assertIsBoundingBox(widget.bbox(0))
        self.assertRaises(tkinter.TclError, widget.bbox, 'noindex')
        self.assertRaises(tkinter.TclError, widget.bbox, None)
        self.assertRaises(TypeError, widget.bbox)
        self.assertRaises(TypeError, widget.bbox, 0, 1)

    def test_selection_methods(self):
        widget = self.create()
        widget.insert(0, '12345')
        self.assertFalse(widget.selection_present())
        widget.selection_range(0, 'end')
        self.assertEqual(widget.selection_get(), '12345')
        self.assertTrue(widget.selection_present())
        widget.selection_from(1)
        widget.selection_to(2)
        self.assertEqual(widget.selection_get(), '2')
        widget.selection_range(3, 4)
        self.assertEqual(widget.selection_get(), '4')
        widget.selection_clear()
        self.assertFalse(widget.selection_present())
        widget.selection_range(0, 'end')
        widget.selection_adjust(4)
        self.assertEqual(widget.selection_get(), '1234')
        widget.selection_adjust(1)
        self.assertEqual(widget.selection_get(), '234')
        widget.selection_adjust(5)
        self.assertEqual(widget.selection_get(), '2345')
        widget.selection_adjust(0)
        self.assertEqual(widget.selection_get(), '12345')

    def test_selection_element(self):
        widget = self.create()
        self.assertEqual(widget.selection_element(), "none")
        widget.selection_element("buttonup")
        self.assertEqual(widget.selection_element(), "buttonup")
        widget.selection_element("buttondown")
        self.assertEqual(widget.selection_element(), "buttondown")


@add_configure_tests(StandardOptionsTests)
class TextTest(AbstractWidgetTest, unittest.TestCase):
    OPTIONS = (
        'autoseparators', 'background', 'blockcursor', 'borderwidth',
        'cursor', 'endline', 'exportselection',
        'font', 'foreground', 'height',
        'highlightbackground', 'highlightcolor', 'highlightthickness',
        'inactiveselectbackground', 'insertbackground', 'insertborderwidth',
        'insertofftime', 'insertontime', 'insertunfocussed', 'insertwidth',
        'maxundo', 'padx', 'pady', 'relief',
        'selectbackground', 'selectborderwidth', 'selectforeground',
        'setgrid', 'spacing1', 'spacing2', 'spacing3', 'startline', 'state',
        'tabs', 'tabstyle', 'takefocus', 'undo', 'width', 'wrap',
        'xscrollcommand', 'yscrollcommand',
    )
    _rounds_pixels = (tk_version < (9, 0))
    _no_round = {'selectborderwidth'}
    _clipped = {'highlightthickness'}

    def create(self, **kwargs):
        return tkinter.Text(self.root, **kwargs)

    def test_configure_autoseparators(self):
        widget = self.create()
        self.checkBooleanParam(widget, 'autoseparators')

    def test_configure_blockcursor(self):
        widget = self.create()
        self.checkBooleanParam(widget, 'blockcursor')

    def test_configure_endline(self):
        widget = self.create()
        text = '\n'.join('Line %d' for i in range(100))
        widget.insert('end', text)
        self.checkParam(widget, 'endline', 200, expected='')
        self.checkParam(widget, 'endline', -10, expected='')
        self.checkInvalidParam(widget, 'endline', 'spam',
                errmsg='expected integer but got "spam"')
        self.checkParam(widget, 'endline', 50)
        self.checkParam(widget, 'startline', 15)
        self.checkInvalidParam(widget, 'endline', 10,
                errmsg='-startline must be less than or equal to -endline')

    def test_configure_height(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'height', 100, 101.2, 102.6, '3c')
        expected = 1 if tk_version < (9, 0) else 0
        self.checkParam(widget, 'height', -100, expected=expected)
        self.checkParam(widget, 'height', 0, expected=expected)

    def test_configure_maxundo(self):
        widget = self.create()
        self.checkIntegerParam(widget, 'maxundo', 0, 5, -1)

    def test_configure_inactiveselectbackground(self):
        widget = self.create()
        self.checkColorParam(widget, 'inactiveselectbackground')

    @requires_tk(8, 6)
    def test_configure_insertunfocussed(self):
        widget = self.create()
        self.checkEnumParam(widget, 'insertunfocussed',
                            'hollow', 'none', 'solid')

    def test_configure_selectborderwidth(self):
        widget = self.create()
        value = -2 if tk_version < (9, 0) else 0
        self.checkPixelsParam(widget, 'selectborderwidth',
                              1.3, 2.6, value, '10p', conv=False)

    def test_configure_spacing1(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'spacing1', 20, 21.4, 22.6, '0.5c')
        self.checkParam(widget, 'spacing1', -5, expected=0)

    def test_configure_spacing2(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'spacing2', 5, 6.4, 7.6, '0.1c')
        self.checkParam(widget, 'spacing2', -1, expected=0)

    def test_configure_spacing3(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'spacing3', 20, 21.4, 22.6, '0.5c')
        self.checkParam(widget, 'spacing3', -10, expected=0)

    def test_configure_startline(self):
        widget = self.create()
        text = '\n'.join('Line %d' for i in range(100))
        widget.insert('end', text)
        self.checkParam(widget, 'startline', 200, expected='')
        self.checkParam(widget, 'startline', -10, expected='')
        self.checkInvalidParam(widget, 'startline', 'spam',
                errmsg='expected integer but got "spam"')
        self.checkParam(widget, 'startline', 10)
        self.checkParam(widget, 'endline', 50)
        self.checkInvalidParam(widget, 'startline', 70,
                errmsg='-startline must be less than or equal to -endline')

    def test_configure_state(self):
        widget = self.create()
        self.checkEnumParam(widget, 'state', 'disabled', 'normal')

    def test_configure_tabs(self):
        widget = self.create()
        self.checkParam(widget, 'tabs', (10.2, 20.7, '1i', '2i'))
        self.checkParam(widget, 'tabs', '10.2 20.7 1i 2i',
                        expected=(10.2, 20.7, '1i', '2i')
                                 if get_tk_patchlevel(self.root) >= (8, 6, 14)
                                 else ('10.2', '20.7', '1i', '2i'))
        self.checkParam(widget, 'tabs', '2c left 4c 6c center',
                        expected=('2c', 'left', '4c', '6c', 'center'))
        self.checkInvalidParam(widget, 'tabs', 'spam',
                errmsg=EXPECTED_SCREEN_DISTANCE_ERRMSG.format('spam'))

    def test_configure_tabstyle(self):
        widget = self.create()
        self.checkEnumParam(widget, 'tabstyle', 'tabular', 'wordprocessor')

    def test_configure_undo(self):
        widget = self.create()
        self.checkBooleanParam(widget, 'undo')

    def test_configure_width(self):
        widget = self.create()
        self.checkIntegerParam(widget, 'width', 402)
        self.checkParam(widget, 'width', -402, expected=1)
        self.checkParam(widget, 'width', 0, expected=1)

    def test_configure_wrap(self):
        widget = self.create()
        self.checkEnumParam(widget, 'wrap', 'char', 'none', 'word')

    def test_bbox(self):
        widget = self.create()
        self.assertIsBoundingBox(widget.bbox('1.1'))
        self.assertIsNone(widget.bbox('end'))
        self.assertRaises(tkinter.TclError, widget.bbox, 'noindex')
        self.assertRaises(tkinter.TclError, widget.bbox, None)
        self.assertRaises(TypeError, widget.bbox)
        self.assertRaises(TypeError, widget.bbox, '1.1', 'end')


@add_configure_tests(PixelSizeTests, StandardOptionsTests)
class CanvasTest(AbstractWidgetTest, unittest.TestCase):
    OPTIONS = (
        'background', 'borderwidth',
        'closeenough', 'confine', 'cursor', 'height',
        'highlightbackground', 'highlightcolor', 'highlightthickness',
        'insertbackground', 'insertborderwidth',
        'insertofftime', 'insertontime', 'insertwidth',
        'offset', 'relief', 'scrollregion',
        'selectbackground', 'selectborderwidth', 'selectforeground',
        'state', 'takefocus',
        'xscrollcommand', 'xscrollincrement',
        'yscrollcommand', 'yscrollincrement', 'width',
    )
    _rounds_pixels = True
    if tk_version < (9, 0):
        _noround = {}
        _clipped = {'highlightthickness'}
    else:
        _no_round = {'borderwidth', 'height', 'highlightthickness', 'width',
                     'xscrollincrement', 'yscrollincrement'}
        _clipped = {'borderwidth', 'height', 'highlightthickness', 'width',
                    'xscrollincrement', 'yscrollincrement'}
    _stringify = True

    def create(self, **kwargs):
        return tkinter.Canvas(self.root, **kwargs)

    def test_configure_closeenough(self):
        widget = self.create()
        self.checkFloatParam(widget, 'closeenough', 24, 2.4, 3.6, -3,
                             conv=float)

    def test_configure_confine(self):
        widget = self.create()
        self.checkBooleanParam(widget, 'confine')

    def test_configure_offset(self):
        widget = self.create()
        self.assertEqual(widget['offset'], '0,0')
        self.checkParams(widget, 'offset',
                'n', 'ne', 'e', 'se', 's', 'sw', 'w', 'nw', 'center')
        self.checkParam(widget, 'offset', '10,20')
        self.checkParam(widget, 'offset', '#5,6')
        self.checkInvalidParam(widget, 'offset', 'spam')

    def test_configure_scrollregion(self):
        widget = self.create()
        self.checkParam(widget, 'scrollregion', '0 0 200 150')
        self.checkParam(widget, 'scrollregion', (0, 0, 200, 150),
                        expected='0 0 200 150')
        self.checkParam(widget, 'scrollregion', '')
        self.checkInvalidParam(widget, 'scrollregion', 'spam',
                               errmsg='bad scrollRegion "spam"')
        self.checkInvalidParam(widget, 'scrollregion', (0, 0, 200, 'spam'))
        self.checkInvalidParam(widget, 'scrollregion', (0, 0, 200))
        self.checkInvalidParam(widget, 'scrollregion', (0, 0, 200, 150, 0))

    def test_configure_state(self):
        widget = self.create()
        self.checkEnumParam(widget, 'state', 'disabled', 'normal',
                errmsg='bad state value "{}": must be normal or disabled')

    def test_configure_xscrollincrement(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'xscrollincrement',
                              40, 0, 41.2, 43.6, -40, '0.5i')

    def test_configure_yscrollincrement(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'yscrollincrement',
                              10, 0, 11.2, 13.6, -10, '0.1i')

    def _test_option_joinstyle(self, c, factory):
        for joinstyle in 'bevel', 'miter', 'round':
            i = factory(joinstyle=joinstyle)
            self.assertEqual(c.itemcget(i, 'joinstyle'), joinstyle)
        self.assertRaises(TclError, factory, joinstyle='spam')

    def _test_option_smooth(self, c, factory):
        for smooth in 1, True, '1', 'true', 'yes', 'on':
            i = factory(smooth=smooth)
            self.assertEqual(c.itemcget(i, 'smooth'), 'true')
        for smooth in 0, False, '0', 'false', 'no', 'off':
            i = factory(smooth=smooth)
            self.assertEqual(c.itemcget(i, 'smooth'), '0')
        i = factory(smooth=True, splinestep=30)
        self.assertEqual(c.itemcget(i, 'smooth'), 'true')
        self.assertEqual(c.itemcget(i, 'splinestep'), '30')
        i = factory(smooth='raw', splinestep=30)
        self.assertEqual(c.itemcget(i, 'smooth'), 'raw')
        self.assertEqual(c.itemcget(i, 'splinestep'), '30')
        self.assertRaises(TclError, factory, smooth='spam')

    def test_create_rectangle(self):
        c = self.create()
        i1 = c.create_rectangle(20, 30, 60, 10)
        self.assertEqual(c.coords(i1), [20.0, 10.0, 60.0, 30.0])
        self.assertEqual(c.bbox(i1), (19, 9, 61, 31))

        i2 = c.create_rectangle([21, 31, 61, 11])
        self.assertEqual(c.coords(i2), [21.0, 11.0, 61.0, 31.0])
        self.assertEqual(c.bbox(i2), (20, 10, 62, 32))

        i3 = c.create_rectangle((22, 32), (62, 12))
        self.assertEqual(c.coords(i3), [22.0, 12.0, 62.0, 32.0])
        self.assertEqual(c.bbox(i3), (21, 11, 63, 33))

        i4 = c.create_rectangle([(23, 33), (63, 13)])
        self.assertEqual(c.coords(i4), [23.0, 13.0, 63.0, 33.0])
        self.assertEqual(c.bbox(i4), (22, 12, 64, 34))

        self.assertRaises(TclError, c.create_rectangle, 20, 30, 60)
        self.assertRaises(TclError, c.create_rectangle, [20, 30, 60])
        self.assertRaises(TclError, c.create_rectangle, 20, 30, 40, 50, 60, 10)
        self.assertRaises(TclError, c.create_rectangle, [20, 30, 40, 50, 60, 10])
        self.assertRaises(TclError, c.create_rectangle, 20, 30)
        self.assertRaises(TclError, c.create_rectangle, [20, 30])
        self.assertRaises(IndexError, c.create_rectangle)
        self.assertRaises(IndexError, c.create_rectangle, [])

    def test_create_line(self):
        c = self.create()
        i1 = c.create_line(20, 30, 40, 50, 60, 10)
        self.assertEqual(c.coords(i1), [20.0, 30.0, 40.0, 50.0, 60.0, 10.0])
        self.assertEqual(c.bbox(i1), (18, 8, 62, 52))
        self.assertEqual(c.itemcget(i1, 'arrow'), 'none')
        self.assertEqual(c.itemcget(i1, 'arrowshape'), '8 10 3')
        self.assertEqual(c.itemcget(i1, 'capstyle'), 'butt')
        self.assertEqual(c.itemcget(i1, 'joinstyle'), 'round')
        self.assertEqual(c.itemcget(i1, 'smooth'), '0')
        self.assertEqual(c.itemcget(i1, 'splinestep'), '12')

        i2 = c.create_line([21, 31, 41, 51, 61, 11])
        self.assertEqual(c.coords(i2), [21.0, 31.0, 41.0, 51.0, 61.0, 11.0])
        self.assertEqual(c.bbox(i2), (19, 9, 63, 53))

        i3 = c.create_line((22, 32), (42, 52), (62, 12))
        self.assertEqual(c.coords(i3), [22.0, 32.0, 42.0, 52.0, 62.0, 12.0])
        self.assertEqual(c.bbox(i3), (20, 10, 64, 54))

        i4 = c.create_line([(23, 33), (43, 53), (63, 13)])
        self.assertEqual(c.coords(i4), [23.0, 33.0, 43.0, 53.0, 63.0, 13.0])
        self.assertEqual(c.bbox(i4), (21, 11, 65, 55))

        self.assertRaises(TclError, c.create_line, 20, 30, 60)
        self.assertRaises(TclError, c.create_line, [20, 30, 60])
        self.assertRaises(TclError, c.create_line, 20, 30)
        self.assertRaises(TclError, c.create_line, [20, 30])
        self.assertRaises(IndexError, c.create_line)
        self.assertRaises(IndexError, c.create_line, [])

        for arrow in 'none', 'first', 'last', 'both':
            i = c.create_line(20, 30, 60, 10, arrow=arrow)
            self.assertEqual(c.itemcget(i, 'arrow'), arrow)
        i = c.create_line(20, 30, 60, 10, arrow='first', arrowshape=[10, 15, 5])
        self.assertEqual(c.itemcget(i, 'arrowshape'), '10 15 5')
        self.assertRaises(TclError, c.create_line, 20, 30, 60, 10, arrow='spam')

        for capstyle in 'butt', 'projecting', 'round':
            i = c.create_line(20, 30, 60, 10, capstyle=capstyle)
            self.assertEqual(c.itemcget(i, 'capstyle'), capstyle)
        self.assertRaises(TclError, c.create_line, 20, 30, 60, 10, capstyle='spam')

        self._test_option_joinstyle(c,
                lambda **kwargs: c.create_line(20, 30, 40, 50, 60, 10, **kwargs))
        self._test_option_smooth(c,
                lambda **kwargs: c.create_line(20, 30, 60, 10, **kwargs))

    def test_create_polygon(self):
        c = self.create()
        tk87 = tk_version >= (8, 7)
        # In Tk < 8.7 polygons are filled, but has no outline by default.
        # This affects its size, so always explicitly specify outline.
        i1 = c.create_polygon(20, 30, 40, 50, 60, 10, outline='red')
        self.assertEqual(c.coords(i1), [20.0, 30.0, 40.0, 50.0, 60.0, 10.0])
        self.assertEqual(c.bbox(i1), (18, 8, 62, 52))
        self.assertEqual(c.itemcget(i1, 'joinstyle'), 'round')
        self.assertEqual(c.itemcget(i1, 'smooth'), '0')
        self.assertEqual(c.itemcget(i1, 'splinestep'), '12')

        i2 = c.create_polygon([21, 31, 41, 51, 61, 11], outline='red')
        self.assertEqual(c.coords(i2), [21.0, 31.0, 41.0, 51.0, 61.0, 11.0])
        self.assertEqual(c.bbox(i2), (19, 9, 63, 53))

        i3 = c.create_polygon((22, 32), (42, 52), (62, 12), outline='red')
        self.assertEqual(c.coords(i3), [22.0, 32.0, 42.0, 52.0, 62.0, 12.0])
        self.assertEqual(c.bbox(i3), (20, 10, 64, 54))

        i4 = c.create_polygon([(23, 33), (43, 53), (63, 13)], outline='red')
        self.assertEqual(c.coords(i4), [23.0, 33.0, 43.0, 53.0, 63.0, 13.0])
        self.assertEqual(c.bbox(i4), (21, 11, 65, 55))

        self.assertRaises(TclError, c.create_polygon, 20, 30, 60)
        self.assertRaises(TclError, c.create_polygon, [20, 30, 60])
        self.assertRaises(IndexError, c.create_polygon)
        self.assertRaises(IndexError, c.create_polygon, [])

        self._test_option_joinstyle(c,
                lambda **kwargs: c.create_polygon(20, 30, 40, 50, 60, 10, **kwargs))
        self._test_option_smooth(c,
                lambda **kwargs: c.create_polygon(20, 30, 40, 50, 60, 10, **kwargs))

    def test_coords(self):
        c = self.create()
        i = c.create_line(20, 30, 40, 50, 60, 10, tags='x')
        self.assertEqual(c.coords(i), [20.0, 30.0, 40.0, 50.0, 60.0, 10.0])
        self.assertEqual(c.coords('x'), [20.0, 30.0, 40.0, 50.0, 60.0, 10.0])
        self.assertEqual(c.bbox(i), (18, 8, 62, 52))

        c.coords(i, 50, 60, 70, 80, 90, 40)
        self.assertEqual(c.coords(i), [50.0, 60.0, 70.0, 80.0, 90.0, 40.0])
        self.assertEqual(c.bbox(i), (48, 38, 92, 82))

        c.coords(i, [21, 31, 41, 51, 61, 11])
        self.assertEqual(c.coords(i), [21.0, 31.0, 41.0, 51.0, 61.0, 11.0])

        c.coords(i, (22, 32), (42, 52), (62, 12))
        self.assertEqual(c.coords(i), [22.0, 32.0, 42.0, 52.0, 62.0, 12.0])

        c.coords(i, [(23, 33), (43, 53), (63, 13)])
        self.assertEqual(c.coords(i), [23.0, 33.0, 43.0, 53.0, 63.0, 13.0])

        c.coords(i, 20, 30, 60, 10)
        self.assertEqual(c.coords(i), [20.0, 30.0, 60.0, 10.0])
        self.assertEqual(c.bbox(i), (18, 8, 62, 32))

        self.assertRaises(TclError, c.coords, i, 20, 30, 60)
        self.assertRaises(TclError, c.coords, i, [20, 30, 60])
        self.assertRaises(TclError, c.coords, i, 20, 30)
        self.assertRaises(TclError, c.coords, i, [20, 30])

        c.coords(i, '20', '30c', '60i', '10p')
        coords = c.coords(i)
        self.assertIsInstance(coords, list)
        self.assertEqual(len(coords), 4)
        self.assertEqual(coords[0], 20)
        for i in range(4):
            self.assertIsInstance(coords[i], float)

    @requires_tk(8, 6)
    def test_moveto(self):
        widget = self.create()
        i1 = widget.create_rectangle(1, 1, 20, 20, tags='group')
        i2 = widget.create_rectangle(30, 30, 50, 70, tags='group')
        x1, y1, _, _ = widget.bbox(i1)
        x2, y2, _, _ = widget.bbox(i2)
        widget.moveto('group', 200, 100)
        x1_2, y1_2, _, _ = widget.bbox(i1)
        x2_2, y2_2, _, _ = widget.bbox(i2)
        self.assertEqual(x1_2, 200)
        self.assertEqual(y1_2, 100)
        self.assertEqual(x2 - x1, x2_2 - x1_2)
        self.assertEqual(y2 - y1, y2_2 - y1_2)
        widget.tag_lower(i2, i1)
        widget.moveto('group', y=50)
        x1_3, y1_3, _, _ = widget.bbox(i1)
        x2_3, y2_3, _, _ = widget.bbox(i2)
        self.assertEqual(y2_3, 50)
        self.assertEqual(x2_3, x2_2)
        self.assertEqual(x2_2 - x1_2, x2_3 - x1_3)
        self.assertEqual(y2_2 - y1_2, y2_3 - y1_3)


@add_configure_tests(IntegerSizeTests, StandardOptionsTests)
class ListboxTest(AbstractWidgetTest, unittest.TestCase):
    OPTIONS = (
        'activestyle', 'background', 'borderwidth', 'cursor',
        'disabledforeground', 'exportselection',
        'font', 'foreground', 'height',
        'highlightbackground', 'highlightcolor', 'highlightthickness',
        'justify', 'listvariable', 'relief',
        'selectbackground', 'selectborderwidth', 'selectforeground',
        'selectmode', 'setgrid', 'state',
        'takefocus', 'width', 'xscrollcommand', 'yscrollcommand',
    )
    _rounds_pixels = (tk_version < (9, 0))
    if tk_version < (9, 0):
        _clipped = {'highlightthickness'}
    else:
        _clipped = { 'borderwidth', 'highlightthickness', 'selectborderwidth'}

    def create(self, **kwargs):
        return tkinter.Listbox(self.root, **kwargs)

    def test_configure_activestyle(self):
        widget = self.create()
        self.checkEnumParam(widget, 'activestyle',
                            'dotbox', 'none', 'underline')

    test_configure_justify = requires_tk(8, 6, 5)(StandardOptionsTests.test_configure_justify)

    def test_configure_listvariable(self):
        widget = self.create()
        var = tkinter.DoubleVar(self.root)
        self.checkVariableParam(widget, 'listvariable', var)

    def test_configure_selectmode(self):
        widget = self.create()
        self.checkParam(widget, 'selectmode', 'single')
        self.checkParam(widget, 'selectmode', 'browse')
        self.checkParam(widget, 'selectmode', 'multiple')
        self.checkParam(widget, 'selectmode', 'extended')

    def test_configure_state(self):
        widget = self.create()
        self.checkEnumParam(widget, 'state', 'disabled', 'normal')

    def test_itemconfigure(self):
        widget = self.create()
        with self.assertRaisesRegex(TclError, 'item number "0" out of range'):
            widget.itemconfigure(0)
        colors = 'red orange yellow green blue white violet'.split()
        widget.insert('end', *colors)
        for i, color in enumerate(colors):
            widget.itemconfigure(i, background=color)
        with self.assertRaises(TypeError):
            widget.itemconfigure()
        with self.assertRaisesRegex(TclError, 'bad listbox index "red"'):
            widget.itemconfigure('red')
        if get_tk_patchlevel(self.root) >= (8, 6, 14):
            prefix = ('background', '', '', '')
        else:
            prefix = ('background', 'background', 'Background', '')
        self.assertEqual(widget.itemconfigure(0, 'background'),
                         (*prefix, 'red'))
        self.assertEqual(widget.itemconfigure('end', 'background'),
                         (*prefix, 'violet'))
        self.assertEqual(widget.itemconfigure('@0,0', 'background'),
                         (*prefix, 'red'))

        d = widget.itemconfigure(0)
        self.assertIsInstance(d, dict)
        for k, v in d.items():
            self.assertIn(len(v), (2, 5))
            if len(v) == 5:
                self.assertEqual(v, widget.itemconfigure(0, k))
                self.assertEqual(v[4], widget.itemcget(0, k))

    def check_itemconfigure(self, name, value):
        widget = self.create()
        widget.insert('end', 'a', 'b', 'c', 'd')
        widget.itemconfigure(0, **{name: value})
        self.assertEqual(widget.itemconfigure(0, name)[4], value)
        self.assertEqual(widget.itemcget(0, name), value)
        with self.assertRaisesRegex(TclError, 'unknown color name "spam"'):
            widget.itemconfigure(0, **{name: 'spam'})

    def test_itemconfigure_background(self):
        self.check_itemconfigure('background', '#ff0000')

    def test_itemconfigure_bg(self):
        self.check_itemconfigure('bg', '#ff0000')

    def test_itemconfigure_fg(self):
        self.check_itemconfigure('fg', '#110022')

    def test_itemconfigure_foreground(self):
        self.check_itemconfigure('foreground', '#110022')

    def test_itemconfigure_selectbackground(self):
        self.check_itemconfigure('selectbackground', '#110022')

    def test_itemconfigure_selectforeground(self):
        self.check_itemconfigure('selectforeground', '#654321')

    def test_box(self):
        lb = self.create()
        lb.insert(0, *('el%d' % i for i in range(8)))
        lb.pack()
        self.assertIsBoundingBox(lb.bbox(0))
        self.assertIsNone(lb.bbox(-1))
        self.assertIsNone(lb.bbox(10))
        self.assertRaises(TclError, lb.bbox, 'noindex')
        self.assertRaises(TclError, lb.bbox, None)
        self.assertRaises(TypeError, lb.bbox)
        self.assertRaises(TypeError, lb.bbox, 0, 1)

    def test_curselection(self):
        lb = self.create()
        lb.insert(0, *('el%d' % i for i in range(8)))
        lb.selection_clear(0, tkinter.END)
        lb.selection_set(2, 4)
        lb.selection_set(6)
        self.assertEqual(lb.curselection(), (2, 3, 4, 6))
        self.assertRaises(TypeError, lb.curselection, 0)

    def test_get(self):
        lb = self.create()
        lb.insert(0, *('el%d' % i for i in range(8)))
        self.assertEqual(lb.get(0), 'el0')
        self.assertEqual(lb.get(3), 'el3')
        self.assertEqual(lb.get('end'), 'el7')
        self.assertEqual(lb.get(8), '')
        self.assertEqual(lb.get(-1), '')
        self.assertEqual(lb.get(3, 5), ('el3', 'el4', 'el5'))
        self.assertEqual(lb.get(5, 'end'), ('el5', 'el6', 'el7'))
        self.assertEqual(lb.get(5, 0), ())
        self.assertEqual(lb.get(0, 0), ('el0',))
        self.assertRaises(TclError, lb.get, 'noindex')
        self.assertRaises(TclError, lb.get, None)
        self.assertRaises(TypeError, lb.get)
        self.assertRaises(TclError, lb.get, 'end', 'noindex')
        self.assertRaises(TypeError, lb.get, 1, 2, 3)
        self.assertRaises(TclError, lb.get, 2.4)


@add_configure_tests(PixelSizeTests, StandardOptionsTests)
class ScaleTest(AbstractWidgetTest, unittest.TestCase):
    OPTIONS = (
        'activebackground', 'background', 'bigincrement', 'borderwidth',
        'command', 'cursor', 'digits', 'font', 'foreground', 'from',
        'highlightbackground', 'highlightcolor', 'highlightthickness',
        'label', 'length', 'orient', 'relief',
        'repeatdelay', 'repeatinterval',
        'resolution', 'showvalue', 'sliderlength', 'sliderrelief', 'state',
        'takefocus', 'tickinterval', 'to', 'troughcolor', 'variable', 'width',
    )
    _rounds_pixels = (tk_version < (9, 0))
    _clipped = {'highlightthickness'}
    default_orient = 'vertical'

    def create(self, **kwargs):
        return tkinter.Scale(self.root, **kwargs)

    def test_configure_bigincrement(self):
        widget = self.create()
        self.checkFloatParam(widget, 'bigincrement', 12.4, 23.6, -5)

    def test_configure_digits(self):
        widget = self.create()
        self.checkIntegerParam(widget, 'digits', 5, 0)

    def test_configure_from(self):
        widget = self.create()
        conv = float if get_tk_patchlevel(self.root) >= (8, 6, 10) else float_round
        self.checkFloatParam(widget, 'from', 100, 14.9, 15.1, conv=conv)

    def test_configure_label(self):
        widget = self.create()
        self.checkParam(widget, 'label', 'any string')
        self.checkParam(widget, 'label', '')

    def test_configure_length(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'length', 130, 131.2, 135.6, '5i')

    def test_configure_resolution(self):
        widget = self.create()
        self.checkFloatParam(widget, 'resolution', 4.2, 0, 6.7, -2)

    def test_configure_showvalue(self):
        widget = self.create()
        self.checkBooleanParam(widget, 'showvalue')

    def test_configure_sliderlength(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'sliderlength',
                              10, 11.2, 15.6, -3, '3m')

    def test_configure_sliderrelief(self):
        widget = self.create()
        self.checkReliefParam(widget, 'sliderrelief')

    def test_configure_tickinterval(self):
        widget = self.create()
        self.checkFloatParam(widget, 'tickinterval', 1, 4.3, 7.6, 0,
                             conv=float_round)
        self.checkParam(widget, 'tickinterval', -2, expected=2,
                        conv=float_round)

    def test_configure_to(self):
        widget = self.create()
        self.checkFloatParam(widget, 'to', 300, 14.9, 15.1, -10,
                             conv=float_round)


@add_configure_tests(PixelSizeTests, StandardOptionsTests)
class ScrollbarTest(AbstractWidgetTest, unittest.TestCase):
    OPTIONS = (
        'activebackground', 'activerelief',
        'background', 'borderwidth',
        'command', 'cursor', 'elementborderwidth',
        'highlightbackground', 'highlightcolor', 'highlightthickness',
        'jump', 'orient', 'relief',
        'repeatdelay', 'repeatinterval',
        'takefocus', 'troughcolor', 'width',
    )
    _rounds_pixels = True
    if tk_version >= (9, 0):
        _no_round = {'borderwidth', 'elementborderwidth', 'highlightthickness',
                     'width'}
    if tk_version < (9, 0):
        _clipped = {'highlightthickness'}
    else:
        _clipped = {'borderwidth', 'highlightthickness', 'width'}
    _stringify = True
    default_orient = 'vertical'

    def create(self, **kwargs):
        return tkinter.Scrollbar(self.root, **kwargs)

    def test_configure_elementborderwidth(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'elementborderwidth', 4.3, 5.6, '1m')
        expected = self._default_pixels if tk_version >= (8, 7) else -2
        self.checkParam(widget, 'elementborderwidth', -2, expected=expected)

    def test_configure_orient(self):
        widget = self.create()
        self.checkEnumParam(widget, 'orient', 'vertical', 'horizontal',
                            fullname='orientation', allow_empty=True)

    def test_activate(self):
        sb = self.create()
        for e in ('arrow1', 'slider', 'arrow2'):
            sb.activate(e)
            self.assertEqual(sb.activate(), e)
        sb.activate('')
        self.assertIsNone(sb.activate())
        self.assertRaises(TypeError, sb.activate, 'arrow1', 'arrow2')

    def test_set(self):
        sb = self.create()
        sb.set(0.2, 0.4)
        self.assertEqual(sb.get(), (0.2, 0.4))
        self.assertRaises(TclError, sb.set, 'abc', 'def')
        self.assertRaises(TclError, sb.set, 0.6, 'def')
        self.assertRaises(TclError, sb.set, 0.6, None)
        self.assertRaises(TypeError, sb.set, 0.6)
        self.assertRaises(TypeError, sb.set, 0.6, 0.7, 0.8)


@add_configure_tests(StandardOptionsTests)
class PanedWindowTest(AbstractWidgetTest, unittest.TestCase):
    OPTIONS = (
        'background', 'borderwidth', 'cursor',
        'handlepad', 'handlesize', 'height',
        'opaqueresize', 'orient',
        'proxybackground', 'proxyborderwidth', 'proxyrelief',
        'relief',
        'sashcursor', 'sashpad', 'sashrelief', 'sashwidth',
        'showhandle', 'width',
    )
    _rounds_pixels = True
    if tk_version < (9, 0):
        _no_round = {'handlesize', 'height', 'proxyborderwidth', 'sashwidth',
                     'selectborderwidth', 'width'}
    else:
        _no_round = {'borderwidth', 'handlepad', 'handlesize', 'height',
                     'proxyborderwidth', 'sashpad', 'sashwidth',
                     'selectborderwidth', 'width'}
    _clipped = {}
    default_orient = 'horizontal'

    def create(self, **kwargs):
        return tkinter.PanedWindow(self.root, **kwargs)

    def test_configure_handlepad(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'handlepad', 5, 6.4, 7.6, -3, '1m')

    def test_configure_handlesize(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'handlesize', 8, 9.4, 10.6, -3, '2m',
                              conv=False)

    def test_configure_height(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'height', 100, 101.2, 102.6, -100, 0, '1i',
                              conv=False)

    def test_configure_opaqueresize(self):
        widget = self.create()
        self.checkBooleanParam(widget, 'opaqueresize')

    @requires_tk(8, 6, 5)
    def test_configure_proxybackground(self):
        widget = self.create()
        self.checkColorParam(widget, 'proxybackground')

    @requires_tk(8, 6, 5)
    def test_configure_proxyborderwidth(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'proxyborderwidth',
                              0, 1.3, 2.9, 6, -2, '10p',
                              conv=False)

    @requires_tk(8, 6, 5)
    def test_configure_proxyrelief(self):
        widget = self.create()
        self.checkReliefParam(widget, 'proxyrelief',
                              allow_empty=(tk_version >= (8, 7)))

    def test_configure_sashcursor(self):
        widget = self.create()
        self.checkCursorParam(widget, 'sashcursor')

    def test_configure_sashpad(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'sashpad', 8, 1.3, 2.6, -2, '2m')

    def test_configure_sashrelief(self):
        widget = self.create()
        self.checkReliefParam(widget, 'sashrelief')

    def test_configure_sashwidth(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'sashwidth', 10, 11.1, 15.6, -3, '1m',
                              conv=False)

    def test_configure_showhandle(self):
        widget = self.create()
        self.checkBooleanParam(widget, 'showhandle')

    def test_configure_width(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'width', 402, 403.4, 404.6, -402, 0, '5i',
                              conv=False)

    def create2(self):
        p = self.create()
        b = tkinter.Button(p)
        c = tkinter.Button(p)
        p.add(b)
        p.add(c)
        return p, b, c

    def test_paneconfigure(self):
        p, b, c = self.create2()
        self.assertRaises(TypeError, p.paneconfigure)
        d = p.paneconfigure(b)
        self.assertIsInstance(d, dict)
        for k, v in d.items():
            self.assertEqual(len(v), 5)
            self.assertEqual(v, p.paneconfigure(b, k))
            self.assertEqual(v[4], p.panecget(b, k))

    def check_paneconfigure(self, p, b, name, value, expected):
        if not self.wantobjects:
            expected = str(expected)
        p.paneconfigure(b, **{name: value})
        self.assertEqual(p.paneconfigure(b, name)[4], expected)
        self.assertEqual(p.panecget(b, name), expected)

    def check_paneconfigure_bad(self, p, b, name, msg):
        with self.assertRaisesRegex(TclError, msg):
            p.paneconfigure(b, **{name: 'badValue'})

    def test_paneconfigure_after(self):
        p, b, c = self.create2()
        self.check_paneconfigure(p, b, 'after', c, str(c))
        self.check_paneconfigure_bad(p, b, 'after',
                                     'bad window path name "badValue"')

    def test_paneconfigure_before(self):
        p, b, c = self.create2()
        self.check_paneconfigure(p, b, 'before', c, str(c))
        self.check_paneconfigure_bad(p, b, 'before',
                                     'bad window path name "badValue"')

    def test_paneconfigure_height(self):
        p, b, c = self.create2()
        self.check_paneconfigure(p, b, 'height', 10, 10)
        self.check_paneconfigure_bad(p, b, 'height',
                EXPECTED_SCREEN_DISTANCE_OR_EMPTY_ERRMSG.format('badValue'))

    def test_paneconfigure_hide(self):
        p, b, c = self.create2()
        self.check_paneconfigure(p, b, 'hide', False, 0)
        self.check_paneconfigure_bad(p, b, 'hide',
                                     'expected boolean value but got "badValue"')

    def test_paneconfigure_minsize(self):
        p, b, c = self.create2()
        self.check_paneconfigure(p, b, 'minsize', 10, 10)
        self.check_paneconfigure_bad(p, b, 'minsize',
                EXPECTED_SCREEN_DISTANCE_ERRMSG.format('badValue'))

    def test_paneconfigure_padx(self):
        p, b, c = self.create2()
        self.check_paneconfigure(p, b, 'padx', 1.3, 1 if tk_version < (9, 0) else 1.3)
        self.check_paneconfigure_bad(p, b, 'padx',
                EXPECTED_SCREEN_DISTANCE_ERRMSG.format('badValue'))

    def test_paneconfigure_pady(self):
        p, b, c = self.create2()
        self.check_paneconfigure(p, b, 'pady', 1.3, 1 if tk_version < (9, 0) else 1.3)
        self.check_paneconfigure_bad(p, b, 'pady',
                EXPECTED_SCREEN_DISTANCE_ERRMSG.format('badValue'))

    def test_paneconfigure_sticky(self):
        p, b, c = self.create2()
        self.check_paneconfigure(p, b, 'sticky', 'nsew', 'nesw')
        self.check_paneconfigure_bad(p, b, 'sticky',
                                     'bad stickyness value "badValue": must '
                                     'be a string containing zero or more of '
                                     'n, e, s, and w')

    def test_paneconfigure_stretch(self):
        p, b, c = self.create2()
        self.check_paneconfigure(p, b, 'stretch', 'alw', 'always')
        self.check_paneconfigure_bad(p, b, 'stretch',
                                     'bad stretch "badValue": must be '
                                     'always, first, last, middle, or never')

    def test_paneconfigure_width(self):
        p, b, c = self.create2()
        self.check_paneconfigure(p, b, 'width', 10, 10)
        self.check_paneconfigure_bad(p, b, 'width',
                EXPECTED_SCREEN_DISTANCE_OR_EMPTY_ERRMSG.format('badValue'))


@add_configure_tests(StandardOptionsTests)
class MenuTest(AbstractWidgetTest, unittest.TestCase):
    OPTIONS = (
        'activebackground', 'activeborderwidth', 'activeforeground',
        'activerelief', 'background', 'borderwidth', 'cursor',
        'disabledforeground', 'font', 'foreground',
        'postcommand', 'relief', 'selectcolor', 'takefocus',
        'tearoff', 'tearoffcommand', 'title', 'type',
    )
    _rounds_pixels = False
    _clipped = {}

    def create(self, **kwargs):
        return tkinter.Menu(self.root, **kwargs)

    def test_indexcommand_none(self):
        widget = self.create()
        i = widget.index('none')
        self.assertIsNone(i)

    test_configure_activerelief = requires_tk(8, 7)(StandardOptionsTests.test_configure_activerelief)

    def test_configure_postcommand(self):
        widget = self.create()
        self.checkCommandParam(widget, 'postcommand')

    def test_configure_tearoff(self):
        widget = self.create()
        self.checkBooleanParam(widget, 'tearoff')

    def test_configure_tearoffcommand(self):
        widget = self.create()
        self.checkCommandParam(widget, 'tearoffcommand')

    def test_configure_title(self):
        widget = self.create()
        self.checkParam(widget, 'title', 'any string')

    def test_configure_type(self):
        widget = self.create()
        values = ('normal', 'tearoff', 'menubar')
        self.checkEnumParam(widget, 'type', *values,
                            allow_empty=tk_version < (8, 7),
                            sort=tk_version >= (8, 7))

    def test_entryconfigure(self):
        m1 = self.create()
        m1.add_command(label='test')
        self.assertRaises(TypeError, m1.entryconfigure)
        with self.assertRaisesRegex(TclError, 'bad menu entry index "foo"'):
            m1.entryconfigure('foo')
        d = m1.entryconfigure(1)
        self.assertIsInstance(d, dict)
        for k, v in d.items():
            self.assertIsInstance(k, str)
            self.assertIsInstance(v, tuple)
            self.assertEqual(len(v), 5)
            self.assertEqual(v[0], k)
            self.assertEqual(m1.entrycget(1, k), v[4])
        m1.destroy()

    def test_entryconfigure_label(self):
        m1 = self.create()
        m1.add_command(label='test')
        self.assertEqual(m1.entrycget(1, 'label'), 'test')
        m1.entryconfigure(1, label='changed')
        self.assertEqual(m1.entrycget(1, 'label'), 'changed')

    def test_entryconfigure_variable(self):
        m1 = self.create()
        v1 = tkinter.BooleanVar(self.root)
        v2 = tkinter.BooleanVar(self.root)
        m1.add_checkbutton(variable=v1, onvalue=True, offvalue=False,
                           label='Nonsense')
        self.assertEqual(str(m1.entrycget(1, 'variable')), str(v1))
        m1.entryconfigure(1, variable=v2)
        self.assertEqual(str(m1.entrycget(1, 'variable')), str(v2))


@add_configure_tests(PixelSizeTests, StandardOptionsTests)
class MessageTest(AbstractWidgetTest, unittest.TestCase):
    OPTIONS = (
        'anchor', 'aspect', 'background', 'borderwidth',
        'cursor', 'font', 'foreground',
        'highlightbackground', 'highlightcolor', 'highlightthickness',
        'justify', 'padx', 'pady', 'relief',
        'takefocus', 'text', 'textvariable', 'width',
    )
    _rounds_pixels = (tk_version < (9, 0))
    _no_round = {'padx', 'pady'}
    if tk_version < (9, 0):
        _clipped = {'highlightthickness'}
    else:
        _clipped = {'borderwidth', 'highlightthickness', 'padx', 'pady'}

    def create(self, **kwargs):
        return tkinter.Message(self.root, **kwargs)

    def test_configure_aspect(self):
        widget = self.create()
        self.checkIntegerParam(widget, 'aspect', 250, 0, -300)

    def test_configure_padx(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'padx', 3, 4.4, 5.6, '12m')
        expected = -2 if tk_version < (9, 0) else self._default_pixels
        self.checkParam(widget, 'padx', -2, expected=expected)

    def test_configure_pady(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'pady', 3, 4.4, 5.6, '12m')
        expected = -2 if tk_version < (9, 0) else self._default_pixels
        self.checkParam(widget, 'pady', -2, expected=expected)

    def test_configure_width(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'width', 402, 403.4, 404.6, 0, '5i')
        expected = 0 if tk_version >= (8, 7) else -402
        self.checkParam(widget, 'width', -402, expected=expected)


class DefaultRootTest(AbstractDefaultRootTest, unittest.TestCase):

    def test_frame(self):
        self._test_widget(tkinter.Frame)

    def test_label(self):
        self._test_widget(tkinter.Label)


if __name__ == '__main__':
    unittest.main()

### File: widget_tests.py

In [ ]:
# Common tests for test_tkinter/test_widgets.py and test_ttk/test_widgets.py

import re
import tkinter
from test.test_tkinter.support import (AbstractTkTest, requires_tk, tk_version,
                                  pixels_conv, tcl_obj_eq)
import test.support

_sentinel = object()

# Options which accept all values allowed by Tk_GetPixels
# borderwidth = bd

class AbstractWidgetTest(AbstractTkTest):
    _default_pixels = ''   # Value for unset pixel options.
    _rounds_pixels = True  # True if some pixel options are rounded.
    _no_round = {}         # Pixel options which are not rounded nonetheless
    _stringify = False     # Whether to convert tuples to strings
    _allow_empty_justify = False

    @property
    def scaling(self):
        try:
            return self._scaling
        except AttributeError:
            self._scaling = float(self.root.call('tk', 'scaling'))
            return self._scaling

    def _str(self, value):
        if not self._stringify and self.wantobjects and tk_version >= (8, 6):
            return value
        if isinstance(value, tuple):
            return ' '.join(map(self._str, value))
        return str(value)

    def assertEqual2(self, actual, expected, msg=None, eq=object.__eq__):
        if eq(actual, expected):
            return
        self.assertEqual(actual, expected, msg)

    def checkParam(self, widget, name, value, *, expected=_sentinel,
                   conv=False, eq=None):
        widget[name] = value
        if expected is _sentinel:
            expected = value
        if name in self._clipped:
            if not isinstance(expected, str):
                expected = max(expected, 0)
        if conv:
            expected = conv(expected)
        if self._stringify or not self.wantobjects:
            if isinstance(expected, tuple):
                expected = tkinter._join(expected)
            else:
                expected = str(expected)
        if eq is None:
            eq = tcl_obj_eq
        self.assertEqual2(widget[name], expected, eq=eq)
        self.assertEqual2(widget.cget(name), expected, eq=eq)
        t = widget.configure(name)
        self.assertEqual(len(t), 5)
        self.assertEqual2(t[4], expected, eq=eq)

    def checkInvalidParam(self, widget, name, value, errmsg=None):
        orig = widget[name]
        if errmsg is not None:
            errmsg = errmsg.format(re.escape(str(value)))
            errmsg = fr'\A{errmsg}\z'
        with self.assertRaisesRegex(tkinter.TclError, errmsg or ''):
            widget[name] = value
        self.assertEqual(widget[name], orig)
        with self.assertRaisesRegex(tkinter.TclError, errmsg or ''):
            widget.configure({name: value})
        self.assertEqual(widget[name], orig)

    def checkParams(self, widget, name, *values, **kwargs):
        for value in values:
            self.checkParam(widget, name, value, **kwargs)

    def checkIntegerParam(self, widget, name, *values, **kwargs):
        self.checkParams(widget, name, *values, **kwargs)
        errmsg = 'expected integer but got "{}"'
        self.checkInvalidParam(widget, name, '', errmsg=errmsg)
        self.checkInvalidParam(widget, name, '10p', errmsg=errmsg)
        self.checkInvalidParam(widget, name, 3.2, errmsg=errmsg)

    def checkFloatParam(self, widget, name, *values, conv=float, **kwargs):
        for value in values:
            self.checkParam(widget, name, value, conv=conv, **kwargs)
        errmsg = 'expected floating-point number but got "{}"'
        self.checkInvalidParam(widget, name, '', errmsg=errmsg)
        self.checkInvalidParam(widget, name, 'spam', errmsg=errmsg)

    def checkBooleanParam(self, widget, name):
        for value in (False, 0, 'false', 'no', 'off'):
            self.checkParam(widget, name, value, expected=0)
        for value in (True, 1, 'true', 'yes', 'on'):
            self.checkParam(widget, name, value, expected=1)
        errmsg = 'expected boolean value but got "{}"'
        self.checkInvalidParam(widget, name, '', errmsg=errmsg)
        self.checkInvalidParam(widget, name, 'spam', errmsg=errmsg)

    def checkColorParam(self, widget, name, *, allow_empty=None, **kwargs):
        self.checkParams(widget, name,
                         '#ff0000', '#00ff00', '#0000ff', '#123456',
                         'red', 'green', 'blue', 'white', 'black', 'grey',
                         **kwargs)
        self.checkInvalidParam(widget, name, 'spam',
                errmsg='unknown color name "spam"')

    def checkCursorParam(self, widget, name, **kwargs):
        self.checkParams(widget, name, 'arrow', 'watch', 'cross', '',**kwargs)
        self.checkParam(widget, name, 'none')
        self.checkInvalidParam(widget, name, 'spam',
                errmsg='bad cursor spec "spam"')

    def checkCommandParam(self, widget, name):
        def command(*args):
            pass
        widget[name] = command
        self.assertTrue(widget[name])
        self.checkParams(widget, name, '')

    def checkEnumParam(self, widget, name, *values,
                       errmsg=None, allow_empty=False, fullname=None,
                       sort=False, **kwargs):
        self.checkParams(widget, name, *values, **kwargs)
        if errmsg is None:
            if sort:
                if values[-1]:
                    values = tuple(sorted(values))
                else:
                    values = tuple(sorted(values[:-1])) + ('',)
            errmsg2 = ' %s "{}": must be %s%s or %s' % (
                    fullname or name,
                    ', '.join(values[:-1]),
                    ',' if len(values) > 2 else '',
                    values[-1] or '""')
            if '' not in values and not allow_empty:
                self.checkInvalidParam(widget, name, '',
                                       errmsg='ambiguous' + errmsg2)
            errmsg = 'bad' + errmsg2
        self.checkInvalidParam(widget, name, 'spam', errmsg=errmsg)

    def checkPixelsParam(self, widget, name, *values, conv=None, **kwargs):
        if not self._rounds_pixels or name in self._no_round:
            conv = False
        elif conv != str:
            conv = round
        for value in values:
            expected = _sentinel
            conv1 = conv
            if isinstance(value, str):
                if not getattr(self, '_converts_pixels', True):
                    conv1 = str
                if conv1 and conv1 is not str:
                    expected = pixels_conv(value) * self.scaling
                    conv1 = round
            self.checkParam(widget, name, value, expected=expected,
                            conv=conv1, **kwargs)
        errmsg = '(bad|expected) screen distance ((or "" )?but got )?"{}"'
        self.checkInvalidParam(widget, name, '6x', errmsg=errmsg)
        self.checkInvalidParam(widget, name, 'spam', errmsg=errmsg)

    def checkReliefParam(self, widget, name, *, allow_empty=False):
        values = ('flat', 'groove', 'raised', 'ridge', 'solid', 'sunken')
        if allow_empty:
            values += ('',)
        self.checkParams(widget, name, *values)
        errmsg = 'bad relief "{}": must be %s, or %s' % (
                ', '.join(values[:-1]),
                values[-1] or '""')
        if tk_version < (8, 6):
            errmsg = None
        self.checkInvalidParam(widget, name, 'spam', errmsg=errmsg)

    def checkImageParam(self, widget, name):
        image = tkinter.PhotoImage(master=self.root, name='image1')
        self.checkParam(widget, name, image, conv=str)
        if tk_version < (9, 0):
            errmsg = 'image "spam" doesn\'t exist'
        else:
            errmsg = 'image "spam" does not exist'
        self.checkInvalidParam(widget, name, 'spam',
                               errmsg=errmsg)
        widget[name] = ''

    def checkVariableParam(self, widget, name, var):
        self.checkParam(widget, name, var, conv=str)

    def assertIsBoundingBox(self, bbox):
        self.assertIsNotNone(bbox)
        self.assertIsInstance(bbox, tuple)
        if len(bbox) != 4:
            self.fail('Invalid bounding box: %r' % (bbox,))
        for item in bbox:
            if not isinstance(item, int):
                self.fail('Invalid bounding box: %r' % (bbox,))
                break


    def test_keys(self):
        widget = self.create()
        keys = widget.keys()
        self.assertEqual(sorted(keys), sorted(widget.configure()))
        for k in keys:
            widget[k]
        # Test if OPTIONS contains all keys
        if test.support.verbose:
            aliases = {
                'bd': 'borderwidth',
                'bg': 'background',
                'bgimg': 'backgroundimage',
                'fg': 'foreground',
                'invcmd': 'invalidcommand',
                'vcmd': 'validatecommand',
            }
            keys = set(keys)
            expected = set(self.OPTIONS)
            for k in sorted(keys - expected):
                if not (k in aliases and
                        aliases[k] in keys and
                        aliases[k] in expected):
                    print('%s.OPTIONS doesn\'t contain "%s"' %
                          (self.__class__.__name__, k))

class PixelOptionsTests:
    """Standard options that accept all formats acceptable to Tk_GetPixels.

    In addition to numbers, these options can be set with distances
    specified as a string consisting of a number followed by a single
    character giving the unit of distance. The allowed units are:
    millimeters ('m'), centimeters ('c'), inches ('i') or points ('p').
    In Tk 9 a cget call for one of these options returns a Tcl_Obj of
    type "pixels", whose string representation is the distance string
    passed to configure.
    """
    PIXEL_OPTIONS = ('activeborderwidth', 'borderwidth', 'highlightthickness',
      'insertborderwidth', 'insertwidth', 'padx', 'pady', 'selectborderwidth')

    def test_configure_activeborderwidth(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'activeborderwidth',
                              0, 1.3, 2.9, 6, -2, '10p')

    def test_configure_borderwidth(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'borderwidth',
                              0, 1.3, 2.6, 6, '10p')
        if tk_version < (9, 0):
            self.checkParam(widget, 'borderwidth', -2)
        else:
            self.checkParam(widget, 'borderwidth', 0)

        if 'bd' in self.OPTIONS:
            self.checkPixelsParam(widget, 'bd', 0, 1.3, 2.6, 6, '10p')
            self.checkParam(widget, 'bd', -2, expected=expected)

    def test_configure_highlightthickness(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'highlightthickness',
                              0, 1.3, 2.6, 6, '10p')
        self.checkParam(widget, 'highlightthickness', -2)

    def test_configure_insertborderwidth(self):
        widget = self.create()
        if tk_version < (9, 0):
            values = (0, 1.3, 2.6, 6, -2, '10p')
            value = -2
        else:
            values = (0, 1, 3, 6, 13)
            value = 0
        self.checkPixelsParam(widget, 'insertborderwidth', *values)
        self.checkParam(widget, 'insertborderwidth', value)

    def test_configure_insertwidth(self):
        widget = self.create()
        if tk_version < (9, 0):
            self.checkPixelsParam(widget, 'insertwidth', 1.3, 2.6, -2, '10p')
        else:
            self.checkPixelsParam(widget, 'insertwidth', 1, 3, 0, 13)

    def test_configure_padx(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'padx', 3, 4.4, 5.6, '12m')
        if tk_version < (9, 0):
            self.checkParam(widget, 'padx', -2)
        else:
            self.checkParam(widget, 'padx', 0)

    def test_configure_pady(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'pady', 3, 4.4, 5.6, '12m')
        if tk_version < (9, 0):
            self.checkParam(widget, 'pady', -2)
        else:
            self.checkParam(widget, 'pady', 0)

    def test_configure_selectborderwidth(self):
        widget = self.create()
        if tk_version < (9, 0):
            values = (1.3, 2.6, -2, '10p')
        else:
            values = (1, 3, 0, 13)
        self.checkPixelsParam(widget, 'selectborderwidth', *values)


class StandardOptionsTests(PixelOptionsTests):

    STANDARD_OPTIONS = ( 'activebackground', 'activeforeground',
    'anchor', 'background', 'bitmap', 'compound', 'cursor',
    'disabledforeground', 'exportselection', 'font', 'foreground',
    'highlightbackground', 'highlightcolor', 'image',
    'insertbackground', 'insertofftime', 'insertontime', 'jump',
    'justify', 'orient', 'relief', 'repeatdelay', 'repeatinterval',
    'selectbackground', 'selectforeground', 'setgrid', 'takefocus',
    'text', 'textvariable', 'troughcolor', 'underline', 'wraplength',
    'xscrollcommand', 'yscrollcommand', ) + PixelOptionsTests.PIXEL_OPTIONS

    def test_configure_activebackground(self):
        widget = self.create()
        self.checkColorParam(widget, 'activebackground')

    def test_configure_activeforeground(self):
        widget = self.create()
        self.checkColorParam(widget, 'activeforeground')

    def test_configure_activerelief(self):
        widget = self.create()
        self.checkReliefParam(widget, 'activerelief')

    def test_configure_anchor(self):
        widget = self.create()
        self.checkEnumParam(widget, 'anchor',
                'n', 'ne', 'e', 'se', 's', 'sw', 'w', 'nw', 'center')

    def test_configure_background(self):
        widget = self.create()
        self.checkColorParam(widget, 'background')
        if 'bg' in self.OPTIONS:
            self.checkColorParam(widget, 'bg')

    @requires_tk(8, 7)
    def test_configure_backgroundimage(self):
        widget = self.create()
        self.checkImageParam(widget, 'backgroundimage')

    def test_configure_bitmap(self):
        widget = self.create()
        self.checkParam(widget, 'bitmap', 'questhead')
        self.checkParam(widget, 'bitmap', 'gray50')
        filename = test.support.findfile('python.xbm', subdir='tkinterdata')
        self.checkParam(widget, 'bitmap', '@' + filename)
        # Cocoa Tk widgets don't detect invalid -bitmap values
        # See https://core.tcl.tk/tk/info/31cd33dbf0
        if not ('aqua' in self.root.tk.call('tk', 'windowingsystem') and
                'AppKit' in self.root.winfo_server()):
            self.checkInvalidParam(widget, 'bitmap', 'spam',
                    errmsg='bitmap "spam" not defined')

    def test_configure_compound(self):
        widget = self.create()
        self.checkEnumParam(widget, 'compound',
                'bottom', 'center', 'left', 'none', 'right', 'top')

    def test_configure_cursor(self):
        widget = self.create()
        self.checkCursorParam(widget, 'cursor')

    def test_configure_disabledforeground(self):
        widget = self.create()
        self.checkColorParam(widget, 'disabledforeground')

    def test_configure_exportselection(self):
        widget = self.create()
        self.checkBooleanParam(widget, 'exportselection')

    def test_configure_font(self):
        widget = self.create()
        self.checkParam(widget, 'font',
                        '-Adobe-Helvetica-Medium-R-Normal--*-120-*-*-*-*-*-*')
        is_ttk = widget.__class__.__module__ == 'tkinter.ttk'
        if not is_ttk:
            errmsg = 'font "" does ?n[o\']t exist'
            self.checkInvalidParam(widget, 'font', '', errmsg=errmsg)

    def test_configure_foreground(self):
        widget = self.create()
        self.checkColorParam(widget, 'foreground')
        if 'fg' in self.OPTIONS:
            self.checkColorParam(widget, 'fg')

    def test_configure_highlightbackground(self):
        widget = self.create()
        self.checkColorParam(widget, 'highlightbackground')

    def test_configure_highlightcolor(self):
        widget = self.create()
        self.checkColorParam(widget, 'highlightcolor')

    def test_configure_image(self):
        widget = self.create()
        self.checkImageParam(widget, 'image')

    def test_configure_insertbackground(self):
        widget = self.create()
        self.checkColorParam(widget, 'insertbackground')

    def test_configure_insertofftime(self):
        widget = self.create()
        self.checkIntegerParam(widget, 'insertofftime', 100)

    def test_configure_insertontime(self):
        widget = self.create()
        self.checkIntegerParam(widget, 'insertontime', 100)

    def test_configure_jump(self):
        widget = self.create()
        self.checkBooleanParam(widget, 'jump')

    def test_configure_justify(self):
        widget = self.create()
        values = ('left', 'right', 'center')
        if self._allow_empty_justify:
            values += ('',)
        self.checkEnumParam(widget, 'justify', *values,
                            fullname='justification')

    def test_configure_orient(self):
        widget = self.create()
        self.assertEqual(str(widget['orient']), self.default_orient)
        self.checkEnumParam(widget, 'orient', 'horizontal', 'vertical')

    @requires_tk(8, 7)
    def test_configure_placeholder(self):
        widget = self.create()
        self.checkParam(widget, 'placeholder', 'xxx')

    @requires_tk(8, 7)
    def test_configure_placeholderforeground(self):
        widget = self.create()
        self.checkColorParam(widget, 'placeholderforeground')

    def test_configure_relief(self):
        widget = self.create()
        self.checkReliefParam(widget, 'relief')

    def test_configure_repeatdelay(self):
        widget = self.create()
        self.checkIntegerParam(widget, 'repeatdelay', -500, 500)

    def test_configure_repeatinterval(self):
        widget = self.create()
        self.checkIntegerParam(widget, 'repeatinterval', -500, 500)

    def test_configure_selectbackground(self):
        widget = self.create()
        self.checkColorParam(widget, 'selectbackground')

    def test_configure_selectforeground(self):
        widget = self.create()
        self.checkColorParam(widget, 'selectforeground')

    def test_configure_setgrid(self):
        widget = self.create()
        self.checkBooleanParam(widget, 'setgrid')

    def test_configure_state(self):
        widget = self.create()
        self.checkEnumParam(widget, 'state', 'active', 'disabled', 'normal')

    def test_configure_takefocus(self):
        widget = self.create()
        self.checkParams(widget, 'takefocus', '0', '1', '')

    def test_configure_text(self):
        widget = self.create()
        self.checkParams(widget, 'text', '', 'any string')

    def test_configure_textvariable(self):
        widget = self.create()
        var = tkinter.StringVar(self.root)
        self.checkVariableParam(widget, 'textvariable', var)

    @requires_tk(8, 7)
    def test_configure_tile(self):
        widget = self.create()
        self.checkBooleanParam(widget, 'tile')

    def test_configure_troughcolor(self):
        widget = self.create()
        self.checkColorParam(widget, 'troughcolor')

    def test_configure_underline(self):
        widget = self.create()
        self.checkParams(widget, 'underline', 0, 1, 10)
        if tk_version >= (8, 7):
            is_ttk = widget.__class__.__module__ == 'tkinter.ttk'
            self.checkParam(widget, 'underline', '',
                            expected='' if is_ttk else self._default_pixels)
            self.checkParam(widget, 'underline', '5+2',
                            expected='5+2' if is_ttk else 7)
            self.checkParam(widget, 'underline', '5-2',
                            expected='5-2' if is_ttk else 3)
            self.checkParam(widget, 'underline', 'end', expected='end')
            self.checkParam(widget, 'underline', 'end-2', expected='end-2')
            errmsg = (r'bad index "{}": must be integer\?\[\+-\]integer\?, '
                      r'end\?\[\+-\]integer\?, or ""')
        else:
            errmsg = 'expected integer but got "{}"'
            self.checkInvalidParam(widget, 'underline', '', errmsg=errmsg)
        self.checkInvalidParam(widget, 'underline', '10p', errmsg=errmsg)
        self.checkInvalidParam(widget, 'underline', 3.2, errmsg=errmsg)

    def test_configure_wraplength(self):
        widget = self.create()
        self.checkPixelsParam(widget, 'wraplength', 100)

    def test_configure_xscrollcommand(self):
        widget = self.create()
        self.checkCommandParam(widget, 'xscrollcommand')

    def test_configure_yscrollcommand(self):
        widget = self.create()
        self.checkCommandParam(widget, 'yscrollcommand')

    # non-standard but common options

    def test_configure_command(self):
        widget = self.create()
        self.checkCommandParam(widget, 'command')

    def test_configure_indicatoron(self):
        widget = self.create()
        self.checkBooleanParam(widget, 'indicatoron')

    def test_configure_offrelief(self):
        widget = self.create()
        self.checkReliefParam(widget, 'offrelief')

    def test_configure_overrelief(self):
        widget = self.create()
        self.checkReliefParam(widget, 'overrelief',
                              allow_empty=(tk_version >= (8, 7)))

    def test_configure_selectcolor(self):
        widget = self.create()
        self.checkColorParam(widget, 'selectcolor')

    def test_configure_selectimage(self):
        widget = self.create()
        self.checkImageParam(widget, 'selectimage')

    def test_configure_tristateimage(self):
        widget = self.create()
        self.checkImageParam(widget, 'tristateimage')

    def test_configure_tristatevalue(self):
        widget = self.create()
        self.checkParam(widget, 'tristatevalue', 'unknowable')

    def test_configure_variable(self):
        widget = self.create()
        var = tkinter.DoubleVar(self.root)
        self.checkVariableParam(widget, 'variable', var)


class IntegerSizeTests:
    """ Tests widgets which only accept integral width and height."""
    def test_configure_height(self):
        widget = self.create()
        if tk_version < (9, 0):
            self.checkIntegerParam(widget, 'height', 100, -100, 0)
        else:
            self.checkIntegerParam(widget, 'height', 100, 0, 0)

    def test_configure_width(self):
        widget = self.create()
        if tk_version < (9, 0):
            self.checkIntegerParam(widget, 'width', 402, -402, 0)
        else:
            self.checkIntegerParam(widget, 'width', 402, 0, 0)


class PixelSizeTests:
    """ Tests widgets which accept screen distances for width and height."""
    def test_configure_height(self):
        widget = self.create()
        value = -100 if tk_version < (9, 0) else 0
        self.checkPixelsParam(
            widget, 'height', 100, 101.2, 102.6, value, 0, '3c'
        )

    def test_configure_width(self):
        widget = self.create()
        value = -402 if tk_version < (9, 0) else 0
        self.checkPixelsParam(
            widget, 'width', 402, 403.4, 404.6, value, 0, '5i'
        )


def add_configure_tests(*source_classes):
    # This decorator adds test_configure_xxx methods from source classes for
    # every xxx option in the OPTIONS class attribute if they are not defined
    # explicitly.
    def decorator(cls):
        for option in cls.OPTIONS:
            methodname = 'test_configure_' + option
            if not hasattr(cls, methodname):
                for source_class in source_classes:
                    if hasattr(source_class, methodname):
                        setattr(cls, methodname,
                                getattr(source_class, methodname))
                        break
                else:
                    def test(self, option=option):
                        widget = self.create()
                        widget[option]
                        raise AssertionError('Option "%s" is not tested in %s' %
                                             (option, cls.__name__))
                    test.__name__ = methodname
                    setattr(cls, methodname, test)
        return cls
    return decorator

def setUpModule():
    if test.support.verbose:
        tcl = tkinter.Tcl()
        print('patchlevel =', tcl.call('info', 'patchlevel'), flush=True)